In [6]:
from dateutil import parser as date_parser
from datetime import datetime

def test_parse_date(date_strings):
    for s in date_strings:
        try:
            parsed = date_parser.parse(s)
            print(f"✅ Parsed '{s}' -> {parsed} (year={parsed.year}, month={parsed.month}, day={parsed.day})")
        except Exception as e:
            # Fallback parsing with specific formats
            parsed = None
            for fmt in ("%m%d%Y", "%d%m%Y"):
                try:
                    parsed = datetime.strptime(s, fmt)
                    print(f"✅ Fallback parsed '{s}' with format {fmt} -> {parsed} (year={parsed.year}, month={parsed.month}, day={parsed.day})")
                    break
                except Exception:
                    continue
            if not parsed:
                print(f"❌ Could not parse '{s}': {e}")

# Example usage
date_samples = [
    "3000-01-01 00:00:00.000",
    "2025-08-25 10:45:59",
    "08/25/2025",
    "25-08-2025",
    "20250825",
    "08252025",
    "25082025",
    "MAR ,3, 2025",
    "03-25-81",
    "6031988",
]

test_parse_date(date_samples)


✅ Parsed '3000-01-01 00:00:00.000' -> 3000-01-01 00:00:00 (year=3000, month=1, day=1)
✅ Parsed '2025-08-25 10:45:59' -> 2025-08-25 10:45:59 (year=2025, month=8, day=25)
✅ Parsed '08/25/2025' -> 2025-08-25 00:00:00 (year=2025, month=8, day=25)
✅ Parsed '25-08-2025' -> 2025-08-25 00:00:00 (year=2025, month=8, day=25)
✅ Parsed '20250825' -> 2025-08-25 00:00:00 (year=2025, month=8, day=25)
✅ Fallback parsed '08252025' with format %m%d%Y -> 2025-08-25 00:00:00 (year=2025, month=8, day=25)
✅ Fallback parsed '25082025' with format %d%m%Y -> 2025-08-25 00:00:00 (year=2025, month=8, day=25)
✅ Parsed 'MAR ,3, 2025' -> 2025-03-03 00:00:00 (year=2025, month=3, day=3)
✅ Parsed '03-25-81' -> 1981-03-25 00:00:00 (year=1981, month=3, day=25)
✅ Fallback parsed '6031988' with format %m%d%Y -> 1988-06-03 00:00:00 (year=1988, month=6, day=3)


In [9]:
import re

text = "Hyperostosis frontalis interna is considered to be incidental and within normal limits"
val = "dent"

pattern = rf"(?i)(?<!\w){re.escape(val)}(?!\w)"
print(re.sub(pattern, "((NAME))", text))


Hyperostosis frontalis interna is considered to be inci<((NAME)) al and within normal limits


In [5]:
text = """<?xml version="1.0" encoding="UTF-8"?>
<Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsd="http://www.w3.org/1999/XMLSchema" xmlns:xsi="http://www.w3.org/1999/XMLSchema-instance">
	<Body>
		<return>
			<fax>
			</fax>
			<GuarantorName>
				Straight, Mary F
			</GuarantorName>
			<GuarantorId>
				644218
			</GuarantorId>
			<patient>
				Straight, Mary F
			</patient>
			<EncounterId>140966</EncounterId>
			<PatientId>
				644218
			</PatientId>
			<ProviderId>
				12600
			</ProviderId>
			<ControlNo>
				644647
			</ControlNo>
			<MrnNo>
			</MrnNo>
			<CHNNo>
			</CHNNo>
			<DOB>
				02/14/1936
			</DOB>
			<phone>
				305-232-3630
			</phone>
			<address>
				703 Eastridge Village, Drive, Miami, FL-33157
			</address>
			<encDate>
				08/06/2013
			</encDate>
			<age>
				77 Y
			</age>
			<sex>
				Female
			</sex>
			<reqNo>
				644218.140966
			</reqNo>
			"""

In [4]:
import xml.etree.ElementTree as ET

# Reuse your cleaning logic
import re

XML_DECLARATION_RE = re.compile(r"<\?xml[^>]*\?>", re.IGNORECASE)
XML_STYLESHEET_RE = re.compile(r"<\?xml-stylesheet[^>]*\?>", re.IGNORECASE)
CDATA_WRAP_RE = re.compile(r"<!\[CDATA\[(.*)\]\]>", re.DOTALL)

def clean_xml_before_deid(raw_xml: str) -> str:
    if not isinstance(raw_xml, str):
        return raw_xml
    cleaned = raw_xml.strip()

    # Unwrap CDATA section if present
    m = CDATA_WRAP_RE.search(cleaned)
    if m:
        cleaned = m.group(1).strip()

    # Remove XML declaration
    cleaned = XML_DECLARATION_RE.sub("", cleaned)

    # Remove XML stylesheet instructions
    cleaned = XML_STYLESHEET_RE.sub("", cleaned)

    cleaned = cleaned.strip()
    return cleaned


def test_xml(label: str, xml_text: str):
    print(f"\n======= Testing {label} =======")
    cleaned = clean_xml_before_deid(xml_text)
    print("Cleaned XML (first 300 chars):")
    print(cleaned[:300], "...\n")

    try:
        root = ET.fromstring(cleaned)
        print("✅ Parsed successfully. Root tag:", root.tag)
        print("Child tags:", [child.tag for child in root])
    except ET.ParseError as e:
        print("❌ Parse error:", e)


if __name__ == "__main__":
    # --- Put your raw XML strings here ---
    PROGRESSNOTE_XML = """<progressnote>
	<![CDATA[
	<?xml version="1.0" encoding="ISO-8859-1"?>
	<?xml-stylesheet href="/mobiledoc/xsl/progressnotes/ModernINotes_2014-05-27 23 41 34.xsl" type="text/xsl"?><SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsd="http://www.w3.org/1999/XMLSchema" xmlns:xsi="http://www.w3.org/1999/XMLSchema-instance"><SOAP-ENV:Body><return><SectionTitle/><SectionSubTitle/><IndentStyle>bullet</IndentStyle><SubTitle>no</SubTitle><fax/><GuarantorName>Abdelqader, Nawal</GuarantorName><GuarantorId>12406</GuarantorId><TrUserId>122</TrUserId><OnDemandLoadThumbnail>no</OnDemandLoadThumbnail><ThumbNailHeight>140</ThumbNailHeight><ShowThumbNailsInProgressNotes>no</ShowThumbNailsInProgressNotes><patient>Abdelqader, Nawal</patient><EncounterId>44037</EncounterId><PatientId>12406</PatientId><ProviderId>122</ProviderId><ControlNo>12406</ControlNo><MrnNo/><CHNNo/><DOB>10/06/1970</DOB><phone>520-903-8423</phone><address>14387 N. Banner Stone Crt, Marana, AZ-85658</address><encDate>07/01/2014</encDate><age>43 Y</age><sex>Female</sex><reqNo>12406.44037</reqNo><provider>Said Ibrahimi, MD</provider><ApptFacility>Karima Neurological Diagnostic Center</ApptFacility><ApptFacilityId>2</ApptFacilityId><Resident/><StLicNo>43362</StLicNo><userName>saidi</userName><TimeStamp>Electronically signed by SAID IBRAHIMI , MD  on 07/01/2014 at 06:20 PM MDT</TimeStamp><SignOffStatus>Sign off status: Completed</SignOffStatus><ShowSignature>Yes</ShowSignature><HospitalName>Karima Neurological Diagnostic Center</HospitalName><HospitalName1>Karima Neurological Diagnostic Center</HospitalName1><HospitalName2/><HospitalAddress1>2055 West Hospital Drive</HospitalAddress1><HospitalAddress2>Ste 205</HospitalAddress2><HospitalAddress3>Tucson, AZ 857047822</HospitalAddress3><HospitalPhone>520-638-6482</HospitalPhone><HospitalFax>520-638-6786</HospitalFax><PrimaryInsName/><PrimaryInsSubscriberNo/><PrimaryInsPrintName/><PrimaryInsPrint/><SecInsName/><SecInsSubscriberNo/><SecInsPrintName/><SecInsPrint/><refPr>Watus B Cooper Jr, PA-C </refPr><NotesType>Progress Notes</NotesType><ChartHeadings><CC>Reason for Appointment</CC><HPI>History of Present Illness</HPI><CurrentMedication>Current Medications</CurrentMedication><MedicalHistory>Past Medical History</MedicalHistory><Allergies>Allergies</Allergies><GynHistory>Gyn History</GynHistory><OBHistory>OB History</OBHistory><SurgicalHistory>Surgical History</SurgicalHistory><Hospitalization>Hospitalization/Major Diagnostic Procedure</Hospitalization><FamilyHistory>Family History</FamilyHistory><SocialHistory>Social History</SocialHistory><ROS>Review of Systems</ROS><Vitals>Vital Signs</Vitals><Examination>Examination</Examination><PhysicalExamination>Physical Examination</PhysicalExamination><Therapeutic>Therapeutic Interventions</Therapeutic><TherapyAssessment>Physical Therapy Assessment</TherapyAssessment><Assessment>Assessments</Assessment><Treatment>Treatment</Treatment><Procedures>Procedures</Procedures><Immunizations>Immunization</Immunizations><Injections>Therapeutic Injections</Injections><DiagnosticImaging>Diagnostic Imaging</DiagnosticImaging><LabReports>Labs</LabReports><PreventiveMedicine>Preventive Medicine</PreventiveMedicine><NextAppointment>Follow Up</NextAppointment><VisitCode>Visit Codes</VisitCode><ProcedureCodes>Procedure Codes</ProcedureCodes><SurgicalPosting>Surgical Posting</SurgicalPosting><PastOrders>Past Orders</PastOrders><DispositionAndCommunication>Disposition &amp; Communication</DispositionAndCommunication><VisionExamination>Vision Examination</VisionExamination><DispositionAndCommunication>Disposition &amp; Communication</DispositionAndCommunication><ProblemList>Active Problem List</ProblemList></ChartHeadings><items/><subItems><subItemsName>Subjective:</subItemsName><cc><itemValue>follow up; MRI results</itemValue></cc><CurrentMeds><category><categoryName>Taking</categoryName><categoryValue>Levothyroxine Sodium 175 MCG Tablet 1 tablet Once a day</categoryValue><categoryValue>Hydromorphone HCl 2 MG Tablet 1 tablet Every 8 Hrs</categoryValue><categoryValue>Gabapentin 300 MG Capsule 1 capsule QHS for 1 week, then increase to 1 cap PO BID</categoryValue><categoryValue>Medication List reviewed and reconciled with the patient</categoryValue></category><itemValue>Taking Levothyroxine Sodium 175 MCG Tablet 1 tablet Once a day</itemValue><itemValue>Taking Hydromorphone HCl 2 MG Tablet 1 tablet Every 8 Hrs</itemValue><itemValue>Taking Gabapentin 300 MG Capsule 1 capsule QHS for 1 week, then increase to 1 cap PO BID</itemValue><itemValue>Medication List reviewed and reconciled with the patient</itemValue></CurrentMeds><PastHistory><itemValue>No Medical History.</itemValue></PastHistory><allergies><itemValue>Tramadol HCl: Allergy</itemValue></allergies><hpi><category><categoryName>Constitutional</categoryName><catId>268981</catId><hpiCatNotes><innerxml-cdata>43 year old female who is presenting to the office after a MVA 10-31-2013. The car accident was on Prince Rd. She was the restrained passenger in the front seat of a Nissen min van and they were rear ended by a small sedan. Both cars were totalled. She went into the hospital at KINO the day after the accident. She was seen in the ER and had xrays and was sent home. The patient states she had some ALOC on the scene, no LOC and the patient did not hit her head. <BR>&#160;&#160;&#160;&#160;&#160;&#160;&#160;ROS shows recent daily headaches with memory loss. The patient is also c/o dizziness with position changes and the room is spinning. She is also c/o difficulty sleeping due to her lower back pain. She is also c/o Right hand weakness and pain. She states her right shoulder also hurts where the seat belt pressed into her arm. <BR>&#160;&#160;&#160;&#160;&#160;&#160;&#160;Physical exam is nonfocal, and the fundoscopic exam is unremarkable. The patient has some subjective sensory loss to temperature in the Right hand. Mild weakness is right hand, negative Tinel. <BR>&#160;&#160;&#160;&#160;&#160;&#160;&#160;Since initial office visit, the patient had a NCS study of the hands done, which showed bilateral median nerve dysfunction across both wrists.<BR>&#160;&#160;&#160;&#160;&#160;&#160;&#160;Patient also started on gabapentin 300mg PO BID for post-concussive state, with some help, but still has significant pain and discomfort. <BR>&#160;&#160;&#160;&#160;&#160;&#160;&#160;Patient's recent brain MRI negative, and C spine MRI showed significant degeneration and central stenosis. Please refer to official report for greater details. Patient's exam does not show any upper motor neuron signs, and no issues with bowel or urine. Patient still complains of chronic neck pain.</innerxml-cdata></hpiCatNotes></category><structDataDisplay>2</structDataDisplay></hpi></subItems><subItems><subItemsName>Objective:</subItemsName><vitals2BR><innerxml-cdata>Temp 97.8, HR 62, BP 132/71, Ht 65, Wt 232.4, BMI 38.67.</innerxml-cdata></vitals2BR><PastOrders/><examination><category><categoryName>General Examination</categoryName><catId>270360</catId><categoryInDetail><cat_det><categorySubName>GENERAL APPEARANCE:</categorySubName><examNotes><innerxml-cdata><FONT size=2>in no acute distress</FONT><FONT size=2>,  </FONT><FONT size=2>well developed, well nourished</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>in no acute distress</FONT><FONT size=2>,  </FONT><FONT size=2>well developed, well nourished</FONT></innerxml-cdata></examNotes2><categoryPropId>23066</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>HEAD:</categorySubName><examNotes><innerxml-cdata><FONT size=2>normocephalic</FONT><FONT size=2>,  </FONT><FONT size=2>atraumatic</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>normocephalic</FONT><FONT size=2>,  </FONT><FONT size=2>atraumatic</FONT></innerxml-cdata></examNotes2><categoryPropId>23067</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>EYES:</categorySubName><examNotes><innerxml-cdata><FONT size=2>pupils equal, round, reactive to light and accommodation</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>pupils equal, round, reactive to light and accommodation</FONT></innerxml-cdata></examNotes2><categoryPropId>23068</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>EARS:</categorySubName><examNotes><innerxml-cdata><FONT size=2>normal</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>normal</FONT></innerxml-cdata></examNotes2><categoryPropId>23069</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>ORAL CAVITY:</categorySubName><examNotes><innerxml-cdata><FONT size=2>mucosa moist</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>mucosa moist</FONT></innerxml-cdata></examNotes2><categoryPropId>23091</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>THROAT:</categorySubName><examNotes><innerxml-cdata><FONT size=2>clear</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>clear</FONT></innerxml-cdata></examNotes2><categoryPropId>23071</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>NECK/THYROID:</categorySubName><examNotes><innerxml-cdata><FONT size=2>neck supple, full range of motion</FONT><FONT size=2>,  </FONT><FONT size=2>no cervical lymphadenopathy</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>neck supple, full range of motion</FONT><FONT size=2>,  </FONT><FONT size=2>no cervical lymphadenopathy</FONT></innerxml-cdata></examNotes2><categoryPropId>23072</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>SKIN:</categorySubName><examNotes><innerxml-cdata><FONT size=2>no suspicious lesions</FONT><FONT size=2>,  </FONT><FONT size=2>warm and dry</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>no suspicious lesions</FONT><FONT size=2>,  </FONT><FONT size=2>warm and dry</FONT></innerxml-cdata></examNotes2><categoryPropId>23078</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>HEART:</categorySubName><examNotes><innerxml-cdata><FONT size=2>no murmurs</FONT><FONT size=2>,  </FONT><FONT size=2>regular rate and rhythm</FONT><FONT size=2>,  </FONT><FONT size=2>S1, S2 normal</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>no murmurs</FONT><FONT size=2>,  </FONT><FONT size=2>regular rate and rhythm</FONT><FONT size=2>,  </FONT><FONT size=2>S1, S2 normal</FONT></innerxml-cdata></examNotes2><categoryPropId>23073</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>LUNGS:</categorySubName><examNotes><innerxml-cdata><FONT size=2>clear to auscultation bilaterally</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>clear to auscultation bilaterally</FONT></innerxml-cdata></examNotes2><categoryPropId>23075</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>ABDOMEN:</categorySubName><examNotes><innerxml-cdata><FONT size=2>normal</FONT><FONT size=2>,  </FONT><FONT size=2>bowel sounds present</FONT><FONT size=2>,  </FONT><FONT size=2>soft, nontender, nondistended</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>normal</FONT><FONT size=2>,  </FONT><FONT size=2>bowel sounds present</FONT><FONT size=2>,  </FONT><FONT size=2>soft, nontender, nondistended</FONT></innerxml-cdata></examNotes2><categoryPropId>23076</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>EXTREMITIES:</categorySubName><examNotes><innerxml-cdata><FONT size=2>no clubbing, cyanosis, or edema</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>no clubbing, cyanosis, or edema</FONT></innerxml-cdata></examNotes2><categoryPropId>23079</categoryPropId><encounterID>44037</encounterID><examId>270360</examId><docCount>0</docCount><document/><struct>0</struct></cat_det></categoryInDetail><categoryNotesBR><innerxml-cdata>fundoscopic exam is unremarkable.</innerxml-cdata></categoryNotesBR></category><category><categoryName>Neurological</categoryName><catId>191812</catId><categoryInDetail><cat_det><categorySubName>CORTICAL FUNCTIONS:</categorySubName><examNotes><innerxml-cdata><FONT size=2>normal</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>normal</FONT></innerxml-cdata></examNotes2><categoryPropId>12134</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>CRANIAL NERVES:</categorySubName><examNotes><innerxml-cdata><FONT size=2>no afferent pupil defect</FONT><FONT size=2>,  </FONT><FONT size=2>No ptosis or nystagmus</FONT><FONT size=2>,  </FONT><FONT size=2>Pinprick, light touch intact in all three divisons</FONT><FONT size=2>,  </FONT><FONT size=2>I - Not Tested.</FONT><FONT size=2>,  </FONT><FONT size=2>II - Pupils 4mms reacting briskly to 2 mms</FONT><FONT size=2>,  </FONT><FONT size=2>III , IV, VI - EOM were full with normal pursuit and saccade</FONT><FONT size=2>,  </FONT><FONT size=2>V - Motor V intact</FONT><FONT size=2>,  </FONT><FONT size=2>VII - No asymmetry or weakness</FONT><FONT size=2>,  </FONT><FONT size=2>VIII - Actuity intact to finger rub bilaterally</FONT><FONT size=2>,  </FONT><FONT size=2>IX , X- Palate rose in midile.</FONT><FONT size=2>,  </FONT><FONT size=2>XI - Sternocleidomastoid, trapezius strength intact.</FONT><FONT size=2>,  </FONT><FONT size=2>XII - Tongue protruded midline w/o atrophy or fasciculations</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>no afferent pupil defect</FONT><FONT size=2>,  </FONT><FONT size=2>No ptosis or nystagmus</FONT><FONT size=2>,  </FONT><FONT size=2>Pinprick, light touch intact in all three divisons</FONT><FONT size=2>,  </FONT><FONT size=2>I - Not Tested.</FONT><FONT size=2>,  </FONT><FONT size=2>II - Pupils 4mms reacting briskly to 2 mms</FONT><FONT size=2>,  </FONT><FONT size=2>III , IV, VI - EOM were full with normal pursuit and saccade</FONT><FONT size=2>,  </FONT><FONT size=2>V - Motor V intact</FONT><FONT size=2>,  </FONT><FONT size=2>VII - No asymmetry or weakness</FONT><FONT size=2>,  </FONT><FONT size=2>VIII - Actuity intact to finger rub bilaterally</FONT><FONT size=2>,  </FONT><FONT size=2>IX , X- Palate rose in midile.</FONT><FONT size=2>,  </FONT><FONT size=2>XI - Sternocleidomastoid, trapezius strength intact.</FONT><FONT size=2>,  </FONT><FONT size=2>XII - Tongue protruded midline w/o atrophy or fasciculations</FONT></innerxml-cdata></examNotes2><categoryPropId>12135</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>MOTOR STRENGTH:</categorySubName><examNotes><innerxml-cdata><FONT size=2>no cogwheeling</FONT><FONT size=2>,  </FONT><FONT size=2>no drift</FONT><FONT size=2>,  </FONT><FONT size=2>varicose veins bilaterally</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>no cogwheeling</FONT><FONT size=2>,  </FONT><FONT size=2>no drift</FONT><FONT size=2>,  </FONT><FONT size=2>varicose veins bilaterally</FONT></innerxml-cdata></examNotes2><categoryPropId>12136</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>SENSORY:</categorySubName><examNotes><innerxml-cdata><FONT size=2>decreased sensory in right hand, - Tinel</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>decreased sensory in right hand, - Tinel</FONT></innerxml-cdata></examNotes2><categoryPropId>12137</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>REFLEXES:</categorySubName><examNotes><innerxml-cdata><FONT size=2>bilaterally symmetrical, babinski negative</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>bilaterally symmetrical, babinski negative</FONT></innerxml-cdata></examNotes2><categoryPropId>12138</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>PLANTARS:</categorySubName><examNotes><innerxml-cdata><FONT size=2>downgoing bilaterally</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>downgoing bilaterally</FONT></innerxml-cdata></examNotes2><categoryPropId>12139</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>CEREBELLAR SIGNS:</categorySubName><examNotes><innerxml-cdata><FONT size=2>absent</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>absent</FONT></innerxml-cdata></examNotes2><categoryPropId>12140</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>TREMORS:</categorySubName><examNotes><innerxml-cdata><FONT size=2>absent</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>absent</FONT></innerxml-cdata></examNotes2><categoryPropId>12141</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>COORDINATION:</categorySubName><examNotes><innerxml-cdata><FONT size=2>finger-to-nose and rapid alternating movements were intact</FONT><FONT size=2>,  </FONT><FONT size=2>no ataxia</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>finger-to-nose and rapid alternating movements were intact</FONT><FONT size=2>,  </FONT><FONT size=2>no ataxia</FONT></innerxml-cdata></examNotes2><categoryPropId>12142</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>GAIT AND STATION:</categorySubName><examNotes><innerxml-cdata><FONT size=2>within normal limits</FONT><FONT size=2>,  </FONT><FONT size=2>Romberg was negative</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>within normal limits</FONT><FONT size=2>,  </FONT><FONT size=2>Romberg was negative</FONT></innerxml-cdata></examNotes2><categoryPropId>12143</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>SPEECH:</categorySubName><examNotes><innerxml-cdata><FONT size=2>normal</FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>normal</FONT></innerxml-cdata></examNotes2><categoryPropId>12144</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det><cat_det><categorySubName>PRONATOR DRIFT:</categorySubName><examNotes><innerxml-cdata><FONT size=2>not present </FONT></innerxml-cdata></examNotes><examNotes2><innerxml-cdata><FONT size=2>not present </FONT></innerxml-cdata></examNotes2><categoryPropId>18342</categoryPropId><encounterID>44037</encounterID><examId>191812</examId><docCount>0</docCount><document/><struct>0</struct></cat_det></categoryInDetail></category><structDataDisplay>2</structDataDisplay><item/></examination></subItems><subItems><subItemsName>Assessment:</subItemsName><assessment>Postconcussion syndrome - 310.2 (Primary)</assessment><assessment>Injury to brachial plexus - 953.4</assessment><assessment>Headache - 784.0</assessment><assessment>Memory loss - 780.93</assessment><assessment>Reactive confusion - 298.2</assessment><assessment>Spinal stenosis in cervical region - 723.0</assessment><notesBR><innerxml-cdata>43 year old female who is presenting to the office after a MVA 10-31-2013. Since the accident it seems the patient has developed a post-concussion state with chronic headaches and memory lost with sleep difficulty. <BR>Also subjective sensory lost and weakness in right hand, and complaints of severe pain in right shoulder and arm. <BR>Since initial office visit, the patient had a NCS study of the hands done, which showed bilateral median nerve dysfunction across both wrists.<BR>Patient also started on gabapentin 300mg PO BID for post-concussive state, with some help, but still has significant pain and discomfort. <BR>Patient's recent brain MRI negative, and C spine MRI showed significant degeneration and central stenosis. Please refer to official report for greater details. Patient's exam does not show any upper motor neuron signs, and no issues with bowel or urine. Patient still complains of chronic neck pain. <BR>EEG was normal.<BR>Plan<BR>-Neurotrax, cognitive testing <BR>-Continue Gabapentin 300mg PO QHS for three nights and then BID for pain. <BR>-Neurosurgery consult for cervical stenosis<BR>-Prednisone 5 day dose for cervical stenosis<BR>-Naproxen 500mg PO Q12 hours PRN for pain and discomfort.</innerxml-cdata></notesBR></subItems><subItems><subItemsName>Plan:</subItemsName><treatment><assessment><id>250009</id><name>Postconcussion syndrome</name><Referral><refId>1858</refId><P2P>1</P2P><RefName>Neurosurgery</RefName><Reason>Please assess for cervical stenosis and radiculopathy. Patient also has carpal tunnel in both wrist. </Reason></Referral></assessment><assessment><id>278390</id><name>Spinal stenosis in cervical region</name><rx>Start PredniSONE Tablet, 20 MG, 2 tablet with food or milk, Orally, Once a day, 5 days, 10 Tablet, Refills 0</rx><rx>Start Naproxen Tablet, 500 MG, 1 tablet as needed, Orally, every 12 hrs PRN for pain, 30 days, 60, Refills 2</rx></assessment></treatment><Immunization><itemName>Immunizations:</itemName></Immunization><Injection><itemName>Therapeutic Injections:</itemName></Injection><xrays/><labs/><procedures/><FollowUp>4 Weeks</FollowUp><disposition/></subItems><HL7ID>0</HL7ID><InvoiceId>3100</InvoiceId><InsuranceId>821</InsuranceId><InsuranceName>Care1st</InsuranceName><PayorID>57116</PayorID><SummeryView>false</SummeryView><CoSignedBy/></return></SOAP-ENV:Body></SOAP-ENV:Envelope>
	]]>
</progressnote>
"""

    MEDICALSUMMARY_XML = """<medicalsummary>
	<![CDATA[
	<?xml version="1.0" encoding="ISO-8859-1"?>
	<?xml-stylesheet href="/mobiledoc/jsp/catalog/xml/printPatientSummary.xsl" type="text/xsl"?><SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsd="http://www.w3.org/1999/XMLSchema" xmlns:xsi="http://www.w3.org/1999/XMLSchema-instance"><SOAP-ENV:Body><return><patient><EncDate>07/08/2014</EncDate><patientName>Abdelqader, Nawal</patientName><DOB>10/06/1970</DOB><Age>43 Y</Age><sex>Female</sex><phone>520-903-8423</phone><email></email><mobileno></mobileno><Address>14387 N. Banner Stone Crt, Marana, AZ, US 85658</Address><AccountNo>12406</AccountNo><WorkPhone></WorkPhone><Pcp></Pcp><insuranceName>Care1st</insuranceName><subscriberNo>A66760264</subscriberNo><PatientPicture>\mobiledoc/tempHubPics/12406.jpg</PatientPicture><Reminders><display>1</display><view>1</view></Reminders><PtReminders/><DxReminders/><RxReminders/><AllImmunizations><view>0</view><display>0</display></AllImmunizations><DistinctImmunizations><view>0</view><display>0</display></DistinctImmunizations><ProblemList><display>1</display><view>1</view><Problem><encId>36894</encId><asmtid>250009</asmtid><code>310.2</code><name>Postconcussion syndrome</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>02/25/2014</logdate><username>Sandoval, Emma</username><addeddate>02/03/2014</addeddate></Problem><Problem><encId>36894</encId><asmtid>263518</asmtid><code>953.4</code><name>Injury to brachial plexus</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>02/25/2014</logdate><username>Sandoval, Emma</username><addeddate>02/03/2014</addeddate></Problem><Problem><encId>36894</encId><asmtid>259457</asmtid><code>784.0</code><name>Headache</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>02/25/2014</logdate><username>Sandoval, Emma</username><addeddate>02/03/2014</addeddate></Problem><Problem><encId>36894</encId><asmtid>259117</asmtid><code>780.93</code><name>Memory loss</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>02/25/2014</logdate><username>Sandoval, Emma</username><addeddate>02/03/2014</addeddate></Problem><Problem><encId>36894</encId><asmtid>249690</asmtid><code>298.2</code><name>Reactive confusion</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>02/25/2014</logdate><username>Sandoval, Emma</username><addeddate>02/03/2014</addeddate></Problem><Problem><encId>38085</encId><asmtid>278369</asmtid><code>354.0</code><name>Carpal tunnel syndrome</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>02/25/2014</logdate><username>Sandoval, Emma</username><addeddate>02/25/2014</addeddate></Problem><Problem><encId>44037</encId><asmtid>278390</asmtid><code>723.0</code><name>Spinal stenosis in cervical region</name><specify></specify><notes></notes><onsetdate></onsetdate><logdate>07/02/2014</logdate><username>IBRAHIMI, SAID M</username><addeddate>07/01/2014</addeddate></Problem><NAP><NAPFlag>0</NAPFlag></NAP><NKP><NKPFlag>0</NKPFlag></NKP></ProblemList><Histories><display>1</display><view>1</view></Histories><CurrentMedicines><display>1</display><view>1</view><CurrentMedicine><IsFutureStartDate>No</IsFutureStartDate><MedicineName>Gabapentin 300 MG Capsule, Sig: 1 capsule Orally QHS for 1 week, then increase to 1 cap PO BID Start Date: 02/03/2014</MedicineName><DaysLeft>0</DaysLeft><StopDate></StopDate><OldRxId>179118</OldRxId></CurrentMedicine><CurrentMedicine><IsFutureStartDate>No</IsFutureStartDate><MedicineName>Hydromorphone HCl 2 MG Tablet, Sig: 1 tablet Orally Every 8 Hrs</MedicineName><DaysLeft>0</DaysLeft><StopDate></StopDate><OldRxId>179117</OldRxId></CurrentMedicine><CurrentMedicine><IsFutureStartDate>No</IsFutureStartDate><MedicineName>Levothyroxine Sodium 175 MCG Tablet, Sig: 1 tablet Orally Once a day</MedicineName><DaysLeft>0</DaysLeft><StopDate></StopDate><OldRxId>179116</OldRxId></CurrentMedicine><CurrentMedicine><IsFutureStartDate>No</IsFutureStartDate><MedicineName>Naproxen 500 MG Tablet, Sig: 1 tablet as needed Orally every 12 hrs PRN for pain Start Date: 07/01/2014</MedicineName><DaysLeft>0</DaysLeft><StopDate></StopDate><OldRxId>179182</OldRxId></CurrentMedicine><CurrentMedicine><IsFutureStartDate>No</IsFutureStartDate><MedicineName>PredniSONE 20 MG Tablet, Sig: 2 tablet with food or milk Orally Once a day Start Date: 07/01/2014</MedicineName><DaysLeft>0</DaysLeft><StopDate></StopDate><OldRxId>179181</OldRxId></CurrentMedicine></CurrentMedicines><Allergies><Allergy><AllergyName>Tramadol HCl</AllergyName><AllergyReaction></AllergyReaction></Allergy></Allergies><Vitals><display>1</display><view>1</view><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Temp</VitalName><VitalValue>97.8</VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>BP</VitalName><VitalValue>132/71</VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>HR</VitalName><VitalValue>62</VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Ht</VitalName><VitalValue>65</VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Wt</VitalName><VitalValue>232.4</VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>HC</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Hearing (Both)</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Pain scale</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Vision</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Wt %</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Ht-cm</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>Wt-kg</VitalName><VitalValue></VitalValue></Vital><Vital><VitalDate>07/01/2014</VitalDate><VitalName>BMI</VitalName><VitalValue>38.67</VitalValue></Vital></Vitals><Surgicals><display>1</display><view>1</view><Surgical><SurgicalReason>Thyroid Cancer</SurgicalReason><SurgicalDate>2010</SurgicalDate></Surgical><Surgical><SurgicalReason>Hemorrhoid</SurgicalReason><SurgicalDate>2004</SurgicalDate></Surgical></Surgicals><Hospitalizations><display>1</display><view>1</view></Hospitalizations><Encounters><display>1</display><view>1</view><Encounter><EncId>44103</EncId><Date>09/02/2014</Date><Summary>F/U</Summary><Status>2 month follow up</Status><Diagnosis/></Encounter><Encounter><EncId>44037</EncId><Date>07/01/2014</Date><Summary>F/U</Summary><Status>follow up; MRI results</Status><Diagnosis><Diag><Name>Postconcussion syndrome</Name><Code>310.2</Code></Diag><Diag><Name>Injury to brachial plexus</Name><Code>953.4</Code></Diag><Diag><Name>Headache</Name><Code>784.0</Code></Diag><Diag><Name>Memory loss</Name><Code>780.93</Code></Diag><Diag><Name>Reactive confusion</Name><Code>298.2</Code></Diag><Diag><Name>Spinal stenosis in cervical region</Name><Code>723.0</Code></Diag></Diagnosis></Encounter><Encounter><EncId>37661</EncId><Date>02/25/2014</Date><Summary>EEG</Summary><Status>EEG</Status><Diagnosis><Diag><Name>Postconcussion syndrome</Name><Code>310.2</Code></Diag><Diag><Name>Injury to brachial plexus</Name><Code>953.4</Code></Diag><Diag><Name>Headache</Name><Code>784.0</Code></Diag><Diag><Name>Memory loss</Name><Code>780.93</Code></Diag><Diag><Name>Reactive confusion</Name><Code>298.2</Code></Diag><Diag><Name>Carpal tunnel syndrome</Name><Code>354.0</Code></Diag></Diagnosis></Encounter><Encounter><EncId>38085</EncId><Date>02/24/2014</Date><Summary>EMG No Con</Summary><Status>Uppers</Status><Diagnosis><Diag><Name>Carpal tunnel syndrome</Name><Code>354.0</Code></Diag><Diag><Name>Postconcussion syndrome</Name><Code>310.2</Code></Diag><Diag><Name>Injury to brachial plexus</Name><Code>953.4</Code></Diag><Diag><Name>Headache</Name><Code>784.0</Code></Diag><Diag><Name>Reactive confusion</Name><Code>298.2</Code></Diag><Diag><Name>Memory loss</Name><Code>780.93</Code></Diag></Diagnosis></Encounter><Encounter><EncId>36894</EncId><Date>02/03/2014</Date><Summary>NP</Summary><Status>Post MVA</Status><Diagnosis><Diag><Name>Postconcussion syndrome</Name><Code>310.2</Code></Diag><Diag><Name>Injury to brachial plexus</Name><Code>953.4</Code></Diag><Diag><Name>Headache</Name><Code>784.0</Code></Diag><Diag><Name>Memory loss</Name><Code>780.93</Code></Diag><Diag><Name>Reactive confusion</Name><Code>298.2</Code></Diag></Diagnosis></Encounter></Encounters><Families><display>1</display><view>1</view><Family/></Families><Socials><display>1</display><view>1</view><Social><SocialName>Tobacco Use/Smoking</SocialName><SocialValue> Are you a: nonsmoker </SocialValue></Social><Social><SocialName>Caffeine:</SocialName><SocialValue>yes  frequency:, 1-2 cups per day Coffee/Soda</SocialValue></Social></Socials><Referrals><display>1</display><view>1</view><Incoming/><Outgoing><Referral><ReferralFrom>SAID M IBRAHIMI</ReferralFrom><ReferralTo>Brian P Callahan, MD</ReferralTo><StartDate>07/01/2014</StartDate><EndDate>07/01/2015</EndDate><Reason>Please assess for cervical stenosis and radiculopathy. Patient also has carpal tunnel in both wrist. </Reason><Date>07/01/2014</Date></Referral></Outgoing></Referrals><OBHistorys><display>1</display><view>1</view></OBHistorys><GYNHistorys><display>1</display><view>1</view></GYNHistorys></patient></return></SOAP-ENV:Body></SOAP-ENV:Envelope>
	]]>
</medicalsummary>
"""

    # Run tests
    test_xml("Progress Note", PROGRESSNOTE_XML)
    test_xml("Medical Summary", MEDICALSUMMARY_XML)



======= Testing Progress Note =======
Cleaned XML (first 300 chars):
<SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsd="http://www.w3.org/1999/XMLSchema" xmlns:xsi="http://www.w3.org/1999/XMLSchema-instance"><SOAP-ENV:Body><return><SectionTitle/><SectionSubTitle/><IndentStyle>bullet</IndentStyle><SubTitle>no</SubTitle><fax/><Guar ...

❌ Parse error: mismatched tag: line 1, column 6514

======= Testing Medical Summary =======
Cleaned XML (first 300 chars):
<SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsd="http://www.w3.org/1999/XMLSchema" xmlns:xsi="http://www.w3.org/1999/XMLSchema-instance"><SOAP-ENV:Body><return><patient><EncDate>07/08/2014</EncDate><patientName>Abdelqader, Nawal</patientName><DOB>10/06/1970</D ...

✅ Parsed successfully. Root tag: {http://schemas.xmlsoap.org/soap/envelope/}Envelope
Child tags: ['{http://schemas.xmlsoap.org/soap/envelope/}Body']


In [5]:
from lxml import etree

def debug_xml(xml_text):
    parser = etree.XMLParser(recover=False)
    try:
        etree.fromstring(xml_text.encode(), parser)
        print("✅ Well-formed XML")
    except etree.XMLSyntaxError as e:
        print("❌ XMLSyntaxError:", e)

# Test with cleaned Progress Note
cleaned_progress = clean_xml_before_deid(PROGRESSNOTE_XML)
debug_xml(cleaned_progress)


❌ XMLSyntaxError: Opening and ending tag mismatch: BR line 1 and innerxml-cdata, line 1, column 6530 (<string>, line 1)


In [18]:
text = """<?xml version='1.0' encoding='utf-8'?> <document id="1275284" documenttype="1" visitid="362300" patientid="20100784"><copyright> Copyright 2000-2003 Greenway Medical Technologies, Inc. All Rights Reserved.</copyright><templates servicecategory="1" /><version value="5" /><features><feature name="DiagnosisSecondaryCodes" /></features><cc ccnote=""><option value="Seizures" selected="true" /></cc><hpi hpinote="" /><ros><root Sig="" Key=""><system caption="TopLevel" uniqueid="1000" base="true" Copen="true" Fopen="false" Topen="false"><bullet caption="Constitutional" V="" uniqueid="1002" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1017"><concept caption="fatigue" enabled="true" uniqueid="0_7882304"><modifier caption="" type="presence" uniqueid="0_5190181" /></concept><concept caption="fever" enabled="true" uniqueid="0_79045"><modifier caption="" type="presence" uniqueid="0_7462457" /></concept><concept caption="chills" enabled="true" uniqueid="0_2884656"><modifier caption="" type="presence" uniqueid="0_8704853" /></concept><concept caption="malaise" enabled="true" uniqueid="0_9673781"><modifier caption="" type="presence" uniqueid="0_7100905" /></concept><concept caption="body aches" enabled="true" uniqueid="0_4130445"><modifier caption="(-)" type="presence" uniqueid="0_9484086" /></concept><concept caption="night sweats" enabled="true" uniqueid="0_3227593"><modifier caption="(-)" type="presence" uniqueid="0_3238784" /></concept><concept caption="weight loss" enabled="true" uniqueid="9_713811E_02"><modifier caption="" type="presence" uniqueid="0_3239843" /></concept><concept caption="weight gain" enabled="true" uniqueid="0_8118418"><modifier caption="" type="presence" uniqueid="0_2805812" /></concept><concept caption="loss of appetite" enabled="true" uniqueid="0_7649269"><modifier caption="" type="presence" uniqueid="0_5776981" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_4646875"><modifier caption="" type="presence" uniqueid="0_365379" /></concept><concept caption="Neurology Review of Systems 17.0 (Released 2012-10-31)" enabled="false" uniqueid="0_7389948"><modifier caption="" type="presence" uniqueid="0_6830822" /></concept></comprehensive></bullet><bullet caption="Eyes" V="" uniqueid="1003" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1019"><concept caption="discharge from eye" enabled="true" uniqueid="7_747829E_03"><modifier caption="" type="presence" uniqueid="0_3299481" /></concept><concept caption="eye discomfort" enabled="true" uniqueid="0_7818859"><modifier caption="(-)" type="presence" uniqueid="0_4612691" /></concept><concept caption="eye pain" enabled="true" uniqueid="0_322735"><modifier caption="" type="presence" uniqueid="0_6818615" /></concept><concept caption="double vision" enabled="true" uniqueid="0_9504022"><modifier caption="" type="presence" uniqueid="0_8386549" /></concept><concept caption="impaired vision" enabled="true" uniqueid="0_5142938"><modifier caption="(-)" type="presence" uniqueid="0_763435" /></concept><concept caption="blurred vision" enabled="true" uniqueid="0_9207357"><modifier caption="" type="presence" uniqueid="0_1721613" /></concept><concept caption="changes in vision" enabled="true" uniqueid="0_652569"><modifier caption="(-)" type="presence" uniqueid="0_4423596" /></concept><concept caption="floaters" enabled="true" uniqueid="0_9524195"><modifier caption="" type="presence" uniqueid="0_3396832" /></concept><concept caption="sudden visual loss" enabled="true" uniqueid="0_5201162"><modifier caption="" type="presence" uniqueid="0_3896449" /></concept><concept caption="visual hallucinations" enabled="true" uniqueid="0_5501651"><modifier caption="" type="presence" uniqueid="0_5247177" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_7555287"><modifier caption="" type="presence" uniqueid="0_7459071" /></concept><concept caption="cataracts" enabled="true" uniqueid="0_2547491"><modifier caption="(-)" type="presence" uniqueid="0_9577186" /></concept><concept caption="drooping eye lids" enabled="true" uniqueid="0_7159366"><modifier caption="(-)" type="presence" uniqueid="0_7774032" /></concept><concept caption="glaucoma" enabled="true" uniqueid="0_7721415"><modifier caption="(-)" type="presence" uniqueid="0_6880121" /></concept><concept caption="vision correction" enabled="true" uniqueid="0_927644"><modifier caption="(-)" type="presence" uniqueid="0_1845175" /></concept></comprehensive></bullet><bullet caption="HENT" V="" uniqueid="1004" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1021"><concept caption="recent head injury" enabled="true" uniqueid="0_3853899"><modifier caption="" type="presence" uniqueid="0_7442632" /></concept><concept caption="sinus pain" enabled="true" uniqueid="0_5709025"><modifier caption="" type="presence" uniqueid="0_3187066" /></concept><concept caption="nose bleeding" enabled="true" uniqueid="0_2101573"><modifier caption="" type="presence" uniqueid="0_9392723" /></concept><concept caption="dental problems" enabled="true" uniqueid="0_8778996"><modifier caption="" type="presence" uniqueid="0_5244244" /></concept><concept caption="neck stiffness" enabled="true" uniqueid="0_8923489"><modifier caption="" type="presence" uniqueid="0_1561252" /></concept><concept caption="neck pain" enabled="true" uniqueid="0_6620987"><modifier caption="" type="presence" uniqueid="0_6897714" /></concept><concept caption="sore throat" enabled="true" uniqueid="0_3145868"><modifier caption="" type="presence" uniqueid="0_8625561" /></concept><concept caption="loss of hearing" enabled="true" uniqueid="0_7127497"><modifier caption="(-)" type="presence" uniqueid="0_8868176" /></concept><concept caption="tinnitus" enabled="true" uniqueid="0_394338"><modifier caption="" type="presence" uniqueid="0_2201045" /></concept><concept caption="discharge from ears" enabled="true" uniqueid="0_385322"><modifier caption="" type="presence" uniqueid="0_899618" /></concept><concept caption="pressure in ears" enabled="true" uniqueid="0_5143839"><modifier caption="" type="presence" uniqueid="0_634625" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_9797892"><modifier caption="" type="presence" uniqueid="0_6005915" /></concept><concept caption="recent sinus infection" enabled="true" uniqueid="0_9580347"><modifier caption="(-)" type="presence" uniqueid="7_283747E_02" /></concept><concept caption="changes in hearing" enabled="true" uniqueid="7_065326E_02"><modifier caption="(-)" type="presence" uniqueid="0_4880335" /></concept><concept caption="mouth sores" enabled="true" uniqueid="0_6909098"><modifier caption="(-)" type="presence" uniqueid="0_6547297" /></concept><concept caption="difficulty swallowing or choking" enabled="true" uniqueid="0_2334238"><modifier caption="(-)" type="presence" uniqueid="0_5243802" /></concept><concept caption="snoring" enabled="true" uniqueid="0_2578952"><modifier caption="(-)" type="presence" uniqueid="0_1756974" /></concept></comprehensive></bullet><bullet caption="Breasts" V="" uniqueid="1007" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1023"><concept caption="lumps" enabled="true" uniqueid="0_7896017"><modifier caption="" type="presence" uniqueid="0_7629508" /></concept><concept caption="tenderness" enabled="true" uniqueid="0_69587"><modifier caption="" type="presence" uniqueid="0_7757134" /></concept><concept caption="swelling" enabled="true" uniqueid="0_5878798"><modifier caption="" type="presence" uniqueid="0_9337903" /></concept><concept caption="nipple discharge" enabled="true" uniqueid="0_7363299"><modifier caption="" type="presence" uniqueid="0_2453983" /></concept><concept caption="enlargement" enabled="true" uniqueid="6_778592E_02"><modifier caption="" type="presence" uniqueid="0_3241502" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_9583383"><modifier caption="" type="presence" uniqueid="0_6621366" /></concept></comprehensive></bullet><bullet caption="Cardiovascular" V="" uniqueid="1001" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1025"><concept caption="chest pain" enabled="true" uniqueid="0_655632"><modifier caption="(-)" type="presence" uniqueid="0_550114" /></concept><concept caption="irregular heart beats" enabled="true" uniqueid="0_6786185"><modifier caption="(-)" type="presence" uniqueid="0_2947531" /></concept><concept caption="rapid heart rate" enabled="true" uniqueid="0_1289055"><modifier caption="" type="presence" uniqueid="0_5162798" /></concept><concept caption="syncope" enabled="true" uniqueid="0_8721622"><modifier caption="(-)" type="presence" uniqueid="0_687999" /></concept><concept caption="loss of body hair on legs" enabled="true" uniqueid="0_8795871"><modifier caption="" type="presence" uniqueid="0_1145767" /></concept><concept caption="discoloration of legs" enabled="true" uniqueid="0_2762461"><modifier caption="" type="presence" uniqueid="0_6288233" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_9071261"><modifier caption="" type="presence" uniqueid="4_050457E_02" /></concept><concept caption="cold hands/feet" enabled="true" uniqueid="0_0595662"><modifier caption="(-)" type="presence" uniqueid="0_3224422" /></concept><concept caption="heart murmur" enabled="true" uniqueid="0_1592415"><modifier caption="(-)" type="presence" uniqueid="0_2100618" /></concept><concept caption="high blood pressure" enabled="true" uniqueid="6_144613E_02"><modifier caption="(-)" type="presence" uniqueid="0_4332575" /></concept></comprehensive></bullet><bullet caption="Respiratory" uniqueid="0_5429041" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1027"><concept caption="shortness of breath" enabled="true" uniqueid="0_8742839"><modifier caption="(-)" type="presence" uniqueid="0_4753351" /></concept><concept caption="wheezing" enabled="true" uniqueid="0_284425"><modifier caption="(-)" type="presence" uniqueid="0_9579245" /></concept><concept caption="cough" enabled="true" uniqueid="0_7886776"><modifier caption="(-)" type="presence" uniqueid="0_366309" /></concept><concept caption="hoarseness" enabled="true" uniqueid="8_403736E_02"><modifier caption="" type="presence" uniqueid="1_648474E_02" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_2228277"><modifier caption="" type="presence" uniqueid="0_3659828" /></concept><concept caption="pneumonia" enabled="true" uniqueid="0_6740897"><modifier caption="(-)" type="presence" uniqueid="0_8622944" /></concept><concept caption="upper respiratory infection" enabled="true" uniqueid="1_441616E_02"><modifier caption="(-)" type="presence" uniqueid="0_3453435" /></concept><concept caption="tuberculosis" enabled="true" uniqueid="0_1781841"><modifier caption="(-)" type="presence" uniqueid="0_6402946" /></concept></comprehensive></bullet><bullet caption="Gastrointestinal" V="106048009" uniqueid="1008" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1029"><concept caption="nausea" enabled="true" uniqueid="0_708906"><modifier caption="" type="presence" uniqueid="0_6831386" /></concept><concept caption="vomiting" enabled="true" uniqueid="0_2760047"><modifier caption="" type="presence" uniqueid="0_8882145" /></concept><concept caption="diarrhea" enabled="true" uniqueid="3_575283E_02"><modifier caption="" type="presence" uniqueid="0_749229" /></concept><concept caption="constipation" enabled="true" uniqueid="1_204532E_02"><modifier caption="" type="presence" uniqueid="0_2059716" /></concept><concept caption="loss of appetite" enabled="true" uniqueid="0_7807446"><modifier caption="" type="presence" uniqueid="0_9873981" /></concept><concept caption="heartburn" enabled="true" uniqueid="5_909503E_03"><modifier caption="" type="presence" uniqueid="0_7635636" /></concept><concept caption="reflux" enabled="true" uniqueid="0_1244013"><modifier caption="(-)" type="presence" uniqueid="0_6831127" /></concept><concept caption="excessive belching" enabled="true" uniqueid="0_870423"><modifier caption="" type="presence" uniqueid="0_3478795" /></concept><concept caption="abdominal pain" enabled="true" uniqueid="0_1513075"><modifier caption="" type="presence" uniqueid="2_052188E_03" /></concept><concept caption="jaundice" enabled="true" uniqueid="0_6629756"><modifier caption="(-)" type="presence" uniqueid="0_5460274" /></concept><concept caption="blood in stools" enabled="true" uniqueid="0_781872"><modifier caption="" type="presence" uniqueid="0_9607006" /></concept><concept caption="hemorrhoids" enabled="true" uniqueid="0_397027"><modifier caption="" type="presence" uniqueid="0_3425306" /></concept><concept caption="bloating" enabled="true" uniqueid="0_7470514"><modifier caption="" type="presence" uniqueid="0_9089123" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_5536441"><modifier caption="" type="presence" uniqueid="7_678068E_02" /></concept></comprehensive></bullet><bullet caption="Genitourinary" V="" uniqueid="1009" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1031"><concept caption="urgency" enabled="true" uniqueid="0_3889315"><modifier caption="" type="presence" uniqueid="8_092844E_02" /></concept><concept caption="frequency" enabled="true" uniqueid="0_1440366"><modifier caption="" type="presence" uniqueid="0_9749148" /></concept><concept caption="change in urine color" enabled="true" uniqueid="0_5660245"><modifier caption="" type="presence" uniqueid="0_4788486" /></concept><concept caption="urinary retention" enabled="true" uniqueid="0_6626785"><modifier caption="(-)" type="presence" uniqueid="0_7817819" /></concept><concept caption="decreased stream" enabled="true" uniqueid="6_012326E_02"><modifier caption="" type="presence" uniqueid="0_4947212" /></concept><concept caption="possible pregnancy" enabled="true" uniqueid="0_8571741"><modifier caption="" type="presence" uniqueid="0_5941693" /></concept><concept caption="hot flashes" enabled="true" uniqueid="0_7907308"><modifier caption="(-)" type="presence" uniqueid="0_437176" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_4365092"><modifier caption="" type="presence" uniqueid="7_409096E_03" /></concept><concept caption="prostate problems" enabled="true" uniqueid="0_7274304"><modifier caption="(-)" type="presence" uniqueid="5_359077E_02" /></concept><concept caption="sexually transmitted disease" enabled="true" uniqueid="0_1365721"><modifier caption="(-)" type="presence" uniqueid="0_640741" /></concept><concept caption="kidney stones" enabled="true" uniqueid="5_639583E_02"><modifier caption="(-)" type="presence" uniqueid="0_4624465" /></concept><concept caption="incontinence" enabled="true" uniqueid="0_9727492"><modifier caption="(-)" type="presence" uniqueid="0_1939772" /></concept></comprehensive></bullet><bullet caption="Integument" V="" uniqueid="1012" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1033"><concept caption="rash" enabled="true" uniqueid="7_972354E_02"><modifier caption="" type="presence" uniqueid="6_714594E_02" /></concept><concept caption="itching" enabled="true" uniqueid="9_602845E_03"><modifier caption="(-)" type="presence" uniqueid="0_9055598" /></concept><concept caption="pigmentation changes" enabled="true" uniqueid="0_7352061"><modifier caption="" type="presence" uniqueid="0_6312159" /></concept><concept caption="skin dryness" enabled="true" uniqueid="0_9758402"><modifier caption="" type="presence" uniqueid="0_2815061" /></concept><concept caption="hair growth change" enabled="true" uniqueid="3_339261E_02"><modifier caption="" type="presence" uniqueid="0_7085801" /></concept><concept caption="nail changes" enabled="true" uniqueid="7_797605E_02"><modifier caption="" type="presence" uniqueid="0_2224128" /></concept><concept caption="new skin lesions" enabled="true" uniqueid="0_241062"><modifier caption="(-)" type="presence" uniqueid="0_733155" /></concept><concept caption="changes to existing skin lesions or moles" enabled="true" uniqueid="0_1865024"><modifier caption="(-)" type="presence" uniqueid="0_9750808" /></concept><concept caption="hirsutism" enabled="false" uniqueid="0_2146018"><modifier caption="" type="presence" uniqueid="0_4751332" /></concept><concept caption="acne" enabled="true" uniqueid="0_7248376"><modifier caption="" type="presence" uniqueid="0_6914961" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="3_195304E_02"><modifier caption="" type="presence" uniqueid="0_6840903" /></concept></comprehensive></bullet><bullet caption="Neurologic" V="" uniqueid="1013" base="true" Copen="true" Fopen="false" Topen="false" freetext="&lt;list&gt;&lt;item&gt;&lt;caption&gt;Admits&lt;/caption&gt;&lt;content&gt;speech difficulties, seizures, headache/migraine, Memory loss&lt;/content&gt;&lt;/item&gt;&lt;/list&gt;" ftb="0"><comprehensive uniqueid="1035"><concept caption="difficulty concentrating" enabled="true" uniqueid="0_3509837"><modifier caption="" type="presence" uniqueid="0_4358606" /></concept><concept caption="memory difficulties" enabled="true" uniqueid="0_6843557"><modifier caption="" type="presence" uniqueid="0_1739038" /></concept><concept caption="speech difficulties" enabled="true" uniqueid="0_5958058" defaultcharge=""><modifier caption="(+)" type="presence" uniqueid="0_5730983" /></concept><concept caption="seizures" enabled="true" uniqueid="0_718729" defaultcharge=""><modifier caption="(+)" type="presence" uniqueid="0_3784012" /></concept><concept caption="black out spells" enabled="true" uniqueid="0_2961385"><modifier caption="" type="presence" uniqueid="0_8627254" /></concept><concept caption="tremors" enabled="true" uniqueid="0_341325"><modifier caption="" type="presence" uniqueid="0_3028494" /></concept><concept caption="incoordination" enabled="true" uniqueid="0_240969"><modifier caption="" type="presence" uniqueid="0_3960316" /></concept><concept caption="loss of balance" enabled="true" uniqueid="0_7100796"><modifier caption="" type="presence" uniqueid="0_3677281" /></concept><concept caption="tingling or numbness" enabled="true" uniqueid="0_2478049"><modifier caption="" type="presence" uniqueid="0_3755238" /></concept><concept caption="falls" enabled="true" uniqueid="0_3877072"><modifier caption="" type="presence" uniqueid="0_4754704" /></concept><concept caption="head injuries" enabled="true" uniqueid="3_780323E_02"><modifier caption="" type="presence" uniqueid="0_3114371" /></concept><concept caption="focal seizures" enabled="true" uniqueid="0_7883919"><modifier caption="" type="presence" uniqueid="0_5726106" /></concept><concept caption="slowness of movements" enabled="true" uniqueid="0_315214"><modifier caption="" type="presence" uniqueid="0_4735897" /></concept><concept caption="tics" enabled="true" uniqueid="0_3651076"><modifier caption="" type="presence" uniqueid="0_3876295" /></concept><concept caption="muscle cramps" enabled="true" uniqueid="0_3932449"><modifier caption="" type="presence" uniqueid="9_435523E_02" /></concept><concept caption="difficulty sleeping" enabled="true" uniqueid="0_610907"><modifier caption="" type="presence" uniqueid="0_3937615" /></concept><concept caption="loud snoring at night" enabled="true" uniqueid="0_5192581"><modifier caption="" type="presence" uniqueid="0_1580882" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_7378715"><modifier caption="(-)" type="presence" uniqueid="0_9834492" /></concept><concept caption="numbness" enabled="true" uniqueid="4_960281E_02"><modifier caption="(-)" type="presence" uniqueid="0_7923958" /></concept><concept caption="burning sensation" enabled="true" uniqueid="0_0541901"><modifier caption="(-)" type="presence" uniqueid="0_7357873" /></concept><concept caption="confusion" enabled="true" uniqueid="0_4775558"><modifier caption="(-)" type="presence" uniqueid="0_33007" /></concept><concept caption="muscle weakness" enabled="true" uniqueid="0_2265679"><modifier caption="(-)" type="presence" uniqueid="0_1124526" /></concept><concept caption="dizziness/vertigo" enabled="true" uniqueid="0_9161633"><modifier caption="(-)" type="presence" uniqueid="0_5584371" /></concept><concept caption="lightheaded" enabled="true" uniqueid="0_1588983"><modifier caption="(-)" type="presence" uniqueid="0_7131666" /></concept><concept caption="staring spells" enabled="true" uniqueid="0_1709061"><modifier caption="(-)" type="presence" uniqueid="0_8735542" /></concept><concept caption="headache/migraine" enabled="true" uniqueid="0_2255098" defaultcharge="(-)"><modifier caption="(+)" type="presence" uniqueid="0_7274433" /></concept></comprehensive></bullet><bullet caption="Musculoskeletal" V="106048009" uniqueid="1011" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1037"><concept caption="joint pain" enabled="true" uniqueid="0_771603"><modifier caption="(-)" type="presence" uniqueid="0_1831578" /></concept><concept caption="joint swelling" enabled="true" uniqueid="0_5792043"><modifier caption="(-)" type="presence" uniqueid="0_622903" /></concept><concept caption="muscle pain" enabled="true" uniqueid="0_3366161"><modifier caption="" type="presence" uniqueid="0_1493735" /></concept><concept caption="limitation of motion" enabled="true" uniqueid="0_6887048"><modifier caption="" type="presence" uniqueid="0_7998946" /></concept><concept caption="muscular weakness" enabled="true" uniqueid="0_2572798"><modifier caption="" type="presence" uniqueid="0_4497603" /></concept><concept caption="muscle cramps" enabled="true" uniqueid="0_7666785"><modifier caption="" type="presence" uniqueid="0_674738" /></concept><concept caption="back pain" enabled="true" uniqueid="0_3741261"><modifier caption="" type="presence" uniqueid="0_2381374" /></concept><concept caption="recent joint injuries" enabled="true" uniqueid="0_3565472"><modifier caption="" type="presence" uniqueid="0_4404104" /></concept><concept caption="recent soft tissue injuries" enabled="true" uniqueid="0_9036035"><modifier caption="" type="presence" uniqueid="0_307176" /></concept><concept caption="recent neck injury" enabled="true" uniqueid="0_9045029"><modifier caption="" type="presence" uniqueid="0_1258888" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_7134518"><modifier caption="" type="presence" uniqueid="6_073654E_02" /></concept></comprehensive></bullet><bullet caption="Endocrine" uniqueid="0_7244793" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1039"><concept caption="loss of hair" enabled="true" uniqueid="0_9375769"><modifier caption="" type="presence" uniqueid="0_1620427" /></concept><concept caption="constipation" enabled="true" uniqueid="0_8340309"><modifier caption="" type="presence" uniqueid="0_9200685" /></concept><concept caption="cold intolerance" enabled="true" uniqueid="0_2406264"><modifier caption="(-)" type="presence" uniqueid="0_7923231" /></concept><concept caption="heat intolerance" enabled="true" uniqueid="8_541411E_02"><modifier caption="(-)" type="presence" uniqueid="0_2990977" /></concept><concept caption="central obesity" enabled="true" uniqueid="0_728301"><modifier caption="" type="presence" uniqueid="4_206789E_02" /></concept><concept caption="weight gain" enabled="true" uniqueid="7_287413E_02"><modifier caption="" type="presence" uniqueid="0_5016346" /></concept><concept caption="weight loss" enabled="true" uniqueid="0_3393156"><modifier caption="" type="presence" uniqueid="0_3918854" /></concept><concept caption="acne" enabled="true" uniqueid="0_5824751"><modifier caption="" type="presence" uniqueid="0_4741744" /></concept><concept caption="hot flashes" enabled="true" uniqueid="0_4165155"><modifier caption="" type="presence" uniqueid="0_93177" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_5945049"><modifier caption="" type="presence" uniqueid="0_7038321" /></concept></comprehensive></bullet><bullet caption="Psychiatric" V="" uniqueid="1014" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1041"><concept caption="anxiety" enabled="true" uniqueid="9_679192E_02"><modifier caption="(-)" type="presence" uniqueid="0_3611556" /></concept><concept caption="depression" enabled="true" uniqueid="0_5919293"><modifier caption="(-)" type="presence" uniqueid="0_2655141" /></concept><concept caption="hallucinations" enabled="true" uniqueid="0_1401674"><modifier caption="" type="presence" uniqueid="0_3428456" /></concept><concept caption="feeling confused" enabled="true" uniqueid="0_3016114"><modifier caption="" type="presence" uniqueid="0_1028985" /></concept><concept caption="difficulty sleeping" enabled="true" uniqueid="4_809719E_02"><modifier caption="" type="presence" uniqueid="0_6028209" /></concept><concept caption="compulsive behaviors" enabled="true" uniqueid="0_8175152"><modifier caption="" type="presence" uniqueid="0_5887563" /></concept><concept caption="moodiness" enabled="true" uniqueid="0_109131"><modifier caption="" type="presence" uniqueid="0_1547047" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_9314534"><modifier caption="" type="presence" uniqueid="0_5192806" /></concept><concept caption="hostility" enabled="true" uniqueid="0_4613844"><modifier caption="(-)" type="presence" uniqueid="0_3033891" /></concept><concept caption="nervousness" enabled="true" uniqueid="0_5321967"><modifier caption="(-)" type="presence" uniqueid="0_5188085" /></concept><concept caption="restless" enabled="true" uniqueid="0_7273691"><modifier caption="(-)" type="presence" uniqueid="0_0339992" /></concept><concept caption="personality change" enabled="true" uniqueid="4_672945E_03"><modifier caption="(-)" type="presence" uniqueid="0_3584598" /></concept><concept caption="incresed stress" enabled="true" uniqueid="0_6871749"><modifier caption="(-)" type="presence" uniqueid="0_9683027" /></concept></comprehensive></bullet><bullet caption="Heme-Lymph" V="" uniqueid="1010" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1043"><concept caption="lightheadedness" enabled="true" uniqueid="0_8114102"><modifier caption="(-)" type="presence" uniqueid="0_6134188" /></concept><concept caption="easy bleeding" enabled="true" uniqueid="0_2717852"><modifier caption="(-)" type="presence" uniqueid="4_100978E_02" /></concept><concept caption="easy bruising" enabled="true" uniqueid="0_6878648"><modifier caption="(-)" type="presence" uniqueid="0_3212132" /></concept><concept caption="lymph node enlargement or tenderness" enabled="true" uniqueid="0_2662554"><modifier caption="" type="presence" uniqueid="0_9908266" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_1312264"><modifier caption="" type="presence" uniqueid="0_6066072" /></concept><concept caption="anemia" enabled="true" uniqueid="1_060027E_02"><modifier caption="(-)" type="presence" uniqueid="0_1641642" /></concept></comprehensive></bullet><bullet caption="Allergic-Immunologic" V="106048009" uniqueid="1006" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1045"><concept caption="sinus allergy symptoms" enabled="true" uniqueid="0_8679124"><modifier caption="" type="presence" uniqueid="0_2669106" /></concept><concept caption="allergic dermatitis" enabled="true" uniqueid="0_5923159"><modifier caption="" type="presence" uniqueid="0_9864002" /></concept><concept caption="frequent illnesses" enabled="true" uniqueid="0_1718088"><modifier caption="(-)" type="presence" uniqueid="0_1154752" /></concept><concept caption="additional symptoms, except as noted in HPI" enabled="true" uniqueid="0_9343099"><modifier caption="" type="presence" uniqueid="0_6271662" /></concept></comprehensive></bullet></system><allnegative checked="false" /></root></ros><rosv2><bullet caption="Constitutional" V="" uniqueid="1002" base="true" check="0" /><bullet caption="Eyes" V="" uniqueid="1003" base="true" check="0" /><bullet caption="HENT" V="" uniqueid="1004" base="true" check="0" /><bullet caption="Breasts" V="" uniqueid="1007" base="true" check="0" /><bullet caption="Cardiovascular" V="" uniqueid="1001" base="true" check="0" /><bullet caption="Respiratory" uniqueid="0_5429041" base="true" check="0" /><bullet caption="Gastrointestinal" V="106048009" uniqueid="1008" base="true" check="0" /><bullet caption="Genitourinary" V="" uniqueid="1009" base="true" check="0" /><bullet caption="Integument" V="" uniqueid="1012" base="true" check="0" /><bullet caption="Neurologic" V="" uniqueid="1013" base="true" check="0" /><bullet caption="Musculoskeletal" V="106048009" uniqueid="1011" base="true" check="0" /><bullet caption="Endocrine" uniqueid="0_7244793" base="true" check="0" /><bullet caption="Psychiatric" V="" uniqueid="1014" base="true" check="0" /><bullet caption="Heme-Lymph" V="" uniqueid="1010" base="true" check="0" /><bullet caption="Allergic-Immunologic" V="106048009" uniqueid="1006" base="true" check="0" /></rosv2><docvitals vitaltable="&#10;&lt;STYLE&gt;&#10; .DataCell&#10; {&#10; text-align:left;&#10; padding-left:0px;&#10; cursor:hand;&#10; font-size:12px;&#10; font-family:tahoma;&#10; line-height:20px;&#10; }&#10; .subheadVitals&#10; {&#10; color:#707c88;&#10; }&#10; &lt;/STYLE&gt;&#10;&#10;&lt;TABLE style=&quot;BORDER-BOTTOM: #999999 0px solid; BORDER-LEFT: #999999 0px solid; TABLE-LAYOUT: fixed; FONT-SIZE: 8pt; BORDER-TOP: #999999 0px solid; BORDER-RIGHT: #999999 0px solid&quot; class=DataCell border=0 cellSpacing=0 cellPadding=2 dwidth=&quot;890px&quot;&gt;&#10;&lt;THEAD&gt;&#10;&lt;COLGROUP&gt;&#10;&lt;COL align=left width=65&gt;&#10;&lt;COL align=left width=55&gt;&#10;&lt;COL align=left width=48&gt;&#10;&lt;COL align=left width=52&gt;&#10;&lt;COL align=left width=55&gt;&#10;&lt;COL align=left width=37&gt;&#10;&lt;COL align=left width=60&gt;&#10;&lt;COL align=left width=50&gt;&#10;&lt;COL align=left width=31&gt;&#10;&lt;COL align=left width=62&gt;&#10;&lt;COL align=left width=55&gt;&#10;&lt;COL align=left width=58&gt;&#10;&lt;COL align=left width=45&gt;&#10;&lt;COL align=left width=45&gt;&#10;&lt;COL align=left width=42&gt;&#10;&lt;COL id=HC1 align=left width=50&gt;&lt;/COLGROUP&gt;&lt;/THEAD&gt;&#10;&lt;TBODY&gt;&#10;&lt;TR&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;Date&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;Time&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;BP&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;Position&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;Site&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;L\R&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;Cuff Size&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;HR&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;RR&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;TEMP(F)&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;WT &lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;HT &lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;BMI kg/m&lt;SUP&gt;2&lt;/SUP&gt;&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;BSA m&lt;SUP&gt;2&lt;/SUP&gt;&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;O2 Sat&lt;/TD&gt;&#10;&lt;TD class=subheadVitals vAlign=bottom&gt;HC &lt;/TD&gt;&lt;/TR&gt;&#10;&lt;TR style=&quot;BACKGROUND-COLOR: #666666; HEIGHT: 1px&quot;&gt;&#10;&lt;TD colSpan=16&gt;&lt;/TD&gt;&lt;/TR&gt;&lt;/TBODY&gt;&lt;/TABLE&gt;&#10;&lt;TABLE style=&quot;BORDER-BOTTOM: #999999 0px solid; BORDER-LEFT: #999999 0px solid; TABLE-LAYOUT: fixed; FONT-SIZE: 8pt; BORDER-TOP: #999999 0px solid; BORDER-RIGHT: #999999 0px solid&quot; dir=ltr id=VitalsContent class=DataCell border=0 cellSpacing=0 cellPadding=2 dwidth=&quot;890px&quot;&gt;&#10;&lt;THEAD&gt;&#10;&lt;COLGROUP&gt;&#10;&lt;COL align=left width=65&gt;&#10;&lt;COL align=left width=55&gt;&#10;&lt;COL align=left width=48&gt;&#10;&lt;COL align=left width=52&gt;&#10;&lt;COL align=left width=55&gt;&#10;&lt;COL align=left width=37&gt;&#10;&lt;COL align=left width=60&gt;&#10;&lt;COL align=left width=50&gt;&#10;&lt;COL align=left width=31&gt;&#10;&lt;COL align=left width=62&gt;&#10;&lt;COL align=left width=55&gt;&#10;&lt;COL align=left width=58&gt;&#10;&lt;COL align=left width=45&gt;&#10;&lt;COL align=left width=45&gt;&#10;&lt;COL align=left width=42&gt;&#10;&lt;COL id=HC2 align=left width=50&gt;&lt;/COLGROUP&gt;&lt;/THEAD&gt;&#10;&lt;TBODY dir=ltr&gt;&#10;&lt;TR style=&quot;&quot; id=138148 language=vbscript ondblclick=&quot;VitalSetEditClicked me&quot; selected=&quot;true&quot; isorthostatic=&quot;0&quot; clinicalvitalgroupid=&quot;137597&quot; visitid=&quot;362300&quot;&gt;&#10;&lt;TD class=printclass title=&quot;Recorded By: kjackson&quot; vAlign=top&gt;2016-12-25&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;09:52 AM&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;118/62&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;Sitting&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;70 - &lt;FONT style=&quot;FONT-FAMILY: tahoma&quot;&gt;R&lt;/FONT&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;204lbs 0oz&lt;/TD&gt;&#10;&lt;TD vAlign=top&gt;5' &amp;nbsp;6&quot;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;32.93&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;2.08&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&#10;&lt;TD class=printclass vAlign=top&gt;&lt;/TD&gt;&lt;/TR&gt;&lt;/TBODY&gt;&lt;/TABLE&gt;&lt;BR&gt;"><clinicalvitalgroup id="137597" /></docvitals><pe version="3" sex="1"><root><s c="Constitutional" id="008" b="1"><b c="Appearance" id="009" st="0" Co="1"><bd e="1"><cp id="14673" ft="well-nourished, well developed, in no acute distress" fd="well-nourished, well developed, in no acute distress" fm="7596, 7603, 7617" df="f"><c c="state of nutrition" e="1" id="13864"><v c="well-nourished" d="1" id="7596" /><v c="cachectic" id="7598" /><v c="obese" id="7599" /><v c="morbidly obese" id="7600" /><v c="overweight" id="7601" /><mds><a><g id="13018" c="Severity"><md c="severely" id="2" /><md c="moderately" id="3" /><md c="mildly" id="4" /></g><g id="F7548B46-5C4B-4BD8-86E7-974D493F7152" c="color"><md c="red" id="0F8D4F47-9D02-48B6-93CB-3C7A6E927D02" /><md c="blue" id="87C196FE-9E0F-41AF-B428-4FBB66B8F2CD" /><md c="green" id="B2C3F0D9-0D0D-42D5-8D65-5073980BE6F7" /></g></a><l /><m /></mds><v c="File Version: Neurology Physical Exam ((IPADDRESS)) (Released 2005-10-31)" id="7602" /><v c="under-weight" id="7CA6923E-D104-417D-9BD0-869451F94BB6" /></c><c c="state of development" e="1" id="13865"><v c="well developed" d="1" id="7603" /><v c="under developed" id="7604" /><v c="poorly developed" id="7605" /><v c="large body habitus" id="7606" /><v c="small body habitus" id="7607" /><v c="normal body habitus" id="7608" /></c><c c="level of consciousness" e="2" id="13866"><v c="alert" d="1" id="7609" /><v c="lethargic" id="7610" /><v c="stuporous" id="7611" /><v c="obtunded" id="7612" /><v c="comatose" id="7613" /><v c="inebriated" id="7614" /><v c="drowsy" id="7615" /><v c="inattentive" id="7616" /></c><c c="level of distress" e="1" id="13867"><v c="in no acute distress" d="1" id="7617" /><v c="in no distress" id="7618" /><v c="in acute distress" id="7619" /><v c="in obvious distress" id="7620" /><v c="in respiratory distress" id="7621" /></c><c c="personal hygiene" e="2" id="13868"><v c="well-tended appearance" d="1" id="7622" /><v c="clean appearance" id="7623" /><v c="dirty appearance" id="7624" /><v c="disheveled appearance" id="7625" /><v c="bizarre appearance" id="7626" /><v c="poor hygiene" id="7627" /></c><c c="appearance relative to age" e="2" id="13869"><v c="appears about reported age" d="1" id="7628" /><v c="appears younger than reported age" id="7629" /><v c="appears older than reported age" id="7630" /></c><c c="body habitus" e="2" id="13870"><v c="normal body habitus" d="1" id="7631" /><v c="small body habitus" id="7632" /><v c="large body habitus" id="7633" /></c><c c="deformities" e="2" id="13871"><v c="no obvious deformities present" d="1" id="7634" /><v c="limb deformities present" id="7635" /><v c="skull deformity present" id="7636" /><v c="trunk deformity present" id="7637" /><v c="limb deformity present" id="7638" /><v c="multiple deformities present" id="7639" /><v c="dysmorphic appearance present" id="7640" /><v c="facial deformity present" id="7641" /><v c="no deformities present" id="7642" /><v c="no gross deformities present" id="7643" /></c><c c="parent child interaction" e="2" id="13872"><v c="parent child interaction appropriate for child's age" d="1" id="7644" /><v c="parent child interaction strained" id="7645" /><v c="parent child interaction appears unhealthy" id="7646" /><v c="parent child interaction appears abnormal" id="7647" /><v c="parent appears to be emotionally detached from child" id="7648" /><v c="child appears to be emotionally detached from parent" id="7649" /><v c="care-giver child interactions normal" id="7650" /><v c="care-giver child interactions abnormal" id="7651" /></c><c c="Glasgow Coma Scale" e="2" id="13873"><v c="Glasgow Coma Scale" d="1" id="7652" /><v c="Glasgow Coma Score: 3/15" id="7653" /><v c="Glasgow Coma Score: 4" id="7654" /><v c="Glasgow Coma Score: 5" id="7655" /><v c="Glasgow Coma Score: 6" id="7656" /><v c="Glasgow Coma Score: 7" id="7657" /><v c="Glasgow Coma Score: 8 " id="7658" /><v c="Glasgow Coma Score: 9" id="7659" /><v c="Glasgow Coma Score: 10" id="7660" /><v c="Glasgow Coma Score: 11" id="7661" /><v c="Glasgow Coma Score: 12" id="7662" /><v c="Glasgow Coma Score: 13" id="7663" /><v c="Glasgow Coma Score: 14" id="7664" /><v c="Glasgow Coma Score: 15" id="7665" /><mds><a /><l><g id="13019" c="Best Eye Response"><md c="no eye opening" id="5" /><md c="eye opening to pain" id="6" /><md c="eye opening to verbal command" id="7" /><md c="eye opening spontaneous" id="8" /></g><g id="13020" c="Best Verbal Response"><md c="No verbal response" id="9" /><md c="Best Verbal Response: Incomprehensible sounds" id="10" /><md c="Best Verbal Response: Inappropriate words" id="11" /><md c="Best Verbal Response: Confused " id="12" /><md c="Best Verbal Response: Oriented" id="13" /></g><g id="13021" c="Best Motor Response"><md c="Best Motor Response: No motor response" id="14" /><md c="Best Motor Response: Extension to pain" id="15" /><md c="Best Motor Response: Flexion to pain" id="16" /><md c="Best Motor Response: Localizes to pain" id="17" /><md c="Best Motor Response: Obeys commands" id="18" /></g></l><m /></mds></c><c c="posture" e="2" id="13874"><v c="normal posture" d="1" id="7666" /><v c="poor posture" id="7667" /><v c="abnormal posture" id="7668" /><v c="stooped posture" id="7669" /><v c="rigid posture" id="7670" /></c><c c="facial expressions" e="2" id="13875"><v c="facial expressions normal" d="1" id="7671" /><v c="facial expression reduced" id="7672" /><v c="mask-like facies" id="7673" /><v c="patient making bizarre facial expressions" id="7674" /></c><c c="eye contact" e="2" id="13876"><v c="eye contact normal" d="1" id="7675" /><v c="eye contact poor" id="7676" /><v c="eye contact excessive" id="7677" /></c><c c="clothing" e="2" id="13877"><v c="clothing appropriate for age and situation" d="1" id="7678" /><v c="clothing soiled" id="7679" /><v c="clothing inappropriate for age and situation" id="7680" /><v c="clothing torn" id="7681" /><v c="clothing disorganized" id="7682" /></c><c c="motor activity" e="2" id="13878"><v c="general level of motor activity normal" d="1" id="7683" /><v c="general level of motor activity increased" id="7684" /><v c="akathisia present" id="7685" /><v c="general level of motor activity reduced" id="7686" /><v c="patient hypokinetic" id="7687" /><v c="patient hyperkinetic" id="7688" /></c><c c="level of cooperation" e="2" id="13879"><v c="cooperative during history and examination" d="1" id="7689" /><v c="patient uncooperative" id="7690" /><v c="patient angry during examination" id="7691" /><v c="patient refused to provide history" id="7692" /><v c="patient refuses to cooperate with examination" id="7693" /></c></cp><mp><el id="00002" idt="b" e="1" /><el id="00063" idt="b" e="1" /><el id="00090" idt="b" e="1" /><el id="00131" idt="b" e="1" /><el id="00167" idt="b" e="1" /><el id="00188" idt="b" e="1" /><el id="00228" idt="b" e="1" /><el id="00253" idt="b" e="1" /><el id="00268" idt="b" e="1" /><el id="00292" idt="b" e="1" /></mp></bd></b></s><s c="Neck" id="022" b="1"><b c="Neck" id="023" st="0" Co="1"><bd e="1"><cp id="14710" ft="neck is non-tender to palpation" fd="normal appearance, no masses or tenderness" df="f"><c c="neck finding" e="1" id="13976"><v c="supple" d="1" id="8397" /><v c="tenderness present" id="8398" /><v c="swelling present" id="8399" /><v c="laceration present" id="8400" /><v c="mass present" id="8401" /><v c="nodule present" id="8402" /><v c="erythema present" id="8403" /><v c="diffuse atrophy present" id="8404" /><v c="atrophy present" id="8405" /><mds><a><g id="13117" c="Severity (appears before finding)"><md c="mild" id="829" /><md c="moderate" id="830" /><md c="severe" id="831" /><md c="mildly" id="832" /><md c="moderately" id="833" /><md c="severely" id="834" /></g><g id="13118" c="Lesion/Mass modifiers (before finding)"><md c="tender" id="835" /><md c="nontender" id="836" /><md c="soft" id="837" /><md c="firm" id="838" /><md c="cystic" id="839" /><md c="fluctuant" id="840" /><md c="indurated" id="841" /><md c="lobulated" id="842" /><md c="multilobulated" id="843" /><md c="mobile" id="844" /><md c="nonmobile" id="845" /></g><g id="13119" c="Laceration modifier (before finding)"><md c="linear" id="846" /><md c="stellate" id="847" /><md c="healed" id="848" /><md c="healing" id="849" /></g></a><l><g c="Location (appear after finding)" id="13120"><md c="right paravertebral region" id="850" /><md c="left paravertebral region" id="851" /><md c="right anterior cervical region" id="852" /><md c="left anterior cervical region" id="853" /><md c="right upper anterior cervical region" id="854" /><md c="left upper anterior cervical region" id="855" /><md c="right lower anterior cervical region" id="856" /><md c="left lower anterior cervical region" id="857" /><md c="right upper posterior cervical region" id="858" /><md c="left upper posterior cervical region" id="859" /><md c="right lower posterior cervical region" id="860" /><md c="left lower posterior cervical region" id="861" /><md c="midline anterior cervical region" id="862" /><md c="midline posterior cervical region" id="863" /></g></l><m><g c="Size" id="13121" /></m></mds><v c="tracheal deviation present" id="8406" /></c><c c="neck deformities" e="2" id="13977"><v c="no deformities" d="1" id="8407" /><v c="webbing present" id="8408" /><v c="cervical scoliosis present" id="8409" /><v c="cervical kyphosis present" id="8410" /><v c="cervical kyphoscoliosis present" id="8411" /><v c="torticollis present" id="8412" /><v c="wry neck present" id="8413" /><v c="deformity present" id="8414" /><v c="post-surgical deformity present" id="8415" /><mds><a><g id="13122" c="Severity (appears before finding)"><md c="mild" id="864" /><md c="moderate" id="865" /><md c="severe" id="866" /></g><g id="13123" c="Laterality (appears before finding)"><md c="right-sided" id="867" /><md c="left-sided" id="868" /><md c="right anterior" id="869" /><md c="left anterior" id="870" /><md c="right posterior" id="871" /><md c="left posterior" id="872" /></g></a><l /><m /></mds></c><c c="Brudzinski's sign" e="2" id="13978"><v c="Brudzinski's sign absent" d="1" id="8416" /><v c="Brudzinski's sign present" id="8417" /></c></cp><mp><el id="00012" idt="b" e="1" /><el id="00106" idt="b" e="1" /><el id="00132" idt="b" e="1" /><el id="00174" idt="b" e="1" /><el id="00272" idt="b" e="1" /></mp></bd></b><b c="Range of Motion" id="056-0-5682917-0-3651548" st="0" Co="1"><bd e="1"><cp id="14711" ft="range of motion within normal limits" fd="cervical range of motion within normal limits" df="f"><c c="range of motion" e="1" id="13979"><v c="range of motion within normal limits" d="1" id="8418" /><v c="range of motion reduced" id="8419" /><v c="reduction in range of motion present" id="8420" /><mds><a><g id="13124" c="Severity (appears before finding)"><md c="mild" id="873" /><md c="moderate" id="874" /><md c="severe" id="875" /></g></a><l /><m><g c="Range of motion measurement" id="13125"><md c="extension" id="876" /><md c="flexion" id="877" /><md c="right lateroflexion" id="878" /><md c="left lateroflexion" id="879" /><md c="right rotation" id="880" /><md c="left rotation" id="881" /><md c="5" id="882" /><md c="10" id="883" /><md c="15" id="884" /><md c="20" id="885" /><md c="25" id="886" /><md c="30" id="887" /><md c="35" id="888" /><md c="40" id="889" /><md c="45" id="890" /><md c="50" id="891" /><md c="55" id="892" /><md c="60" id="893" /><md c="65" id="894" /><md c="70" id="895" /><md c="75" id="896" /><md c="80" id="897" /><md c="85" id="898" /><md c="90" id="899" /><md c="95" id="900" /><md c="100" id="901" /><md c="105" id="902" /><md c="110" id="903" /><md c="115" id="904" /><md c="120" id="905" /><md c="125" id="906" /><md c="130" id="907" /><md c="135" id="908" /><md c="140" id="909" /><md c="145" id="910" /><md c="150" id="911" /><md c="degrees" id="912" /></g></m></mds></c><c c="Brudzinski's sign" e="2" id="13980"><v c="Brudzinski's sign absent" d="1" id="8421" /><v c="Brudzinski's reflex present" id="8422" /></c></cp><mp><el id="00047" idt="b" e="1" /><el id="00092" idt="b" e="1" /><el id="00093" idt="b" e="1" /><el id="00193" idt="b" e="1" /></mp></bd></b></s><s c="Musculoskeletal" id="055" b="1"><b c="Right Upper Extremity" id="060" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14776" ft="" fd="" df="f" /><mp /></bd><b c="Range of Motion" id="060-0-4505171-0-7502062-0-204944-0-5057983" st="0" Co="1"><bd e="1"><cp id="14779" ft="range of motion normal, no pain with joint motion" fd="range of motion normal, no joint crepitus or pain with motion present" df="f"><c c="range of motion" e="1" id="14142"><v c="range of motion normal" d="1" id="9509" /><v c="range of motion restricted" id="9510" /><v c="range of motion mildly restricted" id="9511" /><v c="range of motion moderately restricted" id="9512" /><v c="range of motion severely restricted" id="9513" /><v c="joint frozen" id="9514" /><v c="range of motion" id="9515" /><mds><a><g id="13413" c="Joint (appears before finding)"><md c="shoulder joint" id="3352" /><md c="elbow joint" id="3353" /><md c="wrist joint" id="3354" /></g></a><l /><m><g c="ROM measurement (appears after finding)" id="13414" /></m></mds></c><c c="crepitus" e="1" id="14143"><v c="no joint crepitus present" d="1" id="9516" /><v c="shoulder crepitus present" id="9517" /><v c="elbow crepitus present" id="9518" /><v c="wrist crepitus present" id="9519" /><v c="crepitus present" id="9520" /><mds><a><g id="13415" c="Severity (appears before finding)"><md c="mild" id="3355" /><md c="moderate" id="3356" /><md c="severe" id="3357" /></g></a><l /><m /></mds></c><c c="pain with motion" e="1" id="14144"><v c="no pain with joint motion" d="1" id="9521" /><v c="pain at elbow with motion present" id="9522" /><v c="pain at shoulder with motion present" id="9523" /><mds><a><g id="13416" c="Severity"><md c="mild" id="3358" /><md c="moderate" id="3359" /><md c="severe" id="3360" /></g></a><l /><m /></mds><v c="pain at wrist with motion present" id="9524" /><v c="pain at MP joints with motion present" id="9525" /><v c="pain at PIP joints with motion present" id="9526" /><v c="pain at DIP joints with motion present" id="9527" /></c></cp><mp><el id="00201" idt="b" e="1" /><el id="00049" idt="b" e="1" /></mp></bd></b></b><b c="Left Upper Extremity" id="060-0-5317606" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14780" ft="" fd="" df="f" /><mp /></bd><b c="Range of Motion" id="060-0-4505171-0-7502062-0-204944-0-5057983-0-7942944" st="0" Co="1"><bd e="1"><cp id="14783" ft="range of motion normal, no pain with joint motion" fd="range of motion normal, no joint crepitus present, no pain with joint motion" df="f"><c c="range of motion" e="1" id="14152"><v c="range of motion normal" d="1" id="9556" /><v c="range of motion restricted" id="9557" /><v c="range of motion mildly restricted" id="9558" /><v c="range of motion moderately restricted" id="9559" /><v c="range of motion severely restricted" id="9560" /><v c="joint frozen" id="9561" /><v c="range of motion" id="9562" /><mds><a><g id="13428" c="Joint (appears before finding)"><md c="shoulder joint" id="3425" /><md c="elbow joint" id="3426" /><md c="wrist joint" id="3427" /></g></a><l /><m><g c="ROM measurement (appears after finding)" id="13429" /></m></mds></c><c c="crepitus" e="1" id="14153"><v c="no joint crepitus present" d="1" id="9563" /><v c="shoulder crepitus present" id="9564" /><v c="elbow crepitus present" id="9565" /><v c="wrist crepitus present" id="9566" /><v c="crepitus present" id="9567" /><mds><a><g id="13430" c="Severity (appears before finding)"><md c="mild" id="3428" /><md c="moderate" id="3429" /><md c="severe" id="3430" /></g></a><l /><m /></mds></c><c c="pain with motion" e="1" id="14154"><v c="no pain with joint motion" d="1" id="9568" /><v c="pain at elbow with motion present" id="9569" /><v c="pain at shoulder with motion present" id="9570" /><mds><a><g id="13431" c="Severity"><md c="mild" id="3431" /><md c="moderate" id="3432" /><md c="severe" id="3433" /></g></a><l /><m /></mds><v c="pain at wrist with motion present" id="9571" /><v c="pain at MP joints with motion present" id="9572" /><v c="pain at PIP joints with motion present" id="9573" /><v c="pain at DIP joints with motion present" id="9574" /></c></cp><mp><el id="00205" idt="b" e="1" /><el id="00050" idt="b" e="1" /></mp></bd></b></b><b c="Right Lower Extremity" id="062" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14784" ft="" fd="" df="f" /><mp /></bd><b c="Range of Motion" id="062-0-1446046-0-9158503" st="0" Co="1"><bd e="1"><cp id="14787" ft="range of motion normal, no pain on motion" fd="range of motion normal, no joint crepitations present, no pain on motion" df="f"><c c="range of motion" e="1" id="14159"><v c="range of motion normal" d="1" id="9600" /><v c="range of motion restriction" id="9601" /><v c="joint frozen" id="9602" /><v c="range of motion" id="9603" /><mds><a><g id="13439" c="Severity (appears before finding)"><md c="mild" id="3492" /><md c="moderate" id="3493" /><md c="severe" id="3494" /></g></a><l><g c="Lower Extremity Joint" id="13440"><md c="hip" id="3495" /><md c="knee" id="3496" /><md c="ankle" id="3497" /></g></l><m><g c="ROM measurement (after finding)" id="13441" /></m></mds></c><c c="crepitus" e="1" id="14160"><v c="no joint crepitations present" d="1" id="9604" /><v c="hip crepitus present" id="9605" /><v c="knee crepitus present" id="9606" /><v c="ankle crepitus present" id="9607" /><mds><a><g id="13442" c="Severity (appears before finding)"><md c="mild" id="3498" /><md c="moderate" id="3499" /><md c="severe" id="3500" /></g></a><l /><m /></mds></c><c c="pain" e="1" id="14161"><v c="no pain on motion" d="1" id="9608" /><v c="shoulder pain with motion present" id="9609" /><v c="elbow pain with motion present" id="9610" /><v c="wrist pain with motion present" id="9611" /></c><c c="straight leg raise test" e="2" id="F9D5AF70-9916-4A61-A082-52606971B738"><v c="straight leg raise test negative" d="1" id="5D8EEA7B-F402-4409-8C6C-4C4562E12E72" /><v c="straight leg raise test positive at 30 degrees" id="3A28796C-E5F6-489A-8454-14F9F7DC806D" /><v c="straight leg raise test positive at 45 degrees" id="BBA8DF9E-651D-4A11-B3C5-10B07884C470" /><v c="straight leg raise test positive at 90 degrees" id="40A2BB24-A8DC-46F9-B1F2-43399C782E41" /><v c="straight leg raise test positive" id="63992CE0-E6C1-4320-B636-776C360396E4" /></c></cp><mp><el id="00051" idt="b" e="1" /><el id="00209" idt="b" e="1" /></mp></bd></b></b><b c="Left Lower Extremity" id="062-0-3632695" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14788" ft="" fd="" df="f" /><mp /></bd><b c="Range of Motion" id="062-0-1446046-0-9158503-0-3676513" st="0" Co="1"><bd e="1"><cp id="14791" ft="range of motion normal, no pain on motion" fd="range of motion normal, no joint crepitations present, no pain on motion" df="f"><c c="range of motion" e="1" id="14166"><v c="range of motion normal" d="1" id="9637" /><v c="range of motion restriction" id="9638" /><v c="joint frozen" id="9639" /><v c="range of motion" id="9640" /><mds><a><g id="13450" c="Severity (appears before finding)"><md c="mild" id="3559" /><md c="moderate" id="3560" /><md c="severe" id="3561" /></g></a><l><g c="Lower Extremity Joint" id="13451"><md c="hip" id="3562" /><md c="knee" id="3563" /><md c="ankle" id="3564" /></g></l><m><g c="ROM measurement (after finding)" id="13452" /></m></mds></c><c c="crepitus" e="1" id="14167"><v c="no joint crepitations present" d="1" id="9641" /><v c="hip crepitus present" id="9642" /><v c="knee crepitus present" id="9643" /><v c="ankle crepitus present" id="9644" /><mds><a><g id="13453" c="Severity (appears before finding)"><md c="mild" id="3565" /><md c="moderate" id="3566" /><md c="severe" id="3567" /></g></a><l /><m /></mds></c><c c="pain" e="1" id="14168"><v c="no pain on motion" d="1" id="9645" /><v c="shoulder pain with motion present" id="9646" /><v c="elbow pain with motion present" id="9647" /><v c="wrist pain with motion present" id="9648" /></c><c c="straight leg raise test" e="2" id="C42F0D02-FC47-4CEA-B264-08D81FDE112D"><v c="straight leg raise test negative" d="1" id="22304E69-467E-4D28-A34A-6DD439C7230A" /><v c="straight leg raise test positive at 30 degrees" id="50063E1F-1AAD-4678-B7AE-422E047DA2E4" /><v c="straight leg raise test positive at 45 degrees" id="7BFBF397-957D-485A-86B5-C2F8E2D798D0" /><v c="straight leg raise test positive at 90 degrees" id="E61CEB57-97BB-4D89-B248-0ACCEAA6CD0C" /><v c="straight leg raise test positive" id="53C51660-FD7F-405B-948B-18E04056D36E" /></c></cp><mp><el id="00052" idt="b" e="1" /><el id="00213" idt="b" e="1" /></mp></bd></b></b></s><s c="Neurologic" id="067" b="1"><b c="Mental Status Examination" id="9C8E5A82-815A-4713-AFB5-4EC608F0B38C" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14809" fd="" ft="" fm="" df="f" /><mp /></bd><b c="Orientation" id="073" st="0" Co="1"><bd e="1"><cp id="14810" ft="oriented to person, place and time" fd="awake and alert, cooperative with the exam" fm="10681, DCCC24E0-5D91-4755-B80E-1635DA120A3A" df="f"><c c="general orientation" e="1" id="14233"><v c="awake and alert" d="1" id="10681" /><v c="oriented times 3" id="10682" /><v c="oriented X 3" id="10683" /><v c="oriented to person, place and time" id="10684" /><v c="grossly oriented" id="10685" /><v c="oriented" id="10686" /></c><c c="orientation to person" e="2" id="14234"><v c="oriented to person" d="1" id="10687" /><v c="disoriented to person" id="10688" /><v c="not oriented to person" id="10689" /><v c="does not know own name" id="10690" /><v c="does not know who others are" id="10691" /></c><c c="orientation to place" e="2" id="14235"><v c="oriented to place" d="1" id="10692" /><v c="disoriented to place" id="10693" /><v c="not oriented to place" id="10694" /><v c="not fully oriented to place" id="10695" /></c><c c="orientation to time" e="2" id="14236"><v c="oriented to time" d="1" id="10696" /><v c="disoriented to time" id="10697" /><v c="not oriented to time" id="10698" /></c><c c="cooperative with the exam" e="1" id="EF73F1B0-63AF-4ADC-BC72-0D9961580B1D"><v c="cooperative with the exam" d="1" id="DCCC24E0-5D91-4755-B80E-1635DA120A3A" /></c></cp><mp><el id="00059" idt="b" e="1" /><el id="00236" idt="b" e="1" /><el id="00087" idt="b" e="1" /><el id="00114" idt="b" e="1" /><el id="00128" idt="b" e="1" /><el id="00164" idt="b" e="1" /><el id="00185" idt="b" e="1" /><el id="00225" idt="b" e="1" /><el id="00261" idt="b" e="1" /><el id="00289" idt="b" e="1" /><el id="00314" idt="b" e="1" /></mp></bd></b><b c="Attention" id="1123FB32-13A5-48A5-AB98-056F73E3EB42" st="0" Co="1"><bd e="1"><cp id="14811" ft="attention normal, concentration abilities normal" fd="attention normal, concentration abilities normal" df="f"><c c="attention" e="1" id="14237"><v c="attention normal" d="1" id="10699" /><v c="attention normal on serial 7's" id="10700" /><v c="in attentive" id="10701" /><v c="easily distracted" id="10702" /></c><c c="concentration" e="1" id="14238"><v c="concentration abilities normal" d="1" id="10703" /><v c="concentration abilities poor" id="10704" /><v c="concentration mildly impaired" id="10705" /><v c="vigilance poor" id="10706" /><v c="vigilance normal" id="10707" /><v c="able to remain vigilant for over 30 seconds" id="10708" /><v c="able to remain vigilant for 20-30 seconds" id="10709" /><v c="able to remain vigilant for 10-20 seconds" id="10710" /><v c="able to remain vigilant for 5-10 seconds" id="10711" /></c><c c="neglect" e="1" id="14239"><v c="no evidence of neglect" d="1" id="10712" /><v c="no neglect present" id="10713" /><v c="neglect objects on right side" id="10714" /><v c="neglects object on left side" id="10715" /></c><c c="random letter test" e="2" id="14240"><v c="random letter test: no errors" d="1" id="10716" /><v c="random letter test: one error of ommision" id="10717" /><v c="random letter test: two errors of omission" id="10718" /><v c="random letter test: three errors of omission" id="10719" /><v c="random letter test: one error of comission" id="10720" /><v c="random letter test: two errors of comission" id="10721" /><v c="random letter test: three errors of comission" id="10722" /><v c="perseveration error present" id="10723" /></c><c c="tactile double simultaneous stimulation" e="2" id="14241"><v c="tactile double simultaneous stimulation: no extinction" d="1" id="10724" /><v c="no errors" id="10725" /><v c="extinction present on right" id="10726" /><v c="extinction present on left" id="10727" /></c><c c="auditory double simultaneous stimulation" e="2" id="14242"><v c="auditory double simultaneous stimulation: no extinction" d="1" id="10728" /><v c="auditory double simultaneous stimulation: extinction present on the right" id="10729" /><v c="auditory double simultaneous stimulation: extinction present on the left" id="10730" /></c><c c="visual double simultaneous stimulation" e="2" id="14243"><v c="visual double simultaneous stimulation: no extinction" d="1" id="10731" /><v c="double simultaneous stimulation: extinction present on the right" id="10732" /><v c="double simultaneous stimulation: extinction present on the left" id="10733" /></c><c c="clock-drawing" e="2" id="14244"><v c="clock-drawing normal" d="1" id="10734" /><v c="right-sided neglect present on clock-drawing " id="10735" /><v c="left-sided neglect present on clock-drawing " id="10736" /></c><c c="flower-drawing" e="2" id="14245"><v c="flower drawing normal" d="1" id="10737" /><v c="right-sided neglect present on flower-drawing" id="10738" /><v c="left-sided neglect present on flower-drawing" id="10739" /></c></cp><mp><el id="00238" idt="b" e="1" /><el id="00263" idt="b" e="1" /></mp></bd></b><b c="Language" id="((DRIVERSLICENSE))-4713-4A99-9083-226AF1471A49" st="0" Co="1"><bd e="1"><cp id="14812" ft="normal fluency, no evidence of aphasia or dysarthria" fd="no evidence of aphasia, speech development normal for age" fm="10833, 10861" df="f"><c c="communication ability" e="2" id="14246"><v c="communication ability within normal limits" d="1" id="10740" /><v c="communication ability impaired" id="10741" /><v c="patient refuses to communicate" id="10742" /><v c="patient unable to speak" id="10743" /><v c="speech unintelligible" id="10744" /><v c="communicates nonverbally through sign language" id="10745" /><v c="communication ability within normal limits for age" id="10746" /></c><c c="voice quality" e="2" id="14247"><v c="voice quality normal" d="1" id="10747" /><v c="voice quality hoarse" id="10748" /><v c="voice quality poor" id="10749" /><v c="voice quality weak" id="10750" /><v c="spastic dysphonia present" id="10751" /></c><c c="spontaneity" e="2" id="14248"><v c="speech spontaneity normal" d="1" id="10752" /><v c="lack of spontaneity of speech present" id="10753" /></c><c c="rate" e="2" id="14249"><v c="rate of speech normal" d="1" id="10754" /><v c="rate of speech rapid" id="10755" /><v c="rate of speech slow" id="10756" /><v c="bradylalia present" id="10757" /></c><c c="fluency" e="2" id="14250"><v c="normal fluency" d="1" id="10758" /><v c="language output nonfluent" id="10759" /><v c="fluency impaired" id="10760" /><v c="fluency mildly impaired" id="10761" /><v c="fluency moderately impaired" id="10762" /><v c="fluency markedly impaired" id="10763" /><v c="output sparse and effortful with substantive words" id="10764" /><mds><a /><l><g id="13658" c="Additional fluency descriptors"><md c="agrammatic" id="6015" /><md c="," id="6016" /><md c="and" id="6017" /><md c="contains frequent word finding pauses" id="6018" /></g></l><m /></mds><v c="bradylalia present" id="10765" /><v c="palilalia present" id="10766" /></c><c c="volume" e="2" id="14251"><v c="volume of speech normal" d="1" id="10767" /><v c="speech volume reduced" id="10768" /><v c="volume of speech increased" id="10769" /></c><c c="articulation" e="2" id="14252"><v c="articulation of speech normal" d="1" id="10770" /><v c="speech articulation poor" id="10771" /><v c="speech articulation moderately impaired" id="10772" /><v c="dysarthria present" id="10773" /><v c="labial dysarthria present" id="10774" /><v c="dental dysarthria present" id="10775" /><v c="lingual dysarthria present" id="10776" /><v c="guttural dysarthria present" id="10777" /><v c="rhinolalia present" id="10778" /><v c="scanning dysarthria" id="10779" /></c><c c="melody" e="2" id="14253"><v c="melody of speech normal" d="1" id="10780" /><v c="dysprosody present" id="10781" /><v c="mild dysprosody present" id="10782" /><v c="moderate dysprosody present" id="10783" /><v c="marked dysprosody present" id="10784" /></c><c c="dyslalia" e="2" id="14254"><v c="no dyslalia" d="1" id="10785" /><v c="dyslalia present" id="10786" /></c><c c="coherence" e="2" id="14255"><v c="speech coherent" d="1" id="10787" /><v c="speech at time incoherent" id="10788" /><v c="speech incoherent" id="10789" /></c><c c="comprehension" e="2" id="14256"><v c="comprehension normal" d="1" id="10790" /><v c="auditory and written comprehension impaired" id="10791" /><v c="comprehension normal for written tasks" id="10792" /><v c="able to comprehend &quot;whole-body&quot; tasks" id="10793" /><v c="able to point to four objects (sequentially)" id="10794" /><v c="able to point to three objects (sequentially)" id="10795" /><v c="able to point to two objects (sequentially)" id="10796" /><v c="able to point to one object" id="10797" /><v c="able to follow multi-step commands" id="10798" /><v c="unable to follow multi-step commands" id="10799" /></c><c c="object naming" e="2" id="14257"><v c="object naming normal" d="1" id="10800" /><v c="anomia present" id="10801" /><v c="mild anomia present" id="10802" /><v c="moderate anomia present" id="10803" /><v c="marked anomia present" id="10804" /></c><c c="alexia" e="2" id="14258"><v c="reading normal" d="1" id="10805" /><v c="mild alexia present" id="10806" /><v c="moderate alexia present" id="10807" /><v c="marked alexia present" id="10808" /><v c="alexia without agraphia present" id="10809" /></c><c c="agraphia" e="2" id="14259"><v c="writing normal" d="1" id="10810" /><v c="agraphia present" id="10811" /><v c="mild agraphia present" id="10812" /><v c="moderate agraphia present" id="10813" /><v c="severe agraphia present" id="10814" /></c><c c="repetition" e="2" id="14260"><v c="repetition normal for words and phrases" d="1" id="10815" /><v c="repetition for single words normal" id="10816" /><v c="repetition for single words impaired" id="10817" /><v c="word repetition markedly impaired" id="10818" /><v c="able to repeat &quot;no ifs, ands or buts&quot;" id="10819" /><v c="able to repeat two-word phrases" id="10820" /><v c="able to repeat three-word phrases" id="10821" /><v c="able to repeat four-word phrases" id="10822" /><v c="able to repeat complex phrases" id="10823" /><v c="word repetition:" id="10824" /><v c="phrase repetition:" id="10825" /><mds><a /><l><g id="13659" c="Repetition ability modifiers"><md c="mildly" id="6019" /><md c="moderately" id="6020" /><md c="markedly" id="6021" /><md c="impaired" id="6022" /><md c="unable to repeat" id="6023" /><md c="single words" id="6024" /><md c="single words without frequent errors" id="6025" /><md c="two word phrases" id="6026" /><md c="three word phrases" id="6027" /><md c="four word phrases" id="6028" /><md c="complex phrases" id="6029" /></g></l><m /></mds></c><c c="perseveration" e="2" id="14261"><v c="no perseverations" d="1" id="10826" /><v c="perseverations present" id="10827" /></c><c c="digit repetition" e="2" id="14262"><v c="digit repetition: 9 digits without error" d="1" id="10828" /><v c="digit repetition: " id="10829" /><v c="USE MODIFIERS AS NEEDED" id="10830" /><mds><a /><l><g id="13660" c="Number of digits repeated"><md c="9" id="6030" /><md c="8" id="6031" /><md c="7" id="6032" /><md c="6" id="6033" /><md c="5" id="6034" /><md c="4" id="6035" /><md c="3" id="6036" /><md c="2" id="6037" /></g><g id="13661" c="Error rate"><md c="without errors" id="6038" /><md c="with one error" id="6039" /><md c="with two errors" id="6040" /><md c="with three errors" id="6041" /><md c="with four errors" id="6042" /><md c="with multiple errors" id="6043" /></g></l><m /></mds><v c="patient unable to perform digit repetition task" id="10831" /><v c="patient uncooperative with digit repetition task" id="10832" /></c><c c="aphasia" e="1" id="14263"><v c="no evidence of aphasia" d="1" id="10833" /><v c="expressive aphasia present" id="10834" /><v c="conductive aphasia present" id="10835" /><v c="global aphasia present" id="10836" /><v c="receptive aphasia present" id="10837" /><mds><a><g id="13662" c="Severity"><md c="mild" id="6044" /><md c="moderate" id="6045" /><md c="severe" id="6046" /><md c="stable" id="6047" /><md c="unchanged" id="6048" /><md c="improving" id="6049" /><md c="worsening" id="6050" /></g></a><l /><m /></mds><v c="transcortical motor aphasia present" id="10838" /><v c="transcortical sensory aphasia present" id="10839" /><v c="anomic aphasia present" id="10840" /></c><c c="paraphrasias" e="2" id="14264"><v c="no paraphrasias used" d="1" id="10841" /><v c="paraphrasias present" id="10842" /><v c="occasional paraphrasias present" id="10843" /><v c="word substitutions present" id="10844" /></c><c c="echolalia" e="2" id="14265"><v c="no echolalia" d="1" id="10845" /><v c="echolalia present" id="10846" /></c><c c="neologisms" e="2" id="14266"><v c="no neologisms" d="1" id="10847" /><v c="neologisms present" id="10848" /></c><c c="word-finding" e="2" id="14267"><v c="word-finding ability normal" d="1" id="10849" /><v c="word-finding ability impaired" id="10850" /><v c="word-finding ability mildly impaired" id="10851" /><v c="word-finding ability moderately impaired" id="10852" /><v c="word-finding ability markedly impaired" id="10853" /></c><c c="comprehension (developmental)" e="2" id="14268"><v c="level of language comprehension normal for age" d="1" id="10854" /><v c="level of language comprehension delayed for age" id="10855" /><v c="level of language comprehension accelerated for age" id="10856" /><v c="understands basic spacial relationships" id="10857" /><v c="understands simple sentences" id="10858" /><v c="understands meaning of nouns" id="10859" /><v c="understands complex sentences" id="10860" /></c><c c="speech development" e="1" id="14269"><v c="speech development normal for age" d="1" id="10861" /><v c="cooing present" id="10862" /><v c="babbling present" id="10863" /><v c="able to vocalize single words" id="10864" /><v c="able to vocalize two word phrases" id="10865" /><v c="able to vocalize sentences" id="10866" /><v c="vowel sounds present" id="10867" /><v c="consonant sounds present" id="10868" /><v c="able to say &quot;dada&quot;" id="10869" /><v c="able to say &quot;mama&quot;" id="10870" /><v c="child speaking &quot;gibberish&quot;" id="10871" /><v c="three word sentences present" id="10872" /><v c="four word sentences present" id="10873" /></c><c c="stuttering" e="2" id="14270"><v c="no stuttering" d="1" id="10874" /><v c="stuttering present" id="10875" /></c><c c="stammering" e="2" id="14271"><v c="no stammering" d="1" id="10876" /><v c="stammering present" id="10877" /></c><c c="cluttering" e="2" id="14272"><v c="no cluttering" d="1" id="10878" /><v c="cluttering present" id="10879" /></c><c c="lisping" e="2" id="14273"><v c="no lisping" d="1" id="10880" /><v c="lisping present" id="10881" /></c></cp><mp><el id="00091" idt="b" e="1" /><el id="00239" idt="b" e="1" /></mp></bd></b><b c="Memory" id="074" st="0" Co="1"><bd e="1"><cp id="14813" ft="short term and long term memory intact" fd="recent and remote memory grossly intact, cognitive function is normal" fm="10882, 6CC1BFD6-0675-4E6C-BB8C-A744EE267BA4" df="f"><c c="general" e="1" id="14274"><v c="recent and remote memory grossly intact" d="1" id="10882" /><v c="memory impaired" id="10883" /><v c="intact" id="10884" /><v c="poor recall" id="10885" /></c><c c="immediate object recall" e="2" id="14275"><v c="patient able to recall 3/3 items correctly after 5 minutes" d="1" id="10886" /><v c="patient able to recall of 3/3 items correctly after 5 minutes" id="10887" /><v c="immediate recall poor" id="10888" /><v c="patient able to recall of 4/4 items correctly after 5 minutes" id="10889" /><v c="patient able to recall 2/3 items correctly after 5 minutes" id="10890" /><v c="patient able to recall 1/3 items correctly after 5 minutes" id="10891" /><v c="patient able to recall 0/3 items correctly after 5 minutes" id="10892" /></c><c c="immediate story recall" e="2" id="14276"><v c="immediate story recall normal" d="1" id="10893" /><v c="immediate story recall impaired" id="10894" /></c><c c="visual memory" e="2" id="14277"><v c="visual memory for 3/3 objects normal" d="1" id="10895" /><v c="patient able to recall 2/3 items on visual memory testing" id="10896" /><v c="patient able to recall 1/3 items on visual memory testing" id="10897" /><v c="patient able to recall 0/3 items on visual memory testing" id="10898" /></c><c c="visual design reproduction" e="2" id="14278"><v c="visual design reproduction normal" d="1" id="10899" /><v c="visual design reproduction impaired" id="10900" /></c><c c="paired associated learning" e="2" id="14279"><v c="paired associated learning normal" d="1" id="10901" /><v c="paired associated learning impaired" id="10902" /></c><c c="recent memory" e="2" id="14280"><v c="recent memory intact for recent events" d="1" id="10903" /><v c="recent memory impaired" id="10904" /><v c="recent memory mildly impaired" id="10905" /><v c="recent memory moderately impaired" id="10906" /><v c="recent memory markedly impaired" id="10907" /></c><c c="remote memory" e="2" id="14281"><v c="remote memory intact for significant life events" d="1" id="10908" /><v c="unable to remember current president" id="10909" /><v c="unable to recall past 2 presidents" id="10910" /><v c="unable to recall past 3 presidents" id="10911" /><v c="long-term recall poor" id="10912" /><v c="long-term recall impaired" id="10913" /></c><c c="cognitive function" e="1" id="4ED3BCFE-D987-4A9B-B611-849814CA0AB3"><v c="cognitive function is normal" d="1" id="6CC1BFD6-0675-4E6C-BB8C-A744EE267BA4" /></c></cp><mp><el id="00060" idt="b" e="1" /><el id="00237" idt="b" e="1" /><el id="00262" idt="b" e="1" /></mp></bd></b></b><b c="Cranial Nerves" id="068" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14816" ft="" fd="cranial nerves intact bilaterally" df="f"><c c="general" e="1" id="14298"><v c="cranial nerves intact bilaterally" d="1" id="10968" /><v c="vision, eye movements, pupillary response, facial sensation, facial muscle strength, hearing, gag reflex, shoulder shrug and tongue movements within normal limits" id="10969" /><v c="cranial nerves II-XII intact" id="10970" /><v c="cranial nerves I-XII intact" id="10971" /><v c="cranial nerves grossly intact" id="10972" /></c><c c="smell" e="2" id="14299"><v c="sense of smell intact" d="1" id="10973" /><v c="decreased sense of smell" id="10974" /><v c="decreased sense of smell bilaterally" id="10975" /><v c="decreased sense of smell on the right" id="10976" /><v c="decreased sense of smell on the left" id="10977" /><mds><a><g id="13665" c="Severity (appears before findings)"><md c="mild" id="6060" /><md c="moderate" id="6061" /><md c="severe" id="6062" /></g></a><l /><m /></mds></c><c c="vision (general)" e="2" id="14300"><v c="vision intact bilaterally" d="1" id="10978" /><v c="vision intact OU" id="10979" /><v c="vision reduced bilaterally" id="10980" /><v c="vision reduced on the right" id="10981" /><v c="vision reduced on the left" id="10982" /></c><c c="visual acuity" e="2" id="14301"><v c="visual acuity normal" d="1" id="10983" /><v c="visual acuity 20/20" id="10984" /><v c="visual acuity 20/20 at near" id="10985" /><v c="visual acuity 20/20 at distance" id="10986" /><v c="visual acuity impaired" id="10987" /><v c="visual acuity reduced" id="10988" /><v c="right eye without vision" id="10989" /><v c="left eye without vision" id="10990" /></c><c c="peripheral vision" e="2" id="14302"><v c="peripheral vision normal to confrontation" d="1" id="10991" /><v c="peripheral vision reduced" id="10992" /><v c="peripheral vision reduced diffusely" id="10993" /><v c="homonymous hemianopsia present" id="10994" /><v c="bitemporal hemianopsia present" id="10995" /><v c="vertical hemianopsia present" id="10996" /><v c="scotoma present right eye" id="10997" /><v c="scotoma present left eye" id="10998" /></c><c c="pupillary response to light" e="2" id="14303"><v c="pupillary response to light brisk" d="1" id="10999" /><v c="right eye: decreased pupillary response to light " id="11000" /><v c="left eye: decreased pupillary response to light" id="11001" /><v c="bilateral decreased pupillary response to light" id="11002" /></c><c c="optic discs" e="2" id="14304"><v c="optic discs normal bilaterally" d="1" id="11003" /><v c="papilledema present in both eyes" id="11004" /><v c="papilledema present OU" id="11005" /><v c="papilledema present in right eye" id="11006" /><v c="papilledema present in left eye" id="11007" /><v c="papilledema present OD" id="11008" /><v c="papilledema present OS" id="11009" /><mds><a><g id="13666" c="Severity (appears before finding)"><md c="mild" id="6063" /><md c="moderate" id="6064" /><md c="severe" id="6065" /></g></a><l /><m /></mds></c><c c="eye movements" e="2" id="14305"><v c="eye movements within normal limits" d="1" id="11010" /><v c="right eye mobility impaired" id="11011" /><v c="left eye mobility impaired" id="11012" /><v c="lateral rectus palsy present" id="11013" /><v c="medial rectus palsy present" id="11014" /><v c="superior rectus palsy present" id="11015" /><v c="inferior rectus palsy" id="11016" /><v c="inferior oblique palsy present" id="11017" /><mds><a><g id="13667" c="Laterality (appears before finding)"><md c="right" id="6066" /><md c="left" id="6067" /><md c="bilateral" id="6068" /></g></a><l /><m /></mds></c><c c="nystagmus" e="1" id="14306"><v c="no nystagmus present" d="1" id="11018" /><v c="nystagmus present" id="11019" /><v c="nystagmus present on lateral gaze" id="11020" /><v c="mild nystagmus present" id="11021" /><v c="moderate nystagmus present" id="11022" /><v c="severe nystagmus present" id="11023" /></c><c c="ptosis" e="2" id="14307"><v c="no ptosis present" d="1" id="11024" /><v c="ptosis present OD" id="11025" /><v c="ptosis present OS" id="11026" /><v c="ptosis present OU" id="11027" /><v c="right-sided ptosis present" id="11028" /><v c="left-sided ptosis present" id="11029" /><v c="bilateral ptosis present" id="11030" /><v c="see eyelid examination above" id="11031" /><mds><a><g id="13668" c="Severity (appears before finding)"><md c="mild" id="6069" /><md c="moderate" id="6070" /><md c="severe" id="6071" /></g></a><l /><m /></mds></c><c c="facial sensation" e="2" id="14308"><v c="facial sensation normal bilaterally" d="1" id="11032" /><v c="decreased facial sensation present" id="11033" /><v c="decreased facial sensation to pinprick present" id="11034" /><v c="decreased facial sensation to light touch present" id="11035" /><mds><a><g id="13669" c="Severity (appears before finding)"><md c="mildly" id="6072" /><md c="moderately" id="6073" /><md c="severely" id="6074" /></g></a><l><g c="Distribution" id="13670"><md c="right face" id="6075" /><md c="left face" id="6076" /><md c="bilaterally" id="6077" /><md c="right V1 distribution" id="6078" /><md c="right V2 distribution" id="6079" /><md c="right V3 distribution" id="6080" /><md c="left V1 distribution" id="6081" /><md c="left V2 distribution" id="6082" /><md c="left V3 distribution" id="6083" /></g></l><m /></mds></c><c c="masseter strength" e="2" id="14309"><v c="masseter strength intact bilaterally" d="1" id="11036" /><v c="reduced masseter strength" id="11037" /><v c="reduced right masseter strength" id="11038" /><v c="reduced left masseter strength" id="11039" /><mds><a><g id="13671" c="Severity (appears before finding)"><md c="mild" id="6084" /><md c="moderate" id="6085" /><md c="severe" id="6086" /></g></a><l /><m /></mds></c><c c="facial weakness" e="2" id="14310"><v c="no facial weakness present" d="1" id="11040" /><v c="right facial weakness present" id="11041" /><v c="left facial weakness present" id="11042" /><v c="right upper motor neuron facial weakness present" id="11043" /><v c="right lower motor neuron facial weakness present" id="11044" /><v c="left upper motor neuron facial weakness present" id="11045" /><v c="left lower motor neuron facial weakness present" id="11046" /><mds><a><g id="13672" c="Severity (appears before finding)"><md c="mild" id="6087" /><md c="moderate" id="6088" /><md c="severe" id="6089" /></g></a><l /><m /></mds></c><c c="hearing" e="2" id="14311"><v c="hearing intact bilaterally" d="1" id="11047" /><v c="hearing decreased bilaterally" id="11048" /><v c="hearing decreased on the left" id="11049" /><v c="hearing decreased on the right" id="11050" /><v c="hearing absent on the left" id="11051" /><v c="hearing absent on the right" id="11052" /><v c="bilateral deafness present" id="11053" /></c><c c="pharyngeal sensation" e="2" id="14312"><v c="pharyngeal sensation normal" d="1" id="11054" /><v c="pharyngeal sensation reduced bilaterally" id="11055" /><v c="pharyngeal sensation reduced on the right" id="11056" /><v c="pharyngeal sensation reduced on the left" id="11057" /></c><c c="gag reflex" e="2" id="14313"><v c="normal gag reflex" d="1" id="11058" /><v c="decreased gag reflex" id="11059" /><v c="absent gag reflex" id="11060" /></c><c c="shoulder shrug" e="2" id="14314"><v c="shoulder shrug strength normal" d="1" id="11061" /><v c="reduced shoulder shrug bilaterally" id="11062" /><v c="reduced shoulder shrug on the right" id="11063" /><v c="reduced shoulder shrug on the left" id="11064" /></c><c c="head turning" e="2" id="14315"><v c="head turning strength normal" d="1" id="11065" /><v c="bilateral reduced head turning strength" id="11066" /><v c="reduced head turning strength to the left" id="11067" /><v c="reduced head turning strength to the right" id="11068" /></c><c c="tongue movement" e="2" id="14316"><v c="tongue movements normal" d="1" id="11069" /><v c="tongue midline on protrusion" id="11070" /><v c="tongue deviates to the right on protrusion" id="11071" /><v c="tongue deviates to the left on protrusion" id="11072" /></c></cp><mp><el id="00071" idt="sb" e="1" /><el id="00072" idt="sb" e="1" /><el id="00073" idt="sb" e="1" /><el id="00074" idt="sb" e="1" /><el id="00075" idt="sb" e="1" /><el id="00076" idt="sb" e="1" /><el id="00077" idt="sb" e="1" /><el id="00078" idt="sb" e="1" /><el id="00087" idt="sb" e="1" /><el id="00088" idt="sb" e="1" /><el id="00089" idt="sb" e="1" /><el id="00090" idt="sb" e="1" /><el id="00091" idt="sb" e="1" /><el id="00092" idt="sb" e="1" /><el id="00093" idt="sb" e="1" /><el id="00094" idt="sb" e="1" /></mp></bd><b c="Optic Nerve" id="068-0-8244548-0-6302348-0-922932-0-9941059-0-4387882-0-3686123-0-8649381-0-6819769-0-1999167-0-628047-0-4578841-0-1662956" st="0" Co="1"><bd e="1"><cp id="14818" ft="pupils are equally reactive to light, visual fields normal" fd="peripheral vision normal to confrontation, funduscopic exam reveals sharp discs" fm="11096, 11105" df="f"><c c="vision (general)" e="2" id="14318"><v c="vision intact bilaterally" d="1" id="11081" /><v c="vision intact OU" id="11082" /><v c="vision reduced bilaterally" id="11083" /><v c="vision reduced on the right" id="11084" /><v c="vision reduced on the left" id="11085" /><v c="see Eye Examination above" id="11086" /></c><c c="visual acuity" e="2" id="14319"><v c="visual acuity normal" d="1" id="11087" /><v c="visual acuity 20/20" id="11088" /><v c="visual acuity 20/20 at near" id="11089" /><v c="visual acuity 20/20 at distance" id="11090" /><v c="visual acuity impaired" id="11091" /><v c="visual acuity reduced" id="11092" /><v c="right eye without vision" id="11093" /><v c="left eye without vision" id="11094" /><v c="see Eye Examination above" id="11095" /></c><c c="peripheral vision" e="1" id="14320"><v c="peripheral vision normal to confrontation" d="1" id="11096" /><v c="peripheral vision reduced" id="11097" /><v c="peripheral vision reduced diffusely" id="11098" /><v c="homonymous hemianopsia present" id="11099" /><v c="bitemporal hemianopsia present" id="11100" /><v c="vertical hemianopsia present" id="11101" /><v c="scotoma present right eye" id="11102" /><v c="scotoma present left eye" id="11103" /><v c="see Eye Examination above" id="11104" /></c><c c="optic discs" e="1" id="14321"><v c="funduscopic exam reveals sharp discs" d="1" id="11105" /><v c="papilledema present in both eyes" id="11106" /><v c="papilledema present OU" id="11107" /><v c="papilledema present in right eye" id="11108" /><v c="papilledema present in left eye" id="11109" /><v c="papilledema present OD" id="11110" /><v c="papilledema present OS" id="11111" /><mds><a><g id="13675" c="Severity (appears before finding)"><md c="mild" id="6100" /><md c="moderate" id="6101" /><md c="severe" id="6102" /></g></a><l /><m /></mds><v c="see Funduscopic Examination above" id="11112" /></c><c c="afferent pupillary response" e="2" id="14322"><v c="pupillary response to light brisk" d="1" id="11113" /><v c="right eye: decreased pupillary response to light " id="11114" /><v c="left eye: decreased pupillary response to light" id="11115" /><v c="bilateral decreased pupillary response to light" id="11116" /><v c="see Pupil Examination above under &quot;Eyes&quot;" id="11117" /></c><c c="See eye exam above" e="2" id="14323"><v c="See eye exam above" d="1" id="11118" /></c></cp><mp><el id="00241" idt="b" e="1" /><el id="00071" idt="sb" e="1" /><el id="00087" idt="sb" e="1" /></mp></bd></b><b c="Oculomotor, Trochlear and Abducens Nerves" id="068-0-8244548-0-6302348" st="0" Co="1"><bd e="1"><cp id="14819" ft="primary gaze normal, extraocular movements are full, no ptosis present, no pathologic nystagmus present" fd="pupils are equal, round, and reactive to light, extraocular movements are full, no pathologic nystagmus present" fm="11119, 11131, 11173" df="f"><c c="pupillary response to light" e="1" id="14324"><v c="pupils are equal, round, and reactive to light" d="1" id="11119" /><v c="right eye: decreased pupillary response to light " id="11120" /><v c="left eye: decreased pupillary response to light" id="11121" /><v c="bilateral decreased pupillary response to light" id="11122" /><f e="2" mid="14873" /><f e="2" mid="14874" /><f e="0" mid="14875" /></c><c c="primary gaze" e="2" id="14325"><v c="primary gaze normal" d="1" id="11123" /><mds><a><g id="13676" c="Severity"><md c="mild" id="6103" /><md c="moderate" id="6104" /><md c="severe" id="6105" /></g><g id="13677" c="Laterality"><md c="right" id="6106" /><md c="left" id="6107" /><md c="bilateral" id="6108" /></g></a><l /><m /></mds><f e="2" mid="14873" /><f e="0" mid="14874" /><f e="0" mid="14875" /><v c="primary gaze: right eye &quot;down and out&quot;" id="11124" /><v c="primary gaze: left eye &quot;down and out&quot;" id="11125" /><v c="primary gaze: both eyes &quot;down and out&quot;" id="11126" /><v c="primary gaze: eyes deviated to the right" id="11127" /><v c="primary gaze: eyes deviated to the left" id="11128" /><v c="primary gaze: right eye deviated medially" id="11129" /><v c="primary gaze: left eye deviated medially" id="11130" /></c><c c="CN III eye movements" e="1" id="14326"><v c="extraocular movements are full" d="1" id="11131" /><v c="right eye mobility impaired" id="11132" /><v c="left eye mobility impaired" id="11133" /><v c="medial rectus palsy present" id="11134" /><v c="superior rectus palsy present" id="11135" /><v c="inferior rectus palsy" id="11136" /><v c="inferior oblique palsy present" id="11137" /><mds><a><g id="13678" c="Severity"><md c="mild" id="6109" /><md c="moderate" id="6110" /><md c="severe" id="6111" /><md c="1" id="6112" /><md c="2" id="6113" /><md c="3" id="6114" /><md c="4" id="6115" /><md c="5" id="6116" /><md c="6" id="6117" /><md c="7" id="6118" /><md c="8" id="6119" /><md c="9" id="6120" /><md c="10" id="6121" /><md c="diopter" id="6122" /></g><g id="13679" c="Laterality (appears before finding)"><md c="right" id="6123" /><md c="left" id="6124" /><md c="bilateral" id="6125" /></g></a><l /><m /></mds><f e="2" mid="14873" /><f e="0" mid="14874" /><f e="0" mid="14875" /><v c="tropia increased with medial gaze present" id="11138" /><v c="tropia increase with upward gaze present" id="11139" /></c><c c="superior oblique" e="2" id="14327"><v c="superior oblique movement intact bilaterally" d="1" id="11140" /><v c="right superior oblique paresis present" id="11141" /><v c="left superior oblique paresis present" id="11142" /><v c="bilateral superior oblique paresis present" id="11143" /><f e="0" mid="14873" /><f e="2" mid="14874" /><f e="0" mid="14875" /></c><c c="lateral rectus" e="2" id="14328"><v c="lateral rectus movements intact bilaterally" d="1" id="11144" /><v c="right lateral rectus paresis present" id="11145" /><v c="left lateral rectus paresis present" id="11146" /><mds><a><g id="13680" c="Severity"><md c="mild" id="6126" /><md c="moderate" id="6127" /><md c="severe" id="6128" /><md c="complete" id="6129" /><md c="1" id="6130" /><md c="2" id="6131" /><md c="3" id="6132" /><md c="4" id="6133" /><md c="5" id="6134" /><md c="6" id="6135" /><md c="7" id="6136" /><md c="8" id="6137" /><md c="9" id="6138" /><md c="10" id="6139" /><md c="diopter" id="6140" /><md c="tropia" id="6141" /><md c="phoria" id="6142" /><md c="maximal on lateral gaze" id="6143" /></g></a><l /><m /></mds><f e="0" mid="14873" /><f e="0" mid="14874" /><f e="1" mid="14875" /></c><c c="ptosis" e="2" id="14329"><v c="no ptosis present" d="1" id="11147" /><v c="ptosis present OD" id="11148" /><v c="ptosis present OS" id="11149" /><v c="ptosis present OU" id="11150" /><v c="right-sided ptosis present" id="11151" /><v c="left-sided ptosis present" id="11152" /><v c="bilateral ptosis present" id="11153" /><v c="see eyelid examination above" id="11154" /><mds><a><g id="13681" c="Severity (appears before finding)"><md c="mild" id="6144" /><md c="moderate" id="6145" /><md c="severe" id="6146" /></g></a><l /><m /></mds><f e="2" mid="14873" /><f e="0" mid="14874" /><f e="0" mid="14875" /></c><c c="internuclear ophthalmoplegia" e="2" id="14330"><v c="no internuclear ophthalmoplegia present" d="1" id="11155" /><v c="left internuclear ophthalmoplegia present" id="11156" /><v c="right internuclear ophthalmoplegia present" id="11157" /><v c="anterior right INO present" id="11158" /><v c="anterior left INO present" id="11159" /><v c="posterior right INO present" id="11160" /><v c="posterior left INO present" id="11161" /><v c="bilateral INO's present" id="11162" /><f e="0" mid="14873" /><f e="0" mid="14874" /><f e="0" mid="14875" /></c><c c="vertical gaze" e="2" id="14331"><v c="vertical gaze normal" d="1" id="11163" /><v c="upward gaze impaired bilaterally" id="11164" /><v c="upward gaze impaired on the right" id="11165" /><v c="upward gaze impaired on the left" id="11166" /><v c="downward gaze impaired bilaterally" id="11167" /><v c="downward gaze impaired on the right" id="11168" /><v c="downward gaze impaired on the left" id="11169" /><v c="see-saw nystagmus present" id="11170" /><v c="pendular nystagmus present" id="11171" /><v c="jerk nystagmus present" id="11172" /><f e="2" mid="14873" /><f e="2" mid="14874" /><f e="0" mid="14875" /></c><c c="nystagmus" e="1" id="14332"><v c="no pathologic nystagmus present" d="1" id="11173" /><v c="horizontal right-beating nystagmus present" id="11174" /><v c="horizontal left-beating nystagmus present" id="11175" /><mds><a><g id="13682" c="Rate of fatigue"><md c="physiologic" id="6147" /><md c="sustained" id="6148" /><md c="rapidly fatigued" id="6149" /></g></a><l><g id="13683" c="Eye position"><md c="in primary position" id="6150" /><md c="on right lateral gaze" id="6151" /><md c="on left lateral gaze" id="6152" /><md c="on downgaze" id="6153" /><md c="on upgaze" id="6154" /></g></l><m /></mds><v c="see-saw nystagmus present" id="11176" /><v c="vertical up-beating nystagmus present" id="11177" /><v c="vertical down-beating nystagmus present" id="11178" /><v c="clockwise rotatory nystagmus present" id="11179" /><v c="counter-clockwise rotatory nystagmus present" id="11180" /><v c="pendular nystagmus present" id="11181" /><f e="2" mid="14873" /><f e="2" mid="14874" /><f e="2" mid="14875" /></c><c c="ciliospinal reflex" e="2" id="14333"><v c="ciliospinal reflex intact bilaterally" d="1" id="11182" /><v c="ciliospinal reflex absent bilaterally" id="11183" /><v c="ciliospinal reflex absent on the right" id="11184" /><v c="ciliospinal reflex absent on the left" id="11185" /><f e="0" mid="14873" /><f e="0" mid="14874" /><f e="0" mid="14875" /></c><c c="doll's eyes" e="2" id="14334"><v c="doll's eyes intact" d="1" id="11186" /><v c="doll's eyes absent in both directions" id="11187" /><v c="doll's eyes absent on head turning to the right" id="11188" /><v c="doll's eyes absent on head turning to the left" id="11189" /><f e="0" mid="14873" /><f e="0" mid="14874" /><f e="0" mid="14875" /></c><c c="phorias" e="2" id="14335"><v c="no phorias present" d="1" id="11190" /><v c="esophoria present" id="11191" /><v c="exophoria present" id="11192" /><v c="hypophoria present" id="11193" /><v c="hyperphoria present" id="11194" /><mds><a><g id="13684" c="Severity"><md c="mild" id="6155" /><md c="moderate" id="6156" /><md c="severe" id="6157" /><md c="1" id="6158" /><md c="2" id="6159" /><md c="3" id="6160" /><md c="4" id="6161" /><md c="5" id="6162" /><md c="6" id="6163" /><md c="7" id="6164" /><md c="8" id="6165" /><md c="9" id="6166" /><md c="10" id="6167" /><md c="diopter" id="6168" /></g></a><l><g id="13685" c="Position"><md c="maximal" id="6169" /><md c="in primary gaze" id="6170" /><md c="in right gaze" id="6171" /><md c="in left gaze" id="6172" /><md c="in upgaze" id="6173" /><md c="in downgaze" id="6174" /><md c="on cover-uncover test" id="6175" /></g></l><m /></mds><f e="2" mid="14873" /><f e="2" mid="14874" /><f e="2" mid="14875" /></c><c c="tropias" e="2" id="14336"><v c="no tropias present" d="1" id="11195" /><v c="esotropia present" id="11196" /><v c="exotropia present" id="11197" /><v c="hypertropia present" id="11198" /><v c="hypotropia present" id="11199" /><mds><a><g id="13686" c="Severity"><md c="mild" id="6176" /><md c="moderate" id="6177" /><md c="severe" id="6178" /><md c="1" id="6179" /><md c="2" id="6180" /><md c="3" id="6181" /><md c="4" id="6182" /><md c="5" id="6183" /><md c="6" id="6184" /><md c="7" id="6185" /><md c="8" id="6186" /><md c="9" id="6187" /><md c="10" id="6188" /><md c="diopter" id="6189" /></g></a><l><g id="13687" c="Position"><md c="maximal" id="6190" /><md c="in right-gaze" id="6191" /><md c="in left-gaze" id="6192" /><md c="in up gaze" id="6193" /><md c="in down gaze" id="6194" /><md c="on cover-uncover testing" id="6195" /></g></l><m /></mds><f e="2" mid="14873" /><f e="2" mid="14874" /><f e="2" mid="14875" /></c><c c="optokinetic response" e="2" id="14337"><v c="optokinetic response: normal in both directions" d="1" id="11200" /><f mid="14873" e="0" /><f mid="14874" e="0" /><f mid="14875" e="0" /><v c="optokinetic response: abnormal to the right" id="11201" /><v c="optokinetic response: abnormal to the left" id="11202" /><v c="optokinetic response: abnormal in both directions" id="11203" /></c><c c="Red reflex intact bilaterally" e="2" id="C9E7F6BB-18E8-484A-A43D-3B47AAA97A66"><v c="Red reflex intact bilaterally" d="1" id="5449A395-FABE-4D54-9808-7F759D3D65FA" /><f mid="14873" e="0" /><f mid="14874" e="0" /><f mid="14875" e="0" /><v c="Red reflex absent" id="D102DE6F-F8AB-4855-A194-3F23909C5452" /></c></cp><mp><el id="00072" idt="sb" e="1" /><el id="00242" idt="b" e="1" /><el id="00088" idt="sb" e="1" /></mp><f c="Oculomotor Nerve" id="14873" ft="" fd="" fm="" /><f c="Trochlear Nerve" id="14874" ft="" fd="" fm="" /><f c="Abducens Nerve" id="14875" ft="lateral rectus movements intact bilaterally" fd="lateral rectus movements intact bilaterally" fm="F8FB3CAA-E269-46B3-AF4D-74F670E7ED8A" /></bd></b><b c="Facial Nerve" id="068-0-8244548-0-6302348-0-922932-0-9941059-0-4387882" st="0" Co="1"><bd e="1"><cp id="14821" ft="no facial weakness present" fd="Face is symmetric with normal movement." fm="11220" df="f"><c c="facial weakness" e="1" id="14342"><v c="Face is symmetric with normal movement." d="1" id="11220" /><v c="right facial weakness present" id="11221" /><v c="left facial weakness present" id="11222" /><v c="right upper motor neuron facial weakness present" id="11223" /><v c="right lower motor neuron facial weakness present" id="11224" /><v c="left upper motor neuron facial weakness present" id="11225" /><v c="left lower motor neuron facial weakness present" id="11226" /><mds><a><g id="13691" c="Severity (appears before finding)"><md c="mild" id="6215" /><md c="moderate" id="6216" /><md c="severe" id="6217" /></g></a><l /><m /></mds></c><c c="orbicularis oculi reflex" e="2" id="14343"><v c="orbicularis oculi reflex present" d="1" id="11227" /><v c="orbicularis oculi reflex absent on the right" id="11228" /><v c="orbicularis oculi reflex absent on the left" id="11229" /><v c="orbicularis oculi reflex absent bilaterally" id="11230" /><v c="glabellar reflex absent" id="11231" /></c><c c="supraorbital reflex" e="2" id="14344"><v c="supraorbital reflex present" d="1" id="11232" /><v c="supraorbital reflex absent on the right" id="11233" /><v c="supraorbital reflex absent on the left" id="11234" /><v c="supraorbital reflex absent bilaterally" id="11235" /></c><c c="orbicularis oris reflex" e="2" id="14345"><v c="orbicularis oris reflex" d="1" id="11236" /></c><c c="Chvostek's sign" e="2" id="14346"><v c="Chvostek's sign not present on provocative testing" d="1" id="11237" /><v c="Chvostek's sign present on the right" id="11238" /><v c="Chvostek's sign present on the left" id="11239" /><v c="Chvostek's sign present bilaterally" id="11240" /></c><c c="Bell's phenomenon" e="2" id="14347"><v c="Bell's phenomenon absent" d="1" id="11241" /><v c="Bell's phenomenon present on the right" id="11242" /><v c="Bell's phenomenon present on the left" id="11243" /></c><c c="taste" e="2" id="14348"><v c="taste intact over anterior two thirds of tongue" d="1" id="11244" /><v c="taste impaired over anterior two thirds of tongue on the right" id="11245" /><v c="taste impaired over anterior two thirds of tongue on the left" id="11246" /><v c="taste impaired over anterior two thirds of tongue bilaterally" id="11247" /></c><c c="reinnervation" e="2" id="14349"><v c="no signs of reinnervation" d="1" id="11248" /><v c="reinnervation of facial muscles present on the right" id="11249" /><v c="reinnervation of facial muscles present on the left" id="11250" /><v c="crocodile tears present on the right" id="11251" /><v c="crocodile tears present on the left" id="11252" /></c></cp><mp><el id="00244" idt="b" e="1" /><el id="00074" idt="sb" e="1" /><el id="00090" idt="sb" e="1" /></mp></bd></b><b c="Glossopharyngeal and Vagus Nerves" id="068-0-8244548-0-6302348-0-922932-0-9941059-0-4387882-0-3686123-0-8649381-0-6819769" st="0" Co="1"><bd e="1"><cp id="14823" ft="Palate and uvula are midline and move normally" fd="Palate and uvula are midline and move normally" fm="11291" df="f"><c c="pharyngeal sensation" e="2" id="14357"><v c="pharyngeal sensation normal" d="1" id="11284" /><v c="pharyngeal sensation reduced bilaterally" id="11285" /><v c="pharyngeal sensation reduced on the right" id="11286" /><v c="pharyngeal sensation reduced on the left" id="11287" /></c><c c="gag reflex" e="2" id="14358"><v c="gag reflex normal" d="1" id="11288" /><v c="decreased gag reflex" id="11289" /><v c="absent gag reflex" id="11290" /></c><c c="soft palate" e="1" id="14359"><v c="Palate and uvula are midline and move normally" d="1" id="11291" /><v c="palatal arch lower on right" id="11292" /><v c="palatal arch lower on left" id="11293" /><v c="lateral wall on the right paretic" id="11294" /><v c="lateral wall on the left paretic" id="11295" /><v c="uvula deviates to the right" id="11296" /><v c="uvula deviates to the left" id="11297" /></c><c c="taste" e="2" id="14360"><v c="taste normal over posterior third of tongue" d="1" id="11298" /><v c="taste reduced over posterior third of tongue on the right" id="11299" /><v c="taste reduced over posterior third of tongue on the left" id="11300" /></c></cp><mp><el id="00246" idt="b" e="1" /><el id="00076" idt="sb" e="1" /><el id="00092" idt="sb" e="1" /></mp></bd></b><b c="Spinal Accessory Nerve" id="068-0-8244548-0-6302348-0-922932-0-9941059-0-4387882-0-3686123-0-8649381-0-6819769-0-1999167-0-628047" st="0" Co="1"><bd e="1"><cp id="14824" ft="shoulder shrug and sternocleidomastoid strength normal" fd="shoulder shrug and sternocleidomastoid strength normal" df="f"><c c="shoulder shrug" e="1" id="14361"><v c="shoulder shrug and sternocleidomastoid strength normal" d="1" id="11301" /><v c="reduced shoulder shrug bilaterally" id="11302" /><v c="reduced shoulder shrug on the right" id="11303" /><v c="reduced shoulder shrug on the left" id="11304" /><mds><a><g id="13692" c="Severity"><md c="mildly" id="6218" /><md c="moderately" id="6219" /><md c="markedly" id="6220" /></g></a><l /><m /></mds></c><c c="head turning" e="1" id="14362"><v c="head turning strength normal bilaterally" d="1" id="11305" /><v c="weakness of head turning to the right" id="11306" /><v c="weakness of head turning to the left" id="11307" /><v c="weakness of head turning bilaterally" id="11308" /><mds><a><g id="13693" c="Severity"><md c="mild" id="6221" /><md c="moderate" id="6222" /><md c="severe" id="6223" /></g></a><l /><m /></mds></c><c c="torticollis" e="2" id="14363"><v c="no torticollis" d="1" id="11309" /><v c="right-sided torticollis present" id="11310" /><v c="left-sided torticollis present" id="11311" /></c></cp><mp><el id="00247" idt="b" e="1" /><el id="00077" idt="sb" e="1" /><el id="00093" idt="sb" e="1" /></mp></bd></b><b c="Hypoglossal Nerve" id="068-0-8244548-0-6302348-0-922932-0-9941059-0-4387882-0-3686123-0-8649381-0-6819769-0-1999167-0-628047-0-4578841" st="0" Co="1"><bd e="1"><cp id="14825" ft="tongue movements normal" fd="tongue movements normal" df="f"><c c="tongue movement" e="1" id="14364"><v c="tongue movements normal, tongue extrusion midline" d="1" id="11312" /><v c="tongue midline on protrusion" id="11313" /><v c="tongue deviates to the right on protrusion" id="11314" /><v c="tongue deviates to the left on protrusion" id="11315" /><v c="tongue weakness noted on tongue depressor testing" id="11316" /><v c="right-sided tongue weakness noted on tongue depressor testing" id="11317" /><v c="left-sided tongue weakness noted on tongue depressor testing" id="11318" /></c></cp><mp><el id="00248" idt="b" e="1" /><el id="00078" idt="sb" e="1" /><el id="00094" idt="sb" e="1" /></mp></bd></b></b><b c="Motor Examination" id="062-0-1446046-0-2657754-0-9682123" st="0" Co="1"><bd e="1"><cp id="14826" ft="full strength in all groups" fd="" fm="" df="f" /><mp><el id="00051" idt="b" e="1" /><el id="00211" idt="b" e="1" /></mp></bd></b><b c="Reflexes" id="069" st="0" Co="1"><bd e="1"><cp id="14835" ft="deep tendon reflexes 2+ and symmetric in all four extremities, Plantars downgoing bilaterally" fd="deep tendon reflexes normal, symmetric in all four extremities, Toes are downgoing bilaterally" fm="12361, 882E3CE5-D7AE-4433-8172-C9C06BF0F26D" df="f"><c c="deep tendon reflexes" e="1" id="14531"><v c="deep tendon reflexes normal, symmetric in all four extremities" d="1" id="12361" /><v c="diffuse hyporeflexia present" id="12362" /><v c="deep tendon reflexes absent" id="12363" /><v c="diffuse hyperreflexia present" id="12364" /><mds><a><g id="13762" c="Laterality (appears before finding)"><md c="right" id="6612" /><md c="left" id="6613" /><md c="bilateral" id="6614" /></g></a><l /><m /></mds></c><c c="biceps" e="2" id="14532"><v c="biceps reflex 2+" d="1" id="12365" /><v c="biceps reflex absent" id="12366" /><v c="biceps reflex 0" id="12367" /><v c="biceps reflex trace" id="12368" /><v c="biceps reflex 1+" id="12369" /><v c="biceps reflex 3+" id="12370" /><v c="biceps reflex 4+" id="12371" /><v c="biceps reflex clonus present" id="12372" /><v c="biceps reflex sustained clonus present" id="12373" /><mds><a><g id="13763" c="Laterality (appears before finding)"><md c="right" id="6615" /><md c="left" id="6616" /><md c="bilateral" id="6617" /></g></a><l /><m /></mds></c><c c="triceps" e="2" id="14533"><v c="triceps reflex 2+" d="1" id="12374" /><v c="triceps reflex 0" id="12375" /><v c="triceps reflex absent" id="12376" /><v c="triceps reflex trace" id="12377" /><v c="triceps reflex 1+" id="12378" /><v c="triceps reflex 3+" id="12379" /><v c="triceps reflex 4+" id="12380" /><v c="triceps reflex clonus present" id="12381" /><v c="triceps reflex sustained clonus present" id="12382" /><mds><a><g id="13764" c="Laterality (appears before finding)"><md c="right" id="6618" /><md c="left" id="6619" /><md c="bilateral" id="6620" /></g></a><l /><m /></mds></c><c c="brachioradialis " e="2" id="14534"><v c="brachioradialis reflex 2+" d="1" id="12383" /><v c="brachioradialis reflex absent" id="12384" /><v c="brachioradialis reflex 0" id="12385" /><v c="brachioradialis reflex trace" id="12386" /><v c="brachioradialis reflex 1+" id="12387" /><v c="brachioradialis reflex 3+" id="12388" /><v c="brachioradialis reflex 4+" id="12389" /><v c="brachioradialis reflex clonus present" id="12390" /><v c="sustained brachioradialis reflex clonus present" id="12391" /><mds><a><g id="13765" c="Laterality (appears before finding)"><md c="right" id="6621" /><md c="left" id="6622" /><md c="bilateral" id="6623" /></g></a><l /><m /></mds></c><c c="knee" e="2" id="14535"><v c="knee reflex 2+" d="1" id="12392" /><v c="knee reflex absent" id="12393" /><v c="knee reflex 0" id="12394" /><v c="knee reflex trace" id="12395" /><v c="knee reflex 1+" id="12396" /><v c="knee reflex 3+" id="12397" /><v c="knee reflex 4+" id="12398" /><v c="knee reflex clonus present" id="12399" /><v c="sustained knee reflex clonus present" id="12400" /><mds><a><g id="13766" c="Laterality (appears before finding)"><md c="right" id="6624" /><md c="left" id="6625" /><md c="bilateral" id="6626" /></g></a><l /><m /></mds></c><c c="ankle " e="2" id="14536"><v c="ankle reflex 2+" d="1" id="12401" /><v c="ankle reflex absent" id="12402" /><v c="ankle reflex 0" id="12403" /><v c="ankle reflex trace" id="12404" /><v c="ankle reflex 1+" id="12405" /><v c="ankle reflex 3+" id="12406" /><v c="ankle reflex 4+" id="12407" /><v c="ankle clonus present" id="12408" /><v c="sustained ankle clonus present" id="12409" /><mds><a><g id="13767" c="Laterality (appears before finding)"><md c="right" id="6627" /><md c="left" id="6628" /><md c="bilateral" id="6629" /></g></a><l /><m /></mds></c><c c="Babinski" e="2" id="14537"><v c="Babinski response negative" d="1" id="12410" /><v c="Babinski response positive" id="12411" /><v c="Plantar response negative" id="12412" /><v c="Plantar response positive" id="12413" /><mds><a><g id="13768" c="Laterality (appears before finding)"><md c="right" id="6630" /><md c="left" id="6631" /><md c="bilateral" id="6632" /></g></a><l /><m /></mds></c><c c="Toes are downgoing bilaterally" e="1" id="D7D8BB61-C6A5-420D-A62E-C2AE58EB428C"><v c="Toes are downgoing bilaterally" d="1" id="882E3CE5-D7AE-4433-8172-C9C06BF0F26D" /></c></cp><mp><el id="00056" idt="b" e="1" /><el id="00250" idt="b" e="1" /><el id="00223" idt="b" e="1" /></mp></bd></b><b c="Sensation" id="070" st="0" Co="1"><bd e="1"><cp id="14842" ft="intact to pin prick, light touch and proprioception throughout" fd="" df="f" /><mp><el id="00224" idt="b" e="1" /></mp></bd></b><b c="Cerebellar Function" id="6CD515AC-5872-4B9C-B6E6-4230FD2F6FF3" st="0" Co="1"><bd e="1"><cp id="14850" ft="no dysmetria on finger-to-nose testing, no dysmetria on heel to shin testing" fd="no dysmetria on finger-to-nose testing, no dysmetria on heel to shin testing, no evidence of truncal ataxia" fm="12863, 12876, 612D43B1-6132-4F69-9F3F-4E02793FCEF1" df="f"><c c="finger-to-nose" e="1" id="14641"><v c="no dysmetria on finger-to-nose testing" d="1" id="12863" /><v c="bilateral ataxia present on finger-to-nose-to-finger testing " id="12864" /><v c="right-sided ataxia present on finger-to-nose-to-finger testing " id="12865" /><v c="left-sided ataxia present on finger-to-nose-to-finger testing " id="12866" /><mds><a><g id="13855" c="Severity"><md c="mild" id="7560" /><md c="moderate" id="7561" /><md c="severe" id="7562" /></g></a><l /><m /></mds><v c="right upper extremity dysdiadokokinesia present on finger-nose-finger testing" id="12867" /><v c="left upper extremity dysdiadokokinesia present on finger-nose-finger testing" id="12868" /><v c="bilateral upper extremity dysdiadokokinesia present on finger-nose-finger testing" id="12869" /><v c="past-pointing present" id="12870" /></c><c c="finger-to-finger test" e="2" id="14642"><v c="finger-to-finger test negative" d="1" id="12871" /><v c="finger-to-finger test positive" id="12872" /></c><c c="past-pointing test" e="2" id="14643"><v c="past-pointing test negative" d="1" id="12873" /><v c="past-pointing test positive on the right" id="12874" /><v c="past-pointing test positive on the left" id="12875" /></c><c c="heel-shin" e="1" id="14644"><v c="no dysmetria on heel to shin testing" d="1" id="12876" /><v c="ataxia present on heel-shin cerebellar function testing on the right" id="12877" /><v c="ataxia present on heel-shin cerebellar functions testing on the left" id="12878" /><v c="ataxia present on heel-shin cerebellar function testing bilaterally" id="12879" /><mds><a><g id="13856" c="Severity"><md c="mild" id="7563" /><md c="moderate" id="7564" /><md c="severe" id="7565" /></g></a><l /><m /></mds></c><c c="rapid alternating movements" e="2" id="14645"><v c="rapid alternating movements within normal limits bilaterally" d="1" id="12880" /><v c="dysdiadochokinesis present on the right" id="12881" /><v c="dysdiadochokinesis present on the left" id="12882" /><v c="dysdiadochokinesis present bilaterally" id="12883" /><v c="rapid alternating movements impaired in the right hand" id="12884" /><v c="rapid alternating movements impaired in the left hand" id="12885" /><v c="rapid alternating movements impaired bilaterally" id="12886" /></c><c c="fine motor coordination" e="2" id="14646"><v c="fine motor coordination normal" d="1" id="12887" /><v c="fine motor coordination impairment present" id="12888" /><v c="mild fine motor coordination impairment present" id="12889" /><v c="moderate fine motor coordination impairment present" id="12890" /><v c="severe fine motor coordination impairment present" id="12891" /><mds><a><g id="13857" c="Site"><md c="right upper extremity" id="7566" /><md c="left upper extremity" id="7567" /><md c="diffuse" id="7568" /></g></a><l /><m /></mds></c><c c="straight line walking" e="2" id="14647"><v c="heel-to-toe straight line walking normal" d="1" id="12892" /><v c="mild gait ataxia on heel-to-toe straight line walking" id="12893" /><v c="moderate gait ataxia present on heel-to-toe straight line walking" id="12894" /><v c="severe gait ataxia present on heel-to-toe straight line walking" id="12895" /></c><c c="Romberg sign" e="2" id="14648"><v c="Romberg sign negative" d="1" id="12896" /><v c="Romberg sign positive" id="12897" /></c><c c="Truncal ataxia" e="1" id="2B342DD3-C375-4641-9283-843461EAD52A"><v c="no evidence of truncal ataxia" d="1" id="612D43B1-6132-4F69-9F3F-4E02793FCEF1" /></c></cp><mp><el id="00251" idt="b" e="1" /><el id="00222" idt="b" e="1" /></mp></bd></b><b c="Gait and Station" id="058" st="0" Co="0" up="1" To="0" Fo="0"><bd e="0"><cp id="14851" ft="" fd="Normal including toe, heel and tandem gait" fm="12898" df="f"><c c="native gait" e="1" id="14649"><v c="Normal including toe, heel and tandem gait" d="1" id="12898" /><v c="unstable gait" id="12899" /><v c="ataxic gait present" id="12900" /><v c="wobbly gait present" id="12901" /><v c="wide-based gait present" id="12902" /><v c="feeble gait present" id="12903" /><v c="wheelchair-dependent" id="12904" /><mds><a><g id="13858" c="Severity (appears before finding)"><md c="mild" id="7569" /><md c="moderate" id="7570" /><md c="severe" id="7571" /><md c="mildly" id="7572" /><md c="moderately" id="7573" /><md c="severely" id="7574" /><md c="age appropriate" id="7575" /></g></a><l /><m /></mds><v c="in-toeing present" id="12905" /><v c="able to take a few steps with assistance" id="12906" /><v c="unwilling to walk" id="12907" /></c><c c="station" e="2" id="14650"><v c="able to stand without difficulty" d="1" id="12908" /><v c="unsteady when standing" id="12909" /><v c="retropulsion present with standing" id="12910" /><v c="station steady with walker" id="12911" /><mds><a><g id="13859" c="Severity (appears before finding)"><md c="mild" id="7576" /><md c="moderate" id="7577" /><md c="severe" id="7578" /><md c="mildly" id="7579" /><md c="moderately" id="7580" /><md c="severely" id="7581" /></g></a><l /><m /></mds></c></cp><mp><el id="00045" idt="b" e="1" /><el id="00191" idt="b" e="1" /><el id="00233" idt="b" e="1" /></mp></bd><b c="Gait" id="61091377-041A-4E05-9CA8-3DF99ECE14BB" st="0" Co="1" To="0"><bd e="1"><cp id="14852" ft="normal gait " fd="gait approprate for age" fm="12912" df="f"><c c="gait" e="1" id="14651"><v c="gait approprate for age" d="1" id="12912" /><v c="unstable gait" id="12913" /><v c="ataxic gait present" id="12914" /><v c="wobbly gait present" id="12915" /><v c="wide-based gait present" id="12916" /><v c="feeble gait present" id="12917" /><v c="wheelchair-dependent" id="12918" /><mds><a><g id="13860" c="Severity (appears before finding)"><md c="mild" id="7582" /><md c="moderate" id="7583" /><md c="severe" id="7584" /><md c="mildly" id="7585" /><md c="moderately" id="7586" /><md c="severely" id="7587" /><md c="age appropriate" id="7588" /></g></a><l /><m /></mds><v c="in-toeing present" id="12919" /><v c="able to take a few steps with assistance" id="12920" /><v c="unwilling to walk" id="12921" /><v c="festinating gait present" id="12922" /><v c="march a petits pas present" id="12923" /><v c="wide-based" id="12924" /><v c="double tapping present" id="12925" /><v c="sensory ataxia gait present" id="12926" /><v c="spastic gait present" id="12927" /><v c="steppage gait present" id="12928" /><v c="gait apraxia present" id="12929" /><v c="dystrophic gait present" id="12930" /><v c="hysterical gait present" id="12931" /><v c="astasia-abasia present" id="12932" /><v c="approprate for age" id="1A2A30FB-932E-40CE-BEEB-75B66DEE63A4" /></c></cp><mp><el id="00045" idt="b" e="1" /><el id="00191" idt="b" e="1" /><el id="00233" idt="b" e="1" /><el id="00083" idt="b" e="1" /></mp></bd></b><b c="Station" id="A43DA96F-1ECC-4911-A7DC-7B9B6E4AEC5C" st="0" Co="1"><bd e="1"><cp id="14853" ft="able to stand without difficulty, Romberg's sign absent" fd="able to stand without difficulty, Romberg's sign absent" fm="12933, 12939" df="f"><c c="station" e="1" id="14652"><v c="able to stand without difficulty" d="1" id="12933" /><v c="unsteady when standing" id="12934" /><v c="retropulsion present with standing" id="12935" /><v c="station steady with walker" id="12936" /><mds><a><g id="13861" c="Severity (appears before finding)"><md c="mild" id="7589" /><md c="moderate" id="7590" /><md c="severe" id="7591" /><md c="mildly" id="7592" /><md c="moderately" id="7593" /><md c="severely" id="7594" /></g></a><l /><m /></mds><v c="sways with eyes closed" id="12937" /><v c="unable to maintain station with eyes closed" id="12938" /></c><c c="Romberg" e="1" id="14653"><v c="Romberg's sign absent" d="1" id="12939" /><v c="Romberg's sign present" id="12940" /></c></cp><mp><el id="00045" idt="b" e="1" /><el id="00191" idt="b" e="1" /><el id="00233" idt="b" e="1" /></mp></bd></b></b></s><s c="Psychiatric" id="071" b="1"><b c="Mood and Affect" id="075" st="0" Co="1"><bd e="1"><cp id="14856" ft="mood normal, affect appropriate" fd="mood normal, affect appropriate" df="f"><c c="mood" e="1" id="14661"><v c="mood normal for situation" d="1" id="12966" /><v c="mood: dysphoric" id="12967" /><v c="mood: depressed" id="12968" /><v c="mood: angry" id="12969" /><v c="mood: euphoric" id="12970" /><v c="mood: patient expressing feelings of sadness" id="12971" /><v c="mood: patient feeling hopeless" id="12972" /><v c="mood: patient grieving" id="12973" /><v c="mood: elated" id="12974" /><v c="mood: apathetic" id="12975" /><v c="mood: fluctuating between elevated and depressed" id="12976" /><v c="mood: fluctuating between anger and depression" id="12977" /></c><c c="affect" e="1" id="14662"><v c="affect appropriate" d="1" id="12978" /><v c="affect: inappropriate" id="12979" /><v c="affect: flattened" id="12980" /><v c="affect: blunted" id="12981" /><v c="affect: inappropriate emotional responses to particular situation" id="12982" /><v c="pathologic laughter present" id="12983" /><v c="pathologic crying present" id="12984" /></c></cp><mp><el id="00061" idt="b" e="1" /><el id="00088" idt="b" e="1" /><el id="00115" idt="b" e="1" /><el id="00129" idt="b" e="1" /><el id="00165" idt="b" e="1" /><el id="00186" idt="b" e="1" /><el id="00226" idt="b" e="1" /><el id="00266" idt="b" e="1" /><el id="00290" idt="b" e="1" /><el id="00315" idt="b" e="1" /></mp></bd></b></s></root><bd e="1"><mp><el id="00001" idt="sbullet" e="1" /><el id="00008" idt="sbullet" e="1" /><el id="00015" idt="sbullet" e="1" /><el id="00022" idt="sbullet" e="1" /><el id="00029" idt="sbullet" e="1" /><el id="00036" idt="sbullet" e="1" /><el id="00043" idt="sbullet" e="1" /><el id="00050" idt="sbullet" e="1" /><el id="00057" idt="sbullet" e="1" /><el id="00064" idt="sbullet" e="1" /><el id="00003" idt="sbullet" e="1" /><el id="00010" idt="sbullet" e="1" /><el id="00017" idt="sbullet" e="1" /><el id="00024" idt="sbullet" e="1" /><el id="00031" idt="sbullet" e="1" /><el id="00038" idt="sbullet" e="1" /><el id="00045" idt="sbullet" e="1" /><el id="00052" idt="sbullet" e="1" /><el id="00059" idt="sbullet" e="1" /><el id="00066" idt="sbullet" e="1" /><el id="00006" idt="sbullet" e="1" /><el id="00013" idt="sbullet" e="1" /><el id="00020" idt="sbullet" e="1" /><el id="00027" idt="sbullet" e="1" /><el id="00034" idt="sbullet" e="1" /><el id="00041" idt="sbullet" e="1" /><el id="00048" idt="sbullet" e="1" /><el id="00055" idt="sbullet" e="1" /><el id="00062" idt="sbullet" e="1" /><el id="00069" idt="sbullet" e="1" /><el id="00007" idt="sbullet" e="1" /><el id="00014" idt="sbullet" e="1" /><el id="00021" idt="sbullet" e="1" /><el id="00028" idt="sbullet" e="1" /><el id="00035" idt="sbullet" e="1" /><el id="00042" idt="sbullet" e="1" /><el id="00049" idt="sbullet" e="1" /><el id="00056" idt="sbullet" e="1" /><el id="00063" idt="sbullet" e="1" /><el id="00070" idt="sbullet" e="1" /></mp></bd></pe><histories><historyall selection="C" /><pmh selection="C" /><psh selection="C" /><med selection="C" /><alrg selection="C" /><fmh selection="C" /><gs selection="((PHONE_NUMBER))" /><rh selection="((PHONE_NUMBER))" /><sh selection="C" /><imm selection="((PHONE_NUMBER))" /></histories><assessmentsection assnote="" orderingmdid="1012" CareProviderIsOnStaff="1" plannote=""><diagnoses><diagnosis selected="true" insertactiveproblems="true" caption="Complex partial epilepsy" icd9="345.40" icd10="G40.209" snomedcode="407675009" snomedname="" lexicalID="57805" planuniqueid="345.400" selectedAt="2017-01-24T11:35:08.729" baseLexicalID="57805" baseLabel="Complex partial epilepsy" baseIcd10="G40.209" canPostCoord="1" modifiers="" minor="true" isMapICD10="1" expanded="true" originalCaption="Complex partial epilepsy" realOriginalCaption="Seizure, complex partial" originalICD9="345.40" originalICD10="" originalSNOMED="" invalidDX="false" ruledout="false" complexityvalue="0" newproblem="" workup="" hasUndo="0" isEdit="true" ModType="dx" diagid="480087"><diagnosismodifiers><item id="8" caption="Mild" place="pre" /><item id="7" caption="Moderate" place="pre" /><item id="6" caption="Severe" place="pre" /><item id="11" caption="resolved" place="pre" /><item id="12" caption="acute" place="pre" /><item id="13" caption="chronic" place="pre" /><item id="9" caption="Recurrent" place="pre" /><item id="10" caption="unresponsive to treatment" place="post" /><item id="14" caption="stable" place="post" /><item id="15" caption="improving" place="post" /><item id="16" caption="worsening" place="post" /><item id="17" caption="left" place="pre" /><item id="18" caption="right" place="pre" /><item id="19" caption="bilateral" place="pre" /><item id="20" caption="upper" place="pre" /><item id="21" caption="lower" place="pre" /><item id="22" caption="posterior" place="pre" /><item id="23" caption="anterior" place="pre" /><item id="24" caption="inferior" place="pre" /><item id="25" caption="superior" place="pre" /></diagnosismodifiers></diagnosis></diagnoses><correspondence corrcount="0"><ccthis created="1" createdate="2016-12-25"><cp id="1024" fname="Lawrence" mname="" lname="Adjei" suffix="" cred="MD" preferredname="" addressline1="800 Towne Park Plaza" addressline2="Unit 200" city="Rincon" state="GA" postalcode="31326" primaryphone="((EMPLOYER_PHONE))" faxnumber="((EMPLOYER_PHONE))" fullname="Lawrence Adjei MD" /></ccthis></correspondence><tasklists reftype="A" lastmodifiedby="1058" documentid="1275284" rtnMsg="" /><disp><sys><select id="1001" label="Call or Return if symptoms worsen or persist." active="false" /></sys><custsel><select id="923647F9-E8F5-4969-BF0E-((PHONE_NUMBER))BB" label="Patient/Caregiver was provdied with education on dementia disease management and health behavior changes." active="false" /><select id="515E2E77-4D3E-4A43-8E54-A5FE88F9FB07" label="Patient/Caregiver was counseled on safety with driving/not driving." active="false" /><select id="C17ECBED-3F5A-4D89-8419-A22DBDD4A56F" label="Patient/Caregiver was counseled on every day safety concerns." active="false" /></custsel><rtcreq active="false" linked="false" apptid="" recallid=""><rtc active="false" requestdate="" daybuffer="" /><apptrequest active="false" documentid="" sequenceid="" banycareproviderwilldo="" ballowoverride="" appttypeid="" stepid="" appttext="" patientid="" apptcareproviderid="" apptcareprovidertext="" apptservicelocationid="" origcareproviderid="" origservicelocationid="" userid="" requestdate="" daybuffer="" additionalinfo="" requestortypeid="" /></rtcreq><caretransition active="false" /><summaryofcareprovided active="false" /><summaryofcareprovideddirect active="false" /></disp></assessmentsection><drawings lastrefid="0" /><bailout><em><sketchpads /></em><cc><sketchpads /></cc><pfsh><sketchpads /></pfsh><ros><sketchpads /></ros><ass><sketchpads><sketchpad section="ass" index="0" srcID="assDIV" expandcount="0" expanddistance="50" canvasheight="260"><text data="This is a 22-year-old with localization in partial epilepsy, status post left stereostatic visualized laser amygdalophippocampectomy in June 2013.&#10;&#10;The only adjustment made for VNS was increasing her heart rate sensitivity to 50% so the autostim will not go off quite as much. &#10;&#10;She can go ahead and decrease Topamax 50 mg at night for 1 month and discontinue this.&#10;&#10;The next medication we may decrease his Keppra. She may need a LTG level in the future&#10;&#10;&#10;Follow-up in 2 months with me in 4 months with Dr. Carter" /><canvas /></sketchpad></sketchpads></ass><plan><sketchpads /></plan><vitals><sketchpads /></vitals><hpi><sketchpads><sketchpad section="hpi" index="0" srcID="hpiDiv" expandcount="1" expanddistance="50" canvasheight="1003"><text data="Taylor has had 3 seizures since November. She did not lose consciousness with the seizures. She describes them as &quot;feeling something in the back of her head and then having a headache.&quot; At her last visit, she had an EEG which showed no epileptiform activity, although did show some occasional left temporal polymorphic delta theta activity consistent with left temporal cerebral dysfunction. Dr. Carter decreased her Topamax from 200 mg a day to 100 mg at night. She does not feel she has had increased headaches or seizures and the decreased.&#10;&#10;She continues on Lamictal 500 mg in the morning and 400 mg at night, as well as Keppra 2000 mg twice daily. She is also on folic acid 1 mg daily. These medications were managed by her physician in Atlanta, however she is requesting refills from us today.&#10;&#10;Taylor does note that she is tolerating VNS finding, however she doesn't always stump is off quite frequently during the day, when she is walking upstairs or generally just walking fast.&#10;&#10;&#10;Taylor is a 22-year-old right-handed Caucasian young lady seen in neurologic follow-up for localization focal, partial epilepsy with complex partial seizures, with intractable epilepsy, status post left stereostatic visualized laser amygdalohippocampectomy at Emory June 2013. She was felt to be an excellent candidate for VNS. She had a 106 model with auto stimulation placed 2016-06-07. She is accompanied by her mother today. &#10;&#10;They both report that the VNS has been a wonderful addition. Her last seizure was September 2. It was a complex partial seizure lasting approximately 2 minutes. &#10;&#10;She remains on a regimen of Lamictal, topiramate and Keppra she was last seen for a vagal nerve stimulator adjustment on November 11. The off time was decreased to 3 minutes and this has helped. Now, if she feels a seizure coming on and she was simply swiped the magnet. She would like to come off of Topamax as both she and her mother report that it has increased her word finding difficulty. This is particularly bad when she was on 200 mg twice a day.&#10;&#10;We will plan on decreasing the Topamax. I am somewhat concerned that she is having partial seizures based today on some abnormal eye movements that I haven't appreciated before. She is also repeating herself over and over which I haven't appreciated before.&#10;&#10;She is complaining of pain and sensitivity on the right side of her head as well for the past 2 weeks.&#10;&#10;She continues to work part time at a warehouse and she makes light houses as a craft. &#10;without problems.&#10;&#10;Semiology of partial seizures: she describes as a sharp massive head pain with changing colors in her vision followed by a dull ache or migraine &#10;&#10;She is on Keppra 1500 BID; lamictal 400 bid and topamax is 100 q am and 200 qhs; all brand name necessary. She is also on zyrtec nightly for allergies.&#10;" /><canvas /></sketchpad></sketchpads></hpi><pe><sketchpads /></pe></bailout><history patientid="20100784" MedDisplayFilter="ckshowall"><FaceSheetSection><FaceSheetSectionStatus FacesheetSectionID="1" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="2" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="3" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="4" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="5" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="6" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="7" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="8" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="9" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="10" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="11" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="12" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="13" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="14" FacesheetSectionStatusID="1" /><FaceSheetSectionStatus FacesheetSectionID="15" FacesheetSectionStatusID="1" /></FaceSheetSection><vitals><clinicalvitalgroup clinicalvitalgroupid="133883" datetimeadded="Nov 29 2016 10:48:46:087AM" isorthostatic="0" patientid="20100784" visitid="355779"><clinicalvital clinicalvitalgroupid="133883" clinicalvitalid="134432" bpposition="Sitting" datetimetaken="2016-10-30 10:48:00" diastolicbp="70" systolicbp="100" laterality="" cuffsize="" hrregularity="Regular" heartrate="70" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1338" username="twashington" vitalsite="" weight="91625.658740000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="132670" datetimeadded="Nov 9 2016 11:28:38:597AM" isorthostatic="0" patientid="20100784" visitid="353741"><clinicalvital clinicalvitalgroupid="132670" clinicalvitalid="133215" bpposition="Sitting" datetimetaken="2016-10-10 11:26:00" diastolicbp="70" systolicbp="108" laterality="" cuffsize="" hrregularity="Regular" heartrate="78" height="167.640000000000000" resprate="" temperature="" tempmethod="" userid="1339" username="kjackson" vitalsite="" weight="92532.843480000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="130446" datetimeadded="2016-09-05 8:04:39:913AM" isorthostatic="0" patientid="20100784" visitid="349978"><clinicalvital clinicalvitalgroupid="130446" clinicalvitalid="130984" bpposition="Sitting" datetimetaken="2016-09-05 08:04:00" diastolicbp="62" systolicbp="94" laterality="" cuffsize="" hrregularity="Regular" heartrate="92" height="167.640000000000000" resprate="" temperature="" tempmethod="" userid="1267" username="hwalker" vitalsite="" weight="90718.474000000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="129377" datetimeadded="2016-08-22 8:31:21:360AM" isorthostatic="0" patientid="20100784" visitid="348123"><clinicalvital clinicalvitalgroupid="129377" clinicalvitalid="129913" bpposition="Sitting" datetimetaken="2016-08-22 08:30:00" diastolicbp="78" systolicbp="116" laterality="" cuffsize="" hrregularity="Regular" heartrate="80" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1337" username="ebuckley" vitalsite="" weight="91172.066370000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="128052" datetimeadded="2016-08-01 8:54:46:397AM" isorthostatic="0" patientid="20100784" visitid="345725"><clinicalvital clinicalvitalgroupid="128052" clinicalvitalid="128582" bpposition="Sitting" datetimetaken="2016-08-01 08:54:00" diastolicbp="72" systolicbp="118" laterality="" cuffsize="" hrregularity="Regular" heartrate="70" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1337" username="ebuckley" vitalsite="" weight="68038.855500000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="127047" datetimeadded="2016-07-18 11:02:34:383AM" isorthostatic="0" patientid="20100784" visitid="344035"><clinicalvital clinicalvitalgroupid="127047" clinicalvitalid="127574" bpposition="Sitting" datetimetaken="2016-07-18 11:01:00" diastolicbp="70" systolicbp="110" laterality="" cuffsize="" hrregularity="Regular" heartrate="88" height="167.640000000000000" resprate="" temperature="" tempmethod="" userid="1325" username="dhaggins" vitalsite="" weight="90718.474000000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="125883" datetimeadded="2016-07-04 8:23:08:733AM" isorthostatic="0" patientid="20100784" visitid="342112"><clinicalvital clinicalvitalgroupid="125883" clinicalvitalid="126406" bpposition="Sitting" datetimetaken="2016-07-04 08:22:00" diastolicbp="70" systolicbp="102" laterality="" cuffsize="" hrregularity="Regular" heartrate="84" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1240" username="cboger" vitalsite="" weight="90718.474000000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="125266" datetimeadded="2016-06-20 1:59:01:133PM" isorthostatic="0" patientid="20100784" visitid="340768"><clinicalvital clinicalvitalgroupid="125266" clinicalvitalid="125786" bpposition="Sitting" datetimetaken="2016-06-20 13:58:00" diastolicbp="70" systolicbp="100" laterality="" cuffsize="" hrregularity="Regular" heartrate="84" height="167.640000000000000" resprate="" temperature="" tempmethod="" userid="1325" username="dhaggins" vitalsite="" weight="89811.289260000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="121006" datetimeadded="2016-04-17 11:27:30:337AM" isorthostatic="0" patientid="20100784" visitid="333247"><clinicalvital clinicalvitalgroupid="121006" clinicalvitalid="121518" bpposition="Sitting" datetimetaken="2016-04-17 11:27:00" diastolicbp="78" systolicbp="112" laterality="" cuffsize="" hrregularity="Regular" heartrate="82" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1252" username="tsims" vitalsite="" weight="91625.658740000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="91660" datetimeadded="2015-03-24 11:40:15:273AM" isorthostatic="0" patientid="20100784" visitid="283458"><clinicalvital clinicalvitalgroupid="91660" clinicalvitalid="92072" bpposition="Sitting" datetimetaken="2015-03-24 11:40:00" diastolicbp="78" systolicbp="124" laterality="" cuffsize="" hrregularity="Regular" heartrate="66" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1252" username="tsims" vitalsite="" weight="93440.028220000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="81500" datetimeadded="2014-11-10 1:11:07:310PM" isorthostatic="0" patientid="20100784" visitid="266071"><clinicalvital clinicalvitalgroupid="81500" clinicalvitalid="81855" bpposition="Sitting" datetimetaken="2014-11-10 13:08:00" diastolicbp="76" systolicbp="118" laterality="" cuffsize="" hrregularity="Regular" heartrate="84" height="167.640000000000000" resprate="20" temperature="" tempmethod="" userid="1252" username="tsims" vitalsite="" weight="93440.028220000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="73650" datetimeadded="2014-08-10 3:30:25:057PM" isorthostatic="0" patientid="20100784" visitid="253938"><clinicalvital clinicalvitalgroupid="73650" clinicalvitalid="73967" bpposition="Sitting" datetimetaken="2014-08-10 15:30:00" diastolicbp="66" systolicbp="118" laterality="" cuffsize="" hrregularity="Regular" heartrate="73" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1151" username="mslater" vitalsite="" weight="90718.474000000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="38778" datetimeadded="2013-06-16 11:03:15:107AM" isorthostatic="0" patientid="20100784" visitid="198286"><clinicalvital clinicalvitalgroupid="38778" clinicalvitalid="38932" bpposition="Sitting" datetimetaken="2013-06-16 11:03:00" diastolicbp="78" systolicbp="120" laterality="" cuffsize="" hrregularity="Regular" heartrate="83" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1151" username="mslater" vitalsite="" weight="78017.887640000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="22935" datetimeadded="2012-12-08 2:01:33:077PM" isorthostatic="0" patientid="20100784" visitid="170246"><clinicalvital clinicalvitalgroupid="22935" clinicalvitalid="23002" bpposition="Sitting" datetimetaken="2012-12-08 14:01:00" diastolicbp="76" systolicbp="120" laterality="" cuffsize="" hrregularity="Regular" heartrate="84" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1158" username="dmobley" vitalsite="" weight="78471.480010000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="17709" datetimeadded="2012-09-25 10:12:26:967AM" isorthostatic="0" patientid="20100784" visitid="160469"><clinicalvital clinicalvitalgroupid="17709" clinicalvitalid="17759" bpposition="Sitting" datetimetaken="2012-09-25 10:12:00" diastolicbp="82" systolicbp="108" laterality="" cuffsize="" hrregularity="Regular" heartrate="76" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1151" username="mslater" vitalsite="" weight="79832.257120000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="15179" datetimeadded="2012-08-22 11:04:10:217AM" isorthostatic="0" patientid="20100784" visitid="155363"><clinicalvital clinicalvitalgroupid="15179" clinicalvitalid="15216" bpposition="Sitting" datetimetaken="2012-08-22 11:03:00" diastolicbp="86" systolicbp="130" laterality="" cuffsize="" hrregularity="Regular" heartrate="78" height="167.640000000000000" resprate="18" temperature="" tempmethod="" userid="1151" username="mslater" vitalsite="" weight="74389.148680000000000" headcircumference="" spo2="" /></clinicalvitalgroup><clinicalvitalgroup clinicalvitalgroupid="12032" datetimeadded="2012-07-11 1:22:35:780PM" isorthostatic="0" patientid="20100784" visitid="149150"><clinicalvital clinicalvitalgroupid="12032" clinicalvitalid="12050" bpposition="Sitting" datetimetaken="2012-07-11 13:22:00" diastolicbp="82" systolicbp="136" laterality="" cuffsize="" hrregularity="Regular" heartrate="101" height="167.640000000000000" resprate="20" temperature="" tempmethod="" userid="1158" username="dmobley" vitalsite="" weight="79378.664750000000000" headcircumference="" spo2="" /></clinicalvitalgroup></vitals><medications recorded="1" PatientID="20100784" MedReconDate="Nov 29 2016 11:26AM" TransitionOfCareDate="" VisitID="355779" UserID="1083"><ClinicalPatMeds ClinicalPatMedID="361778" OriginalPatMedID="361778" PatientID="20100784" UserID="1252" CareProviderID="-1" StartDT="" EffectiveDT="" expiredate="" MedicationID="264890" OrderingDocumentID="0" OrderingSeqNumber="0" MedicationName="folic acid" InterfaceTransactionTypeID="0" MedicalConditionID="" MedicalCondition="" Description="folic acid 1 mg oral tablet" NameType="Disp" DrugNameID="4071" MedicationStrength="1" MedicationStrengthUnit="mg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="tablet" DoseQuantity="1" RouteID="24" Route="oral" Frequency="1" DurationUnitID="0" DurationUnitText="" NumRefills="0" DispenseAmount="0" DispenseUnit="tablet" Enable="1" StatusID="1" SIG="take 1 tablet (1 mg) by oral route once daily" SIGModeID="1" Status="Record" UserName="tsims" FirstName="Tiffany" MiddleName="" LastName="Sims" FullName="Sheley, Taylor" DocName="" DispenseAsWritten="0" DEACode="0" GCNSeqNo="2366" OSID="2842" FrequencyCode="QD" StartDateText="" SigInd="" ObsoleteDate="" FDBMNID="4071" FDBNDC="00591521601" CreateDate="2014-11-10 1:08PM" External="1" Expired="0" LastActionID="0" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" ePrescribeOnly="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="0" EPrescribeDate="" InterfaceEventID="0" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="310410" /><ClinicalPatMeds ClinicalPatMedID="507436" OriginalPatMedID="399291" PatientID="20100784" UserID="1150" CareProviderID="1009" StartDT="2016-04-23" EffectiveDT="2016-04-23" expiredate="" MedicationID="173420" OrderingDocumentID="0" OrderingSeqNumber="0" MedicationName="ibuprofen" InterfaceTransactionTypeID="1021" MedicalConditionID="" MedicalCondition="" Description="ibuprofen 800 mg oral tablet" NameType="Disp" DrugNameID="5042" MedicationStrength="800" MedicationStrengthUnit="mg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="" DoseQuantity="0" RouteID="24" Route="oral" Frequency="" DurationUnitID="0" DurationUnitText="" NumRefills="5" DispenseAmount="30" DispenseUnit="Tablet" Enable="1" StatusID="2" SIG="TAKE ONE TABLET BY MOUTH EVERY 8 HOURS WITH FOOD AS NEEDED FOR SORENESS" SIGModeID="0" Status="Prescribe" UserName="agibson" FirstName="April" MiddleName="" LastName="Gibson" FullName="Sheley, Taylor" DocName="Jessica F. Carter MD" DispenseAsWritten="0" DEACode="0" GCNSeqNo="8350" OSID="0" FrequencyCode="" StartDateText="2016-04-23" SigInd="" ObsoleteDate="" FDBMNID="5042" FDBNDC="00440162830" CreateDate="2016-04-23 11:02AM" External="0" Expired="0" LastActionID="1" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="1" EPrescribeDate="2016-04-23 11:02:21 AM" InterfaceEventID="358210" StatusName="Successful" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="197807" /><ClinicalPatMeds ClinicalPatMedID="528973" OriginalPatMedID="528973" PatientID="20100784" UserID="1325" CareProviderID="-1" StartDT="" EffectiveDT="" expiredate="" MedicationID="545075" OrderingDocumentID="0" OrderingSeqNumber="0" MedicationName="Keppra" InterfaceTransactionTypeID="0" MedicalConditionID="" MedicalCondition="" Description="Keppra 1,000 mg oral tablet" NameType="Disp" DrugNameID="16999" MedicationStrength="1,000" MedicationStrengthUnit="mg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="tablet" DoseQuantity="1" RouteID="24" Route="oral" Frequency="2" DurationUnitID="0" DurationUnitText="" NumRefills="0" DispenseAmount="0" DispenseUnit="tablet" Enable="1" StatusID="1" SIG="take 2 tablet (1,000 mg) by oral route every 12 hours" SIGModeID="3" Status="Record" UserName="dhaggins" FirstName="Delores" MiddleName="" LastName="Haggins" FullName="Sheley, Taylor" DocName="" DispenseAsWritten="0" DEACode="0" GCNSeqNo="47077" OSID="0" FrequencyCode="Q12H" StartDateText="" SigInd="" ObsoleteDate="" FDBMNID="16999" FDBNDC="50474059766" CreateDate="2016-07-18 11:01AM" External="1" Expired="0" LastActionID="0" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="0" EPrescribeDate="" InterfaceEventID="0" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="387003" /><ClinicalPatMeds ClinicalPatMedID="506005" OriginalPatMedID="90471" PatientID="20100784" UserID="1078" CareProviderID="1009" StartDT="2016-04-17" EffectiveDT="2016-04-17" expiredate="2016-09-14" MedicationID="249474" OrderingDocumentID="1111731" OrderingSeqNumber="1" MedicationName="Lamictal" InterfaceTransactionTypeID="0" MedicalConditionID="" MedicalCondition="" Description="Lamictal 100 mg oral tablet" NameType="Disp" DrugNameID="8084" MedicationStrength="100" MedicationStrengthUnit="mg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="tablet" DoseQuantity="0" RouteID="24" Route="oral" Frequency="" Duration="30" DurationUnitID="0" DurationUnitText="" NumRefills="4" DispenseAmount="180" DispenseUnit="tablets" Enable="1" StatusID="2" SIG="Take 3 Capsule po BID" SIGModeID="3" Status="Prescribe" UserName="jcarter" FirstName="Jessica" MiddleName="" LastName="Carter" FullName="Sheley, Taylor" DocName="Jessica F. Carter MD" DispenseAsWritten="1" DEACode="0" GCNSeqNo="17871" OSID="0" FrequencyCode="" StartDateText="2016-04-17" SigInd="" ObsoleteDate="" FDBMNID="8084" FDBNDC="00173064255" CreateDate="2016-04-17 12:06PM" External="0" Expired="1" LastActionID="2" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="0" EPrescribeDate="" InterfaceEventID="0" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="105018"><Indication ClinicalPatMedID="506005" Code="06.345900" Description="Epilepsy" IndicationType="FDBIndication" /></ClinicalPatMeds><ClinicalPatMeds ClinicalPatMedID="528974" OriginalPatMedID="528974" PatientID="20100784" UserID="1325" CareProviderID="-1" StartDT="" EffectiveDT="" expiredate="" MedicationID="229897" OrderingDocumentID="0" OrderingSeqNumber="0" MedicationName="Lamictal" InterfaceTransactionTypeID="0" MedicalConditionID="" MedicalCondition="" Description="Lamictal 200 mg oral tablet" NameType="Disp" DrugNameID="8084" MedicationStrength="200" MedicationStrengthUnit="mg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="tablet" DoseQuantity="1" RouteID="24" Route="oral" Frequency="1" DurationUnitID="0" DurationUnitText="" NumRefills="0" DispenseAmount="0" DispenseUnit="tablet" Enable="1" StatusID="1" SIG="take 1 tablet (200 mg) by oral route once daily" SIGModeID="1" Status="Record" UserName="dhaggins" FirstName="Delores" MiddleName="" LastName="Haggins" FullName="Sheley, Taylor" DocName="" DispenseAsWritten="0" DEACode="0" GCNSeqNo="22551" OSID="672" FrequencyCode="QD" StartDateText="" SigInd="" ObsoleteDate="" FDBMNID="8084" FDBNDC="00173064460" CreateDate="2016-07-18 11:01AM" External="0" Expired="0" LastActionID="0" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="0" EPrescribeDate="" InterfaceEventID="0" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="108782" /><ClinicalPatMeds ClinicalPatMedID="89109" OriginalPatMedID="89109" PatientID="20100784" UserID="1158" CareProviderID="-1" StartDT="" EffectiveDT="" expiredate="" MedicationID="213323" OrderingDocumentID="0" OrderingSeqNumber="0" MedicationName="levothyroxine" InterfaceTransactionTypeID="0" MedicalConditionID="" MedicalCondition="" Description="levothyroxine 75 mcg Oral tablet" NameType="Disp" DrugNameID="1423" MedicationStrength="75" MedicationStrengthUnit="mcg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="tablet" DoseQuantity="1" RouteID="24" Route="Oral" Frequency="1" DurationUnitID="0" DurationUnitText="" NumRefills="0" DispenseAmount="0" DispenseUnit="tablet" Enable="1" StatusID="1" SIG="take 1 tablet (75 mcg) by oral route once daily" SIGModeID="1" Status="Record" UserName="dmobley" FirstName="Deidre" MiddleName="" LastName="Mobley" FullName="Sheley, Taylor" DocName="" DispenseAsWritten="0" DEACode="0" GCNSeqNo="6650" OSID="4684" FrequencyCode="QD" StartDateText="" SigInd="" ObsoleteDate="" FDBMNID="1423" FDBNDC="00378180510" CreateDate="2012-07-11 1:17PM" External="1" Expired="0" LastActionID="0" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" ePrescribeOnly="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="0" EPrescribeDate="" InterfaceEventID="0" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="966165" /><ClinicalPatMeds ClinicalPatMedID="533210" OriginalPatMedID="533210" PatientID="20100784" UserID="1310" CareProviderID="1009" StartDT="2016-08-01" EffectiveDT="2016-08-01" expiredate="2017-01-28" MedicationID="285722" OrderingDocumentID="0" OrderingSeqNumber="0" MedicationName="topiramate" InterfaceTransactionTypeID="0" MedicalConditionID="" MedicalCondition="" Description="topiramate 100 mg oral tablet" NameType="Disp" DrugNameID="13560" MedicationStrength="100" MedicationStrengthUnit="mg" DoseFormID="81" DoseForm="tablet" DoseQuantityUnit="tablet" DoseQuantity="1" RouteID="24" Route="oral" Frequency="2" Duration="90" DurationUnitID="0" DurationUnitText="" NumRefills="1" DispenseAmount="180" DispenseUnit="tablets" Enable="1" StatusID="2" SIG="take 1 tablet (100 mg) by oral route 2 times per day for 90 days" SIGModeID="1" Status="Prescribe" UserName="fgarnett" FirstName="Faith" MiddleName="" LastName="Garnett" FullName="Sheley, Taylor" DocName="Jessica F. Carter MD" DispenseAsWritten="0" DEACode="0" GCNSeqNo="26170" OSID="333" FrequencyCode="BID" StartDateText="2016-08-01" SigInd="" ObsoleteDate="" FDBMNID="13560" FDBNDC="10544048960" CreateDate="2016-08-01 10:31AM" External="0" Expired="0" LastActionID="0" ValidScript="1" SampleGiven="0" AdministeredInOffice="0" Maintenance="0" Class="0" PreferredPharmacyID="1327" PreferredPharmacy="KROGER ATLANTA 423" PreferredPharmacyDisplayName="KROGER ATLANTA 423" PharmacyAddress1="5633 HIGHWAY 21" PharmacyAddress2="" PharmacyCity="RINCON" PharmacyState="GA" PharmacyZip="31326" PharmacyPhone="((PHONE_NUMBER))" PharmacyFax="((PHONE_NUMBER))" PharmacyInDrFirst="011423" DOB="1994-01-27T00:00:00" MedicationPreferredPharmacyID="1327" MedicationPreferredPharmacy="KROGER ATLANTA 423" MedicationPreferredPharmacyDisplayName="KROGER ATLANTA 423" MedicationPharmacyAddress1="5633 HIGHWAY 21" MedicationPharmacyAddress2="" MedicationPharmacyCity="RINCON" MedicationPharmacyState="GA" MedicationPharmacyZip="31326" MedicationPharmacyPhone="((PHONE_NUMBER))" MedicationPharmacyFax="((PHONE_NUMBER))" EPrescribeDone="0" EPrescribeDate="" InterfaceEventID="0" NewRx="N" ControlledSub="N" PControlledSub="N" PharmacyNote="" Comment="" PharmacyRequestID="0" RxNorm="151228"><Indication ClinicalPatMedID="533210" Code="345.40" Description="Seizure, complex partial" ProblemListID="16705" IndicationType="ProblemList" /></ClinicalPatMeds></medications><pmh><item caption="Epilepsy" V="ICD9-345.01" altsystem="ICD9" altsystemcode="345.01" notes="" dateOnset="" CreateDate="2012-07-11 1:19PM" CatItemID="2973" PL="0" SNOMEDCodeUS="TBD" LastChanged="2012-07-11 1:19PM" ICD9Code="345.01" ICD10Code="" SNOMEDCode="84757009" LexicalID="16067" HoverText="Epilepsy&#10;ICD-9: 345.01&#10;ICD-10: &#10;SNOMED: 84757009" /><item caption="Headaches" V="ICD9-346.20" altsystem="ICD9" altsystemcode="346.20" notes="" dateOnset="" CreateDate="2012-07-11 1:18PM" CatItemID="3045" PL="0" SNOMEDCodeUS="TBD" LastChanged="2012-07-11 1:18PM" ICD9Code="346.20" ICD10Code="" SNOMEDCode="" LexicalID="45537" HoverText="Headaches&#10;ICD-9: 346.20&#10;ICD-10: &#10;SNOMED: " /><item caption="Seizure, complex partial" V="ICD9-345.40" altsystem="ICD9" altsystemcode="345.40" notes="" dateOnset="2012-07-11" CreateDate="2012-07-11 4:09PM" CatItemID="6976" PL="1" problemlistid="16705" statusid="1001" displayorder="0" SNOMEDCodeUS="TBD" LastChanged="2012-07-11 4:09PM" ICD9Code="345.40" ICD10Code="" SNOMEDCode="" LexicalID="" ProblemNote="" ProblemStartDate="2012-07-11" IsMapICD10="0" HoverText="Seizure, complex partial&#10;ICD-9: 345.40&#10;ICD-10: &#10;SNOMED: "><modifier /><occurrences /></item><item caption="Thyroid Disease" V="ICD9-246.9" altsystem="ICD9" altsystemcode="246.9" notes="" dateOnset="" CreateDate="2012-07-11 1:19PM" CatItemID="3079" PL="0" SNOMEDCodeUS="TBD" LastChanged="2012-07-11 1:19PM" ICD9Code="246.9" ICD10Code="E07.9" SNOMEDCode="14304000" LexicalID="318145" HoverText="Thyroid disease&#10;ICD-9: 246.9&#10;ICD-10: E07.9&#10;SNOMED: 14304000" /></pmh><allergy_list><item caption=" NO KNOWN ALLERGIES" V="FDBALLERGP-900388" altsystem="FDBALLERGP" altsystemcode="900388" notes="" reaction="" CatItemID="4019" DateOfEntry="2012-07-11 1:17PM" dateOnset="" PL="0" ICD9Code="" ICD10Code="" SNOMEDCode="" /><item caption=" NO KNOWN DRUG ALLERGIES" V="FDBALLERGP-900590" altsystem="FDBALLERGP" altsystemcode="900590" notes="- Phreesia 2016-06-20" reaction="" CatItemID="9497" DateOfEntry="2016-06-20 1:38PM" dateOnset="" PL="0" ICD9Code="" ICD10Code="" SNOMEDCode="" /><item caption="No Known Food or Environmental Allergies" V="" altsystem="" altsystemcode="" notes="- Phreesia 2016-06-20" reaction="" CatItemID="19893" DateOfEntry="2016-06-20 1:38PM" dateOnset="" PL="0" ICD9Code="" ICD10Code="" SNOMEDCode="" /></allergy_list><surgery_list><item CatItemID="1835" caption="Adenoidectomy" V="" altsystem="" altsystemcode="" notes="" procedureDate="" PL="0" ICD9Code="" ICD10Code="" SNOMEDCode="" CPTCode="" /><item CatItemID="3138" caption="Tonsillectomy" V="" altsystem="" altsystemcode="" notes="" procedureDate="" PL="0" ICD9Code="" ICD10Code="" SNOMEDCode="" CPTCode="" /><item CatItemID="8960" caption="Tube placement in ears" V="" altsystem="" altsystemcode="" notes="when pt was 18 mos" procedureDate="" PL="0" ICD9Code="" ICD10Code="" SNOMEDCode="" CPTCode="" /></surgery_list><social_history><item CatItemID="1383" caption="Alcohol" V="1004059815053" altsystem="" altsystemcode="1004059815053" notes="" agestart="" agestop="" status="4" ScreeningDate="" SubstanceUseStatusDesc="Never" quantity="" PL="0" category="Substance Use" ICD9Code="" ICD10Code="" SNOMEDCode="" /><item CatItemID="1363" caption="Caffeine" V="" altsystem="" altsystemcode="" notes="" agestart="" agestop="" status="2" ScreeningDate="" SubstanceUseStatusDesc="Current some day" quantity="" PL="0" category="Substance Use" ICD9Code="" ICD10Code="" SNOMEDCode="" /><item CatItemID="1627" caption="Minimal Amount of Exercise (Once weekly or less)" V="" altsystem="" altsystemcode="" notes="" agestart="" agestop="" ScreeningDate="" quantity="" PL="0" category="Exercise" ICD9Code="" ICD10Code="" SNOMEDCode="" /><item CatItemID="1386" caption="Tobacco" V="1004059815052" altsystem="" altsystemcode="1004059815052" notes="" agestart="" agestop="" status="4" ScreeningDate="" SubstanceUseStatusDesc="Never" quantity="" PL="0" category="Substance Use" ICD9Code="305.1" ICD10Code="Z72.0" SNOMEDCode="110483000" /></social_history><genetic /><family_history><item CatItemID="4053" caption="alzheimer's dementia" V="" altsystem="" altsystemcode="" PL="0" ICD9Code="331.0" ICD10Code="G30.9" SNOMEDCode="26929004" HoverText="Alzheimer's dementia&#10;ICD-9: 331.0&#10;ICD-10: G30.9&#10;SNOMED: 26929004"><modifier /></item><item CatItemID="4015" caption="cancer" V="" altsystem="" altsystemcode="" PL="0" ICD9Code="199.1" ICD10Code="C80.1" SNOMEDCode="363346000" HoverText="Cancer&#10;ICD-9: 199.1&#10;ICD-10: C80.1&#10;SNOMED: 363346000"><modifier /></item><item CatItemID="3097" caption="Diabetes, Type II" V="ICD9-250.00" altsystem="ICD9" altsystemcode="250.00" PL="0" ICD9Code="250.00" HoverText="Diabetes, Type II&#10;ICD-9: 250.00&#10;ICD-10: &#10;SNOMED: "><modifier /></item><item CatItemID="3737" caption="Hypertension" V="ICD9-401.0" altsystem="ICD9" altsystemcode="401.0" PL="0" ICD9Code="401.0" SNOMEDCode="38341003" HoverText="Hypertension&#10;ICD-9: 401.0&#10;ICD-10: &#10;SNOMED: 38341003"><modifier /></item><item CatItemID="3117" caption="Thyroid Disease" V="ICD9-246.9" altsystem="ICD9" altsystemcode="246.9" PL="0" ICD9Code="246.9" ICD10Code="E07.9" SNOMEDCode="14304000" HoverText="Thyroid disease&#10;ICD-9: 246.9&#10;ICD-10: E07.9&#10;SNOMED: 14304000"><modifier /></item></family_history><reproductive_history><pregnancy /></reproductive_history><orders><order orderid="228187" createdate="2016-11-29T11:26:59.727" IssuingDocumentID="1236003" ProcedureCode="95816" unit="1.000000000000000e+000" description="EEG Interpretation; AWAKE AND DROWSY" statusid="1004" disabled="0"><SLU StatusDescription="Returned" /></order><order orderid="228312" createdate="2016-11-29T16:15:40.780" IssuingDocumentID="1236379" ProcedureCode="95819" unit="1.000000000000000e+000" description="SLEEP DEPRIVED EEG (EEG AWAKE AND ASLEEP)" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="228184" createdate="2016-11-29T11:06:37.440" IssuingDocumentID="1236087" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS interrogation" statusid="1007" disabled="0"><SLU StatusDescription="Scheduled" /></order><order orderid="222148" createdate="2016-10-05T08:04:00.260" IssuingDocumentID="1203189" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS interrogation" statusid="1007" disabled="0"><SLU StatusDescription="Scheduled" /></order><order orderid="220149" createdate="2016-09-21T08:39:42.133" IssuingDocumentID="1192747" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS interrogation" statusid="1007" disabled="0"><SLU StatusDescription="Scheduled" /></order><order orderid="217858" createdate="2016-08-31T10:48:38.167" IssuingDocumentID="1180388" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS interrogation" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="217824" createdate="2016-08-31T09:52:57.917" IssuingDocumentID="1180288" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS interrogation" statusid="1003" disabled="0"><SLU StatusDescription="Cancelled" /></order><order orderid="217977" createdate="2016-08-31T19:43:08.580" IssuingDocumentID="1180204" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS programming" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="213538" createdate="2016-08-03T10:10:42.650" IssuingDocumentID="1159962" ProcedureCode="95974" unit="1.000000000000000e+000" description="VNS interrogation" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="204207" createdate="2016-05-19T15:09:11.457" IssuingDocumentID="1113767" ProcedureCode="CONSU" unit="1.000000000000000e+000" description="ENT" statusid="1001" disabled="0"><SLU StatusDescription="Pending" /></order><order orderid="203783" createdate="2016-05-17T13:09:53.213" IssuingDocumentID="1111731" ProcedureCode="61885, 61886" unit="1.000000000000000e+000" description="Vagal nerve stimulator implant" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="131986" createdate="2014-12-10T13:56:20.690" IssuingDocumentID="741085" ProcedureCode="85025" unit="1.000000000000000e+000" description="CBC w diff" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="131987" createdate="2014-12-10T13:56:20.707" IssuingDocumentID="741085" ProcedureCode="80053" unit="1.000000000000000e+000" description="CMP" statusid="1001" disabled="0"><SLU StatusDescription="Pending" /></order><order orderid="131988" createdate="2014-12-10T13:56:20.720" IssuingDocumentID="741085" ProcedureCode="80299" unit="1.000000000000000e+000" description="Lamotrigine-Lamictal" statusid="1001" disabled="0"><SLU StatusDescription="Pending" /></order><order orderid="131989" createdate="2014-12-10T13:56:20.730" IssuingDocumentID="741085" ProcedureCode="80299" unit="1.000000000000000e+000" description="Levetiracet-Keppra" statusid="1001" disabled="0"><SLU StatusDescription="Pending" /></order><order orderid="131990" createdate="2014-12-10T13:56:20.743" IssuingDocumentID="741085" ProcedureCode="80201" unit="1.000000000000000e+000" description="Topiramate (Topamax) blood level" statusid="1001" disabled="0"><SLU StatusDescription="Pending" /></order><order orderid="34717" createdate="2013-01-14T11:45:53.800" IssuingDocumentID="258504" ProcedureCode="80299" unit="1.000000000000000e+000" description="Keppra level" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="34718" createdate="2013-01-14T11:45:53.800" IssuingDocumentID="258504" ProcedureCode="80299" unit="1.000000000000000e+000" description="Lamotrigine level" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="16972" createdate="2012-08-10T16:12:01.273" IssuingDocumentID="166845" ProcedureCode="95819" unit="1.000000000000000e+000" description="Electroencephalogram (EEG); awake and asleep" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order><order orderid="16976" createdate="2012-08-10T16:16:54.603" IssuingDocumentID="166845" ProcedureCode="CONSU" unit="1.000000000000000e+000" description="OUTSIDE CONSULT; consult to emory and mcg for epilepsy monitoring. needs referral to epilepsy for presurgical evaluation" statusid="1006" disabled="0"><SLU StatusDescription="Reviewed" /></order></orders><alerts /><OverrideReason><ClinicalAlertOverrideReason ReasonID="1" Desc="Patient has been made aware of clinical needs" /><ClinicalAlertOverrideReason ReasonID="2" Desc="Refused: Patient Preference" /><ClinicalAlertOverrideReason ReasonID="3" Desc="Refused: Financial Coverage" /><ClinicalAlertOverrideReason ReasonID="4" Desc="Deferred: Clinician Reasoning" /><ClinicalAlertOverrideReason ReasonID="5" Desc="Other" /></OverrideReason><immunizations><ImmPreviousDoses /><ImmDueDoses><ImmDueDose PatientID="20100784" VaccineID="11" VacSeriesID="13" VacSeriesVersionDoseID="47" VacSeriesVersionID="13" DoseTypeID="0" SeriesDefault="1" DisplayOrder="24" AdultOrChild="2" VaccineName="HPV" VacSeriesName="HPV" PatAgeInDays="8398" DOB="1993-12-28" DoseNo="1" DateDue="2002-12-28" DateOverDue="2019-12-28" RecommendedAge="9" RecommendedAgeUOM="Years" OverdueAge="26" OverdueAgeUOM="Years" MaxAge="0" MaxAgeUOM="" CatchUpInterval="0" CatchUpIntervalUOM="Days" CatchUpOverDue="0" CatchUpOverDueUOM="Days" A="1" B="0" C="0" D="0" E="0" Score="16" Status="Due" CareProviderID="-1" Notes="" CatchupApplies="0" Frequency="0" FrequencyUOM="Days" DoseNote="HPV inital dose." PatientConfig="SHOW" QuestionableDose="0" /><ImmDueDose PatientID="20100784" VaccineID="11" VacSeriesID="13" VacSeriesVersionDoseID="48" VacSeriesVersionID="13" DoseTypeID="2" SeriesDefault="1" DisplayOrder="24" AdultOrChild="2" VaccineName="HPV" VacSeriesName="HPV" PatAgeInDays="8398" DOB="1993-12-28" DoseNo="2" DateDue="2003-02-25" DateOverDue="2003-02-25" RecommendedAge="0" RecommendedAgeUOM="Years" OverdueAge="0" OverdueAgeUOM="Years" MaxAge="0" MaxAgeUOM="" CatchUpInterval="2" CatchUpIntervalUOM="Months" CatchUpOverDue="3" CatchUpOverDueUOM="Months" A="1" B="1" C="0" D="0" E="0" Score="24" Status="Overdue" CareProviderID="-1" Notes="" CatchupApplies="0" Frequency="0" FrequencyUOM="Days" DoseNote="First subsequent HPV dose." PatientConfig="SHOW" QuestionableDose="0" /><ImmDueDose PatientID="20100784" VaccineID="11" VacSeriesID="13" VacSeriesVersionDoseID="49" VacSeriesVersionID="13" DoseTypeID="2" SeriesDefault="1" DisplayOrder="24" AdultOrChild="2" VaccineName="HPV" VacSeriesName="HPV" PatAgeInDays="8398" DOB="1993-12-28" DoseNo="3" DateDue="2003-04-27" DateOverDue="2003-04-27" RecommendedAge="0" RecommendedAgeUOM="Years" OverdueAge="0" OverdueAgeUOM="Years" MaxAge="0" MaxAgeUOM="" CatchUpInterval="4" CatchUpIntervalUOM="Months" CatchUpOverDue="5" CatchUpOverDueUOM="Months" A="1" B="1" C="0" D="0" E="0" Score="24" Status="Overdue" CareProviderID="-1" Notes="" CatchupApplies="0" Frequency="0" FrequencyUOM="Days" DoseNote="Second subsequent HPV dose." PatientConfig="SHOW" QuestionableDose="0" /></ImmDueDoses></immunizations><Injections /><PatHistModifiers><PatHistModifier PatHistModifierID="6" IsPreModifier="1" ModifierOrder="3" ModifierText="Severe" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="7" IsPreModifier="1" ModifierOrder="2" ModifierText="Moderate" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="8" IsPreModifier="1" ModifierOrder="1" ModifierText="Mild" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="9" IsPreModifier="1" ModifierOrder="7" ModifierText="Recurrent" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="10" IsPreModifier="0" ModifierOrder="1" ModifierText="Unresponsive to treatment" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="11" IsPreModifier="1" ModifierOrder="4" ModifierText="Resolved" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="12" IsPreModifier="1" ModifierOrder="5" ModifierText="Acute" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="13" IsPreModifier="1" ModifierOrder="6" ModifierText="Chronic" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="14" IsPreModifier="0" ModifierOrder="2" ModifierText="Stable" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="15" IsPreModifier="0" ModifierOrder="3" ModifierText="Improving" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="16" IsPreModifier="0" ModifierOrder="4" ModifierText="Worsening" Type="mod" Enabled="1" /><PatHistModifier PatHistModifierID="17" IsPreModifier="1" ModifierOrder="1" ModifierText="Left" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="18" IsPreModifier="1" ModifierOrder="2" ModifierText="Right" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="19" IsPreModifier="1" ModifierOrder="3" ModifierText="Bilateral" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="20" IsPreModifier="1" ModifierOrder="4" ModifierText="Upper" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="21" IsPreModifier="1" ModifierOrder="5" ModifierText="Lower" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="22" IsPreModifier="1" ModifierOrder="6" ModifierText="Posterior" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="23" IsPreModifier="1" ModifierOrder="7" ModifierText="Anterior" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="24" IsPreModifier="1" ModifierOrder="8" ModifierText="Inferior" Type="loc" Enabled="1" /><PatHistModifier PatHistModifierID="25" IsPreModifier="1" ModifierOrder="9" ModifierText="Superior" Type="loc" Enabled="1" /></PatHistModifiers><ProblemListStatuses><ProblemListStatus ProblemListStatusID="1001" ProblemListStatusDesc="Active" Enabled="1" /><ProblemListStatus ProblemListStatusID="1002" ProblemListStatusDesc="Inactive" Enabled="1" /><ProblemListStatus ProblemListStatusID="1003" ProblemListStatusDesc="Resolved" Enabled="1" /></ProblemListStatuses><tasklists /><FlowSheets><FlowSheet FlowsheetID="1065" FlowSheetName="0001 Asthma Assessment" Description="0001 Asthma Assessment" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1066" FlowSheetName="0012 Prenatal Care: Screening for HIV" Description="0012 Prenatal Care: Screening for Human Immunodeficiency Virus (HIV)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1067" FlowSheetName="0014 Prenatal Care: Anti-D Immune Globulin" Description="0014 Prenatal Care: Anti-D Immune Globulin" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1068" FlowSheetName="0018 Controlling High Blood Pressure" Description="0018 Controlling High Blood Pressure" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1069" FlowSheetName="0027 Smoking and Tobacco Use Cessation, Medical assistance" Description="0027 a. Advising Smokers +Tobacco Users to Quit, b. Discussing Cessation Medications, c. Discussing Cessation Strategies" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1070" FlowSheetName="0038 Childhood immunization Status" Description="0038 Childhood immunization Status" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1071" FlowSheetName="0041 Prev Care/Screening: Influenza Immunization, 50 and older" Description="0041 Preventive Care and Screening: Influenza Immunization for Patients 50 Years and Older" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1064" FlowSheetName="0047 Asthma Pharmacologic Therapy" Description="0047 Asthma Pharmacologic Therapy" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1072" FlowSheetName="0056 Diabetes: Foot Exam" Description="0056 Diabetes: Foot Exam" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1073" FlowSheetName="0067 CAD: Oral Antiplatelet Therapy Prescribed" Description="0067 Coronary Artery Disease (CAD): Oral Antiplatelet Therapy Prescribed for Patients with CAD" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1074" FlowSheetName="0070 CAD: Beta-Blocker Therapy for CAD Patients with Prior MI" Description="0070 Coronary Artery Disease (CAD): Beta-Blocker Therapy for CAD Patients with Prior Myocardial Infarction (MI)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1075" FlowSheetName="0074 CAD: Drug Therapy for Lowering LDL-Cholesterol" Description="0074 Coronary Artery Disease (CAD): Drug Therapy for Lowering LDL-Cholesterol" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1076" FlowSheetName="0081 Heart Failure: ACE Inhibitor or ARB Therapy for LVSD" Description="0081 Heart Failure: Angiotensin-Converting Enzyme (ACE) Inhibitor or Angiotensin Receptor Blocker (ARB) Therapy for LVSD" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="0" LastChangedUserName="System" LastChangedDate="2011-09-13 06:49 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1077" FlowSheetName="0083 Heart Failure: Beta-Blocker Therapy for LVSD" Description="0083 Heart Failure (HF): Beta-Blocker Therapy for Left Ventricular Systolic Dysfunction (LVSD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1078" FlowSheetName="0084 Heart Failure: Warfarin Therapy for Patients w/Atrial Fib" Description="0084 Heart Failure: Warfarin Therapy for Patients with Atrial Fibrillation" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1079" FlowSheetName="0086 Primary Open Angle Glaucoma (POAG): Optic Nerve Evaluation" Description="0086 Primary Open Angle Glaucoma (POAG): Optic Nerve Evaluation" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1080" FlowSheetName="0088 Diabetic Retinopathy: Macular Edema + Retinopathy Severity" Description="0088 Diabetic Retinopathy: Documentation of Presence or Absence of Macular Edema and Level of Severity of Retinopathy" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1081" FlowSheetName="0089 Diabetic Retinopathy: Communication w/Managing Physician" Description="0089 Diabetic Retinopathy: Communication with the Physician Managing On-going Diabetes Care" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1082" FlowSheetName="0385 Oncology Colon Cancer: Chemo for Stage III Colon Cancer" Description="0385 Oncology Colon Cancer: Chemotherapy for Stage III Colon Cancer Patients" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1083" FlowSheetName="0387 Oncology Breast Cancer" Description="0387 Oncology Breast Cancer: Hormonal Therapy for Stage IC-IIIC Estrogen Receptor/Progesterone Receptor (ER/PR) Positive Breast" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1085" FlowSheetName="0389 Prostate Cancer" Description="0389 Prostate Cancer: Avoidance of Overuse of Bone Scan for Staging Low Risk Prostate Cancer Patients" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:49 PM" LastChangedBy="0" LastChangedUserName="System" LastChangedDate="2011-09-13 06:49 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1084" FlowSheetName="0421 Adult Weight Screening and Follow-Up" Description="0421 Adult Weight Screening and Follow-Up" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:48 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:48 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1002" FlowSheetName="Anti Convulsant Flow Sheet" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2012-10-06 12:30 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1088" FlowSheetName="Anti-Convulsant Drug Level Flow Sheet" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2012-03-20 08:58 PM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2012-12-09 12:51 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1099" FlowSheetName="B12/Folate" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-05-02 08:09 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-05-02 08:09 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1000" FlowSheetName="Basic Metabolic Profile" Description="Basic Metabolic Profile" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="1012" LastChangedUserName="maldrich" LastChangedDate="2013-11-05 10:19 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1105" FlowSheetName="Care Plans &amp; Goals" Description="Care Plans &amp; Goals" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2014-07-22 09:26 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2014-07-22 09:26 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1093" FlowSheetName="Cognitive Profile" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-02-01 09:59 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-02-01 09:59 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1102" FlowSheetName="Comprehensive Metabollic Profile" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-06-28 11:12 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-06-28 11:12 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1097" FlowSheetName="Coumadin (INR)" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-04-28 09:45 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-04-28 09:45 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1095" FlowSheetName="Coumadin (INR) Flowsheet" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-04-28 09:44 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-05-04 12:39 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1096" FlowSheetName="Coumadin Flow Sheet" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-04-28 09:45 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-04-28 09:45 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1101" FlowSheetName="Coumdain (INR) Monitoring" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-05-04 12:38 PM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-05-04 12:38 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1092" FlowSheetName="CPK" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2012-12-09 09:40 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2012-12-09 09:40 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1090" FlowSheetName="Dementia Labs" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2012-12-09 09:36 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-02-01 10:00 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1089" FlowSheetName="EEG FLOWSHEET" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2012-05-09 11:30 AM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2012-05-09 11:30 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1006" FlowSheetName="Endocrine" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="0" LastChangedUserName="System" LastChangedDate="2009-02-21 10:37 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1003" FlowSheetName="Hematology" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="0" LastChangedUserName="System" LastChangedDate="2009-02-21 10:37 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1005" FlowSheetName="Hepatic Profile" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="1012" LastChangedUserName="maldrich" LastChangedDate="2013-06-23 10:03 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1004" FlowSheetName="Lipid Profile" Description="Cardiovascular" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="0" LastChangedUserName="System" LastChangedDate="2009-02-21 10:37 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1007" FlowSheetName="OB Lab Flowsheet" Description="Custom OB Lab Flowsheet" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="0" LastChangedUserName="System" LastChangedDate="2009-02-21 10:37 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1086" FlowSheetName="Office Visit" Description="Routine" AllData="0" TimeValue="3" TimeUnits="months" CreatedBy="1004" CreatedUserName="mknorr" CreateDate="2011-11-22 12:09 PM" LastChangedBy="1004" LastChangedUserName="mknorr" LastChangedDate="2011-11-22 12:10 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1091" FlowSheetName="Peripheral Neuropathy Labs" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2012-12-09 09:40 AM" LastChangedBy="1012" LastChangedUserName="maldrich" LastChangedDate="2014-08-06 10:00 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="1" /><FlowSheet FlowsheetID="1010" FlowSheetName="PQRS/MU: Asthma" Description="PQRS (#53,#64)/ MU (#0047,#0001): Asthma" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1012" FlowSheetName="PQRS/MU: Breast Cancer" Description="PQRS (#71,#99)/ MU (#0387): Breast Cancer" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1014" FlowSheetName="PQRS/MU: Coronary Artery Disease (CAD)" Description="PQRS (#6,#7,#114,#115,#118,#196,#197)/ MU (#0027,#0067,#0070,#0074): Coronary Artery Disease (CAD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1019" FlowSheetName="PQRS/MU: Diabetes Mellitus" Description="Diabetes Mellitus PQRS (#1,#2,#3,#117,#119,#126,#127,#163)/ MU (#0055,#0056,#0059,#0061,#0062,#0064)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1020" FlowSheetName="PQRS/MU: Diabetic Retinopathy" Description="PQRS (#18,#19)/ MU (#0088,#0089): Diabetic Retinopathy" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1025" FlowSheetName="PQRS/MU: Heart Failure" Description="PQRS (#5,#8,#84,#114,#115,#198,#199)/ MU (#0027,#0081,#0083,#0084): Heart Failure" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1028" FlowSheetName="PQRS/MU: Ischemic Vascular Disease (IVD)" Description="PQRS (#114,#115,#201,#202,#203,#204)/ MU (#0027,#0068,#0073): Ischemic Vascular Disease (IVD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1029" FlowSheetName="PQRS/MU: Major Depressive Disorder (MDD)" Description="PQRS (#9,#106,#107)/ MU (#0105): Major Depressive Disorder (MDD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1039" FlowSheetName="PQRS/MU: Preventive Care and Screening" Description="PQRS (#39,#48,#110,#111,#112,#113,#114,#115,#128,#173)/ MU (#0027,#0031,#0034,#0041,#0043,#0421): Preventive Care and Screening" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1038" FlowSheetName="PQRS/MU: Primary Open Angle Glaucoma (POAG)" Description="PQRS (#12,#141)/ MU (#0086): Primary Open Angle Glaucoma (POAG)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1040" FlowSheetName="PQRS/MU: Prostate Cancer" Description="PQRS (#102,#104,#105)/ MU (#0389): Prostate Cancer" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1094" FlowSheetName="PQRS: Epilepsy" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1004" CreatedUserName="mknorr" CreateDate="2013-03-30 09:54 AM" LastChangedBy="1004" LastChangedUserName="mknorr" LastChangedDate="2013-04-07 12:41 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1021" FlowSheetName="PQRS: 12-Lead Electrocardiogram (ECG)" Description="PQRS (#54,#55): 12-Lead Electrocardiogram (ECG)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1052" FlowSheetName="PQRS: Acute Bronchitis, Adult" Description="PQRS (#116): Acute Bronchitis, Adult" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1045" FlowSheetName="PQRS: Acute Myocardial Infarction (AMI)" Description="PQRS (#28): Acute Myocardial Infarction (AMI)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1009" FlowSheetName="PQRS: Acute Otitis Externa (AOE)" Description="PQRS (#91,#92,#93): Acute Otitis Externa (AOE)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1061" FlowSheetName="PQRS: Adenomatous Polyps" Description="PQRS (#185): Adenomatous Polyps" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1046" FlowSheetName="PQRS: Advance Care Plan" Description="PQRS (#47): Advance Care Plan" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1008" FlowSheetName="PQRS: Age-Related Macular Degeneration (AMD)" Description="PQRS (#14,#140): Age-Related Macular Degeneration (AMD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1058" FlowSheetName="PQRS: Autogenous Arterial Venous (AV) Fistula" Description="PQRS (#172): Autogenous Arterial Venous (AV) Fistula" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1011" FlowSheetName="PQRS: Back Pain" Description="PQRS (#148,#149, #150, #151): Back Pain" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1057" FlowSheetName="PQRS: Carotid Endarterectomy" Description="PQRS (#158): Carotid Endarterectomy" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1063" FlowSheetName="PQRS: Carotid Imaging Studies" Description="PQRS (#195): Carotid Imaging Studies" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1016" FlowSheetName="PQRS: Cataracts" Description="PQRS (#191,#192): Cataracts" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1050" FlowSheetName="PQRS: Catheter-Related Bloodstream Infections (CRBSI)" Description="PQRS (#76): Catheter-Related Bloodstream Infections (CRBSI)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1060" FlowSheetName="PQRS: Chiropractic Care" Description="PQRS (#182): Chiropractic Care" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1017" FlowSheetName="PQRS: Chronic Kidney Disease (CKD)" Description="PQRS (#121,#122,#123,#135,#153): Coronary Artery Disease (CAD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1049" FlowSheetName="PQRS: Chronic Lymphocytic Leukemia (CLL)" Description="PQRS (#70): Chronic Lymphocytic Leukemia (CLL)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1018" FlowSheetName="PQRS: Chronic Obstructive Pulmonary Disease (COPD)" Description="PQRS (#51,#52): Chronic Obstructive Pulmonary Disease (COPD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1054" FlowSheetName="PQRS: Clinical Depression" Description="PQRS (#134): Clinical Depression" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1051" FlowSheetName="PQRS: Colorectal Cancer" Description="PQRS (#100): Colorectal Cancer" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1015" FlowSheetName="PQRS: Community-Acquired Pneumonia (CAP)" Description="PQRS (#56,#57,#58,#59): Community-Acquired Pneumonia (CAP)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1013" FlowSheetName="PQRS: Coronary Artery Bypass Graft (CABG)" Description="PQRS (#43,#44,#164,#165,#166,#167,#168,#169,#170,#171): Coronary Artery Bypass Graft (CABG)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1059" FlowSheetName="PQRS: Elder Maltreatment Screen" Description="PQRS (#181): Elder Maltreatment Screen" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1022" FlowSheetName="PQRS: End Stage Renal Disease (ESRD)" Description="PQRS (#79,#81,#82,#174,#175): End Stage Renal Disease (ESRD)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1023" FlowSheetName="PQRS: Falls" Description="PQRS (#154,#155): Falls" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1024" FlowSheetName="PQRS: Functional Communication Measure" Description="PQRS (#209,#210,#211,#212,#213,#214,#215,#216): Functional Communication Measure" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1026" FlowSheetName="PQRS: Hepatitis C" Description="PQRS (#83,#84,#85,#86,#87,#88,#89,#90,#183,#184): Hepatitis C" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1027" FlowSheetName="PQRS: HIV/AIDS" Description="PQRS (#159,#160,#161,#162,#205,#206,#207,#208): HIV/AIDS" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1031" FlowSheetName="PQRS: Medication Reconciliation" Description="PQRS (#46,#130): Medication Reconciliation" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1032" FlowSheetName="PQRS: Melanoma" Description="PQRS (#136,#137,#138): Melanoma" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1048" FlowSheetName="PQRS: Multiple Myeloma" Description="PQRS (#69): Multiple Myeloma" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1030" FlowSheetName="PQRS: Myelodysplastic Syndrome (MDS) and Acute Leukemias" Description="PQRS (#67,#68): Myelodysplastic Syndrome (MDS) and Acute Leukemias" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1055" FlowSheetName="PQRS: Nuclear Medicine" Description="PQRS (#147): Nuclear Medicine" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1033" FlowSheetName="PQRS: Oncology" Description="PQRS (#143,#144,#156,#194): Oncology" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1034" FlowSheetName="PQRS: Osteoarthritis (OA)" Description="PQRS (#109,#142): Osteoarthritis (OA)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1035" FlowSheetName="PQRS: Osteoporosis" Description="PQRS (#24,#40,#41): Osteoporosis" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1036" FlowSheetName="PQRS: Otologic" Description="PQRS (#94,#188,#189,#190): Otologic" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1053" FlowSheetName="PQRS: Pain Assessment" Description="PQRS (#131): Pain Assessment" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1037" FlowSheetName="PQRS: Perioperative Care" Description="PQRS (#20,#21,#22,#23,#30,#45,#193): Perioperative Care" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1042" FlowSheetName="PQRS: Radiology" Description="PQRS (#145,#146): Radiology" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1041" FlowSheetName="PQRS: Rheumatoid Arthritis (RA)" Description="PQRS (#108,#176,#177,#178,#179,#180): Rheumatoid Arthritis (RA)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1043" FlowSheetName="PQRS: Stroke and Stroke Rehabilitation" Description="PQRS (#108,#176,#177,#178,#179,#180): Stroke and Stroke Rehabilitation" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1056" FlowSheetName="PQRS: Thoracic Surgery" Description="PQRS (#157): Thoracic Surgery" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1047" FlowSheetName="PQRS: Upper Respiratory Infection (URI), Pediatric" Description="PQRS (#65): Upper Respiratory Infection (URI), Pediatric" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1062" FlowSheetName="PQRS: Wound Care" Description="PQRS (#186): Wound Care" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1087" FlowSheetName="Routine Office Visit" Description="Routine" AllData="0" TimeValue="3" TimeUnits="months" CreatedBy="1004" CreatedUserName="mknorr" CreateDate="2011-11-22 12:10 PM" LastChangedBy="1004" LastChangedUserName="mknorr" LastChangedDate="2011-11-22 12:10 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1104" FlowSheetName="RPR w/Reflex to Titer" Description="RPR w/Reflex to Titer" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1012" CreatedUserName="maldrich" CreateDate="2013-11-05 10:47 AM" LastChangedBy="1012" LastChangedUserName="maldrich" LastChangedDate="2013-11-05 10:47 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1098" FlowSheetName="Sed Rate" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-05-02 08:08 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-05-02 08:08 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1106" FlowSheetName="SPEP" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1012" CreatedUserName="maldrich" CreateDate="2014-08-06 09:47 AM" LastChangedBy="1012" LastChangedUserName="maldrich" LastChangedDate="2014-08-06 09:47 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="1" /><FlowSheet FlowsheetID="1100" FlowSheetName="Thyroid Profile" Description="" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1009" CreatedUserName="wgarrett" CreateDate="2013-05-02 08:58 AM" LastChangedBy="1009" LastChangedUserName="wgarrett" LastChangedDate="2013-05-02 08:58 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1044" FlowSheetName="Urinary Incontinence" Description="PQRS (#48,#49,#50): Urinary Incontinence" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1" CreatedUserName="Admin" CreateDate="2011-09-13 06:45 PM" LastChangedBy="1" LastChangedUserName="Admin" LastChangedDate="2011-09-13 06:45 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /><FlowSheet FlowsheetID="1001" FlowSheetName="Vitals" Description="Vitals Flowsheet" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="0" CreatedUserName="System" CreateDate="2009-02-21 10:37 AM" LastChangedBy="1044" LastChangedUserName="chunt" LastChangedDate="2015-06-28 01:49 PM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="1" /><FlowSheet FlowsheetID="1103" FlowSheetName="Vitamin D (25-Hydroxy)" Description="Vitamin D (25-Hydroxy)" AllData="1" TimeValue="0" TimeUnits="" CreatedBy="1012" CreatedUserName="maldrich" CreateDate="2013-11-05 10:20 AM" LastChangedBy="1012" LastChangedUserName="maldrich" LastChangedDate="2013-11-05 10:28 AM" Enabled="1" UserFavorite="0" IsAttachedToPatient="0" GroupByDateTime="0" /></FlowSheets><confidential /><ClinicalFormGroups /><ProblemListSort ProblemListSortID="-1" ProblemListSortName="Default View"><Sort ICD9Code="345.40" PatHistSectionDescription="pmh" PatHistProblemListID="16705" ProblemCategory="DO NOT USE" ProblemDescription="Seizure, complex partial" ProblemType="Medical" ProblemID="6976" ProblemStatus="1001" DisplayOrder="0"><modifier /></Sort></ProblemListSort><Visits><Visit VisitID="362300" VisitTypeID="1001" VisitTypeName="Office Visit" CareProviderID="1012" CareProviderName="Moye, Angie R. NP" BillableCareProviderID="1009" BillableCareProviderName="Carter, Jessica F. MD" FromDateTime="2016-12-25 09:36:23" CheckedOutFlag="0" ReleasedToPrimePatient="0" /><Visit VisitID="355829" VisitTypeID="1002" VisitTypeName="Procedure" CareProviderID="1009" CareProviderName="Carter, Jessica F. MD" BillableCareProviderID="1009" BillableCareProviderName="Carter, Jessica F. MD" FromDateTime="2016-10-30 12:13:29" Quality="1/4" CheckedOutFlag="1" ReleasedToPrimePatient="0" ThroughDate="2016-11-29T14:24:31.530" /><Visit VisitID="353741" VisitTypeID="1001" VisitTypeName="Office Visit" CareProviderID="1012" CareProviderName="Moye, Angie R. NP" BillableCareProviderID="1009" BillableCareProviderName="Carter, Jessica F. MD" FromDateTime="2016-10-10 11:03:26" PrimaryDiagnosis="Partial symptomatic epilepsy with complex partial seizures, intractable, without status epilepticus - G40.219" Quality="1/4" CheckedOutFlag="1" ReleasedToPrimePatient="0" ThroughDate="2016-11-09T11:50:23.160" /><Visit VisitID="349978" VisitTypeID="1001" VisitTypeName="Office Visit" CareProviderID="2686" CareProviderName="Merritt, Amanda B. NP" BillableCareProviderID="1009" BillableCareProviderName="Carter, Jessica F. MD" FromDateTime="2016-09-05 07:46:33" Quality="1/4" CheckedOutFlag="1" ReleasedToPrimePatient="0" ThroughDate="2016-10-05T08:23:18.497" /><Visit VisitID="348123" VisitTypeID="1001" VisitTypeName="Office Visit" CareProviderID="2686" CareProviderName="Merritt, Amanda B. NP" BillableCareProviderID="1009" BillableCareProviderName="Carter, Jessica F. MD" FromDateTime="2016-08-22 08:22:35" Quality="1/4" CheckedOutFlag="1" ReleasedToPrimePatient="0" ThroughDate="2016-09-21T09:32:06.387" /></Visits><Referrals><InsurancePreCert InsPreCertID="103989" AuthNumber="NO PRECERT REQUIRED" VisitsLeft="1" ReferredDate="2016-10-30" ReferredFrom="Jessica F. Carter MD" ReferredTo="EEG Testing" SpecialtyId=" " AuthorizationTypeID="1023" AuthorizationType="Pre-AUTHORIZATION" AuthorizationSubTypeID="0" AuthorizationCategoryID="0" AuthorizationCategory="Inbound" AuthorizationStatusID="1009" AuthorizationStatus="Approved" VisitID="355829" VisitDate="2016-10-30" ReferralReason="TESTING" Overdue="0" /><InsurancePreCert InsPreCertID="103990" AuthNumber="NO PRECERT REQUIRED" VisitsLeft="1" ReferredDate="2016-10-30" ReferredFrom="Jessica F. Carter MD" ReferredTo="EEG Testing" SpecialtyId=" " AuthorizationTypeID="1023" AuthorizationType="Pre-AUTHORIZATION" AuthorizationSubTypeID="0" AuthorizationCategoryID="0" AuthorizationCategory="Inbound" AuthorizationStatusID="1000" AuthorizationStatus="Ordered" VisitID="355829" VisitDate="2016-10-30" ReferralReason="TESTING" Overdue="0" /><InsurancePreCert InsPreCertID="103402" AuthNumber="no precert" VisitsLeft="1" ReferredDate="2016-10-10" ReferredFrom="Jessica F. Carter MD" ReferredTo="Jessica F. Carter MD" SpecialtyId="013" Speciality="Neurology" AuthorizationTypeID="1024" AuthorizationType="Pre-Certification" AuthorizationSubTypeID="0" AuthorizationCategoryID="0" AuthorizationCategory="Inbound" AuthorizationStatusID="1009" AuthorizationStatus="Approved" VisitID="353741" VisitDate="2016-10-10" ReferralReason="VNS" Overdue="0" /><InsurancePreCert InsPreCertID="102476" AuthNumber="no precert" VisitsLeft="1" ReferredDate="2016-09-05" ReferredFrom="Jessica F. Carter MD" ReferredTo="Jessica F. Carter MD" SpecialtyId="013" Speciality="Neurology" AuthorizationTypeID="1024" AuthorizationType="Pre-Certification" AuthorizationSubTypeID="0" AuthorizationCategoryID="0" AuthorizationCategory="Inbound" AuthorizationStatusID="1009" AuthorizationStatus="Approved" VisitID="0" ReferralReason="VNS" Overdue="0" /><InsurancePreCert InsPreCertID="44491" AuthNumber="((PHONE_NUMBER))" VisitsLeft="0" AuthorizationTypeID="1024" AuthorizationType="Pre-Certification" AuthorizationSubTypeID="0" AuthorizationCategoryID="0" AuthorizationCategory="Inbound" AuthorizationStatusID="1009" AuthorizationStatus="Approved" ReferralReason="" Overdue="0" /></Referrals><hra_data><unreviewed_hras alertenabled="0" /><reviewed_hra_count value="0" /></hra_data><CareTeam /><reference_documents /><PopulationHealth><RiskScoreList /><CareGapList /></PopulationHealth><problemList><item caption="Seizure, complex partial" V="ICD9-345.40" altsystem="ICD9" altsystemcode="345.40" notes="" dateOnset="2012-07-11" CreateDate="2012-07-11 4:09PM" CatItemID="6976" PL="1" problemlistid="16705" statusid="1001" displayorder="0" SNOMEDCodeUS="TBD" LastChanged="2012-07-11 4:09PM" ICD9Code="345.40" ICD10Code="" SNOMEDCode="" LexicalID="" ProblemNote="" ProblemStartDate="2012-07-11" IsMapICD10="0" HoverText="Seizure, complex partial&#10;ICD-9: 345.40&#10;ICD-10: &#10;SNOMED: " parentcategory="pmh"><modifier /><occurrences /></item></problemList></history><plansection><medsreconciliation active="true" /><medstransitionofcare active="false" /><ProvidedEducationMaterials active="false" /></plansection><emcoding> <risk caption="Risk of complications and/or Morbidity or Mortality" value="0" /> <reviewed> <labs caption="Reviewed/Ordered lab test" value="0" /> <radiographic caption="Reviewed/Ordered radiographic tests" value="0" /> <diagnostic caption="Reviewed/Ordered other diagnostic studies" value="0" /> <discussedtest caption="Discussed test results with performing physician." value="0" /> <obtainedoldrecords caption="Obtained old records or history from additional person" value="0" /> <reviewoldrecords caption="Reviewed old records" value="0" /> <independentreview caption="Independently reviewed data(i.e. lab specimen)" value="0" /> </reviewed> <suggestedcode value="99213" /> <actualcode value="99213" Mod1="" Mod2="" Mod3="" Mod4="" Modified="I" procedureexportid="330941"><mappeddxs><dx icd9="345.40" title="Complex partial epilepsy" planuniqueid="345.400" icd10="G40.209" /></mappeddxs></actualcode> <newpatient caption="Is the patient new" value="false" /><primaryfocus caption="Is visit focus primarily counseling or coordination of care?" value="0" timeamount="0" /></emcoding></document>"""

In [20]:
import re

def deidentify_text(text: str, original: str, replacement: str) -> str:
    """
    Replace occurrences of `original` in `text` with `replacement`
    using same regex logic as NotesRule.
    """
    try:
        pattern = re.escape(original)  # escape original to avoid regex issues
        #text = re.sub(rf"(?<!\d){pattern}(?!\d)", replacement, text)
        text = re.sub(rf"(?<!\d){pattern}(?!\d)", replacement, text)
    except Exception as e:
        print(f"⚠️ Regex error for pattern {original}: {e}")
    return text


# 🔹 Example usage
#text = 'Patient ID is patientid="20204615". Appointment linked to 12345A, encounter 67890.'
original = "20100784"
replacement = "100000000000328892378978"

result = deidentify_text(text, original, replacement)
#print("Before:", text)
print("After :", result)


After : <?xml version='1.0' encoding='utf-8'?> <document id="1275284" documenttype="1" visitid="362300" patientid="100000000000328892378978"><copyright> Copyright 2000-2003 Greenway Medical Technologies, Inc. All Rights Reserved.</copyright><templates servicecategory="1" /><version value="5" /><features><feature name="DiagnosisSecondaryCodes" /></features><cc ccnote=""><option value="Seizures" selected="true" /></cc><hpi hpinote="" /><ros><root Sig="" Key=""><system caption="TopLevel" uniqueid="1000" base="true" Copen="true" Fopen="false" Topen="false"><bullet caption="Constitutional" V="" uniqueid="1002" base="true" Copen="true" Fopen="false" Topen="false"><comprehensive uniqueid="1017"><concept caption="fatigue" enabled="true" uniqueid="0_7882304"><modifier caption="" type="presence" uniqueid="0_5190181" /></concept><concept caption="fever" enabled="true" uniqueid="0_79045"><modifier caption="" type="presence" uniqueid="0_7462457" /></concept><concept caption="chills" enabled="true" 

In [12]:
#%load_ext autoreload
#%autoreload 2
%reload_ext autoreload

import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/karanchilwal/Documents/project/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from core.process_df.rules  import DateOffsetRule  # adjust import to your project

if __name__ == "__main__":
    # Sample dataframe to test different date formats
    data = {
        "dates": [
            
            "3000-01-01 00:00:00.000",  # with ms
            "3000-01-01 00:00:00",      # without ms
            "3000-01-01",               # only date
            "12/26/2024",               # US format
            "25-12-2024",               # EU format
            "Feb 20 2025",              # Month name
            "20240101",                 # compact, should NOT match
            "no date here",              # text, should NOT match
            "2000-01-01 00:00:00.000",  # with ms
        ],
        "_resolved_offset": [10] * 9  # apply +10 days offset
    }
    df = pd.DataFrame(data)

    # Run the rule
    rule = DateOffsetRule(format_as_datetime=True)
    result = rule.apply(df.copy(), {"column_name": "dates"})

    # Print side-by-side for comparison
    print("\n=== Input vs Output ===")
    print(pd.DataFrame({"input": df["dates"], "output": result["dates"]}))


2025-09-03 12:37:58,399 - deIdentification.nd_logger - INFO - [DateOffsetRule] Starting DateOffsetRule for column: dates
2025-09-03 12:37:58,400 - deIdentification.nd_logger - INFO - [DateOffsetRule] Found 8 rows with date patterns.
2025-09-03 12:37:58,402 - deIdentification.nd_logger - INFO - [DateOffsetRule] DateOffsetRule completed.



=== Input vs Output ===
                     input               output
0  3000-01-01 00:00:00.000  3000-01-11 00:00:00
1      3000-01-01 00:00:00  3000-01-11 00:00:00
2               3000-01-01  3000-01-11 00:00:00
3               12/26/2024  2025-01-05 00:00:00
4               25-12-2024  2025-01-04 00:00:00
5              Feb 20 2025  2025-03-02 00:00:00
6                 20240101  2024-01-11 00:00:00
7             no date here                 None
8  2000-01-01 00:00:00.000  2000-01-11 00:00:00


In [13]:
xml_encoded = b'0x1F8B08000000000004009D90C14AC4500C457FE5D2B5D885DB5A180BE26CC691EA07BC794DFB1E3E1349D291FEBD1D18507470E12E219793C36DBA9239C75076E2B4757A6B9B51C479DDDA260A3BB1C37C29D4C940B7D59D94A1BA7879E181748551D5F6E1189843C28E669522D382FE9D620E259B1BF65D539F01EDB7E9BFDF9EE61C5F71F2BF883D28EA3FE00FAB92E882C7117B253B65B6A53099FD646C462785275199A78498823A948E993EAE1070BFE99F61519420236E9082E140C4086679621AAECF2AF557C1F5AFF63F0185ACC1A990010000'

In [14]:
import zlib
import re
def clean_xml(xml_bytes):
    """Decode, strip illegal characters, and try to recover broken XML."""
    # Decode to string
    xml_string = xml_bytes.decode("utf-8", errors="ignore")

    # Remove invalid XML control characters
    xml_string = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", xml_string)

    # Validate & repair XML using lxml
    try:
        parser = etree.XMLParser(recover=True)  # recover=True repairs bad XML
        root = etree.fromstring(xml_string.encode("utf-8"), parser)
        # Return canonicalized string
        return etree.tostring(root, encoding="utf-8").decode("utf-8")
    except Exception as e:
        print(f"❌ Skipping invalid XML: {e}")
        return None



binary_data = zlib.decompress(xml_encoded)
xml_string = clean_xml(binary_data)
print(xml_string)


error: Error -3 while decompressing data: incorrect header check

In [16]:
import zlib
import re
from lxml import etree

def clean_xml(xml_bytes):
    """Decode, strip illegal characters, and try to recover broken XML."""
    xml_string = xml_bytes.decode("utf-8", errors="ignore")

    # Remove invalid XML control characters
    xml_string = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", xml_string)

    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(xml_string.encode("utf-8"), parser)
        return etree.tostring(root, encoding="utf-8").decode("utf-8")
    except Exception as e:
        print(f"❌ Skipping invalid XML: {e}")
        return None


# Your SQL Server hex blob
xml_encoded = b''
# 1. Strip the `0x` prefix
if xml_encoded.startswith(b"0x"):
    xml_encoded = xml_encoded[2:]

# 2. Convert hex → bytes
compressed_data = bytes.fromhex(xml_encoded.decode("utf-8"))

# 3. Decompress with gzip (since it starts with 1F8B, that’s gzip header)
binary_data = zlib.decompress(compressed_data, 16+zlib.MAX_WBITS)

# 4. Clean & validate XML
xml_string = clean_xml(binary_data)
print(xml_string)


error: Error -3 while decompressing data: incorrect header check